In [ ]:
%pwd

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix
from sklearn.metrics import roc_curve, roc_auc_score
import seaborn as sns
from sklearn.preprocessing import StandardScaler

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
file_path = "/content/drive/My Drive/WRM_F.csv"
wrm_df=pd.read_csv(file_path)

wrm_df.columns
wrm_df=wrm_df.rename(columns={'BFI L':'BFI_L_Value', 'BFI C ':'BFI_C_Value', 'BFI R ':'BFI_R_Value'})
print(wrm_df.columns)
print(wrm_df.shape)

Feature Selection: Initially selected using domain knowledge and later some dropped using feature selection methodologiesdrop nan, features with less variance( because they are less informative,

In [ ]:
wrm_df.isnull().sum()
selected_columns=['LProf62', 'RProf62', 'LAlign62', 'RAlign62', 'Gage',
                'Warp62', 'Twist31', 'BFI_L_Value', 'BFI_C_Value', 'BFI_R_Value','Crosslevel','TotalLatForce', 'Left Cant', 'Right Cant','Curvature','Speed']
wrm_df=wrm_df[selected_columns]
df=wrm_df.abs()

In [ ]:
df.head()

In [ ]:
from sklearn.feature_selection import VarianceThreshold # remove features with low variance<0.01, likely uninformative
sel = VarianceThreshold(threshold=0.01)
df_high_var = sel.fit_transform(df)

In [ ]:
df=pd.DataFrame(df_high_var, columns=selected_columns)
print(df.columns)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
plt.figure(figsize=(10, 8))
sns.heatmap(df.corr(), annot=True, fmt=".2f", cmap="coolwarm", square=True)
plt.title("Correlation Matrix of Selected Features", fontsize=14)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

print(df.head(),df.shape)

In [ ]:
# Compute correlation matrix
corr_matrix = df.corr().abs()

# Select upper triangle of correlation matrix
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

# Find features with correlation > 0.85
to_drop = [column for column in upper.columns if any(upper[column] > 0.85)]

# Drop those features
df_reduced = df.drop(columns=to_drop)

print("Dropped features:", to_drop)



In [ ]:
df=df_reduced # dropped twist31 and curvature
print(df.columns)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
corr_matrix = df.corr().abs()
plt.figure(figsize=(10, 8))
sns.heatmap(df.corr(), annot=True, fmt=".2f", cmap="coolwarm", square=True)
plt.title("Correlation Matrix", fontsize=14)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

Data Cleaning

In [ ]:
df.head()

In [ ]:
df.describe()

In [ ]:
print("Missing values:\n", df.isnull().sum())

In [ ]:
print("\nData types:\n", df.dtypes)

In [ ]:
print("\nSummary statistics:\n", df.describe())

EDA snapshot for feature distribution before attack of SCADA

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt


for col in df:
    plt.figure(figsize=(6, 3))
    sns.histplot(df[col], kde=True, bins=50)
    plt.title(f"{col} Distribution (Normal Data Only)")
    plt.grid(True)
    plt.tight_layout()
    plt.show()


In [ ]:
fig, (ax1,ax2,ax3)=plt.subplots(1,3,figsize=(8,6))

sns.scatterplot(data=df,x="Gage",y="Warp62",ax=ax1)
ax1.set_title("Gage vs Warp62 (normal)")
ax1.set_xlabel("Gage")
ax1.set_ylabel("Warp62")

sns.scatterplot(data=df,x="Gage",y="Crosslevel",ax=ax2)
ax2.set_title("Gage vs Crosslevel (normal)")
ax2.set_xlabel("Gage")
ax2.set_ylabel("Crosslevel")

sns.scatterplot(data=df,x="Crosslevel",y="Warp62",ax=ax3)
ax3.set_title("Warp62 vs Crosslevel (normal)")
ax3.set_xlabel("Crosslevel")
ax3.set_ylabel("Warp")

plt.tight_layout()
plt.show()

In [ ]:
df.var().sort_values(ascending=False)


Feature	Use for Injection?	Suggested Attack Logic
Gage	✅ Yes (continuous + independent)	Additive spike (e.g., +1.5)
Warp62	✅ Yes (skewed + sensitive)	Multiply by ×2
Crosslevel	✅ Yes (impacts safety + quantized)	Subtract offset (e.g., –0.8)
BFI_L_Value	❌ No (discrete + limited range)	Will look unnatural if perturbed
Curvature	❓ Caution — already multimodal	Could combine with Crosslevel for multi-feature intrusion
TotalLatForce	❌ No (has distinct sensor regimes)	Too risky to inject without domain calibration

Unsupervised Learning to be able to identify intrusion and outliers
first creat sliding without before the use of PCA

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import seaborn as sns


# Standardize the features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df)

# Convert back to DataFrame for easier handling if needed, keeping column names
X_scaled_df = pd.DataFrame(X_scaled, columns=df.columns)

print("Scaled data shape:", X_scaled_df.shape)
print("Scaled data head:\n", X_scaled_df.head())

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.svm import OneClassSVM
from sklearn.cluster import DBSCAN

# Dictionary to store predictions and scores for each model
results = {}

# --- Isolation Forest ---
print("\n--- Isolation Forest ---")
iso_forest = IsolationForest(contamination='auto', random_state=42)
iso_forest.fit(X_scaled_df)
y_pred_iso = iso_forest.predict(X_scaled_df)
scores_iso = iso_forest.score_samples(X_scaled_df)
n_anomalies_iso = np.sum(y_pred_iso == -1)
print(f"Detected anomalies: {n_anomalies_iso} out of {len(X_scaled_df)} samples")
print(f"Percentage of anomalies: {n_anomalies_iso / len(X_scaled_df) * 100:.2f}%")

# Visualize Isolation Forest scores
plt.figure(figsize=(8, 5))
sns.histplot(scores_iso, bins=50, kde=True, color='skyblue')
plt.title("Isolation Forest Anomaly Score Distribution Before Injection")
plt.xlabel("Anomaly Score (Lower is more anomalous)")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()

# Store predictions and scores
X_scaled_df_temp = X_scaled_df.copy()
X_scaled_df_temp['iso_forest_prediction'] = y_pred_iso
X_scaled_df_temp['iso_forest_score'] = scores_iso
print("\nTop 10 most anomalous points according to Isolation Forest:")
print(X_scaled_df_temp.sort_values(by='iso_forest_score').head(10))
results['Isolation Forest'] = (y_pred_iso, scores_iso)



# Clean up temporary DataFrame
del X_scaled_df_temp

Auto Encoder for Anomaly Detection before injection or SCADA attack

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import Input, Dense, Dropout, BatchNormalization
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping
# Removed precision_score, recall_score, f1_score, accuracy_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA # Import PCA for latent space visualization
from sklearn.manifold import TSNE # Optional: Import t-SNE

# ====================
# CONFIGURATION
# ====================
input_dim = X_scaled_df.shape[1]
encoding_dim = 7 # Your chosen latent dimension
batch_size = 64
validation_split = 0.2
dropout_rate = 0.2

# Emphasize safety-critical features like Gage and Align
feature_weights = np.array([1.5 if 'Gage' in col or 'Align' in col else 1.0 for col in X_scaled_df.columns])

# ====================
# MODEL BUILDING
# ====================
def build_autoencoder():
    input_layer = Input(shape=(input_dim,))
    x = Dense(64, activation='relu')(input_layer)
    x = BatchNormalization()(x)
    x = Dropout(dropout_rate)(x)
    x = Dense(32, activation='relu')(x)
    x = Dense(16, activation='relu')(x)
    encoded = Dense(encoding_dim, activation='relu', name='encoder_output')(x) # Give encoder layer a name

    x = Dense(16, activation='relu')(encoded)
    x = Dense(32, activation='relu')(x)
    x = Dropout(dropout_rate)(x)
    x = BatchNormalization()(x)
    decoded = Dense(input_dim, activation='linear')(x)

    autoencoder = Model(inputs=input_layer, outputs=decoded)
    autoencoder.compile(optimizer='adam', loss='mse')
    return autoencoder

# ====================
# TRAINING
# ====================
autoencoder = build_autoencoder()
history = autoencoder.fit(
    X_scaled_df, X_scaled_df,
    epochs=100,
    batch_size=batch_size,
    validation_split=validation_split,
    callbacks=[EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)],
    verbose=1
)

# ====================
# ANOMALY SCORING
# ====================
reconstructions = autoencoder.predict(X_scaled_df, verbose=0)
errors = np.square(X_scaled_df - reconstructions) * feature_weights
mse_weighted = np.mean(errors, axis=1)

# Use fixed quantile threshold (e.g., 90.05% = 9.95% anomalies)
threshold = np.percentile(mse_weighted, 90)
# Identify anomalies based on the threshold
anomalies_ae = (mse_weighted > threshold).astype(int) # Renamed to anomalies_ae for clarity
detected_anomalies_count = np.sum(anomalies_ae)
detected_anomalies_percentage = (detected_anomalies_count / len(X_scaled_df)) * 100

print("\n=== Autoencoder Anomaly Detection (Pre-Injection) ===")
print(f"Threshold (based on top 9.95% error): {threshold:.4f}")
print(f"Detected anomalies (based on threshold): {detected_anomalies_count} out of {len(X_scaled_df)} samples")
print(f"Percentage of detected anomalies: {detected_anomalies_percentage:.2f}%")


# ====================
# VISUALIZATIONS
# ====================

# 1. Weighted MSE Distribution Plot (Already exists, keeping it)
plt.figure(figsize=(10, 6))
sns.histplot(mse_weighted, bins=50, kde=True, color='blue', alpha=0.6)
plt.axvline(threshold, color='red', linestyle='--', label=f'Threshold = {threshold:.4f}')
plt.title("Autoencoder Weighted MSE Distribution (Pre-Injection)")
plt.xlabel("Weighted Reconstruction Error")
plt.ylabel("Frequency")
plt.legend()
plt.tight_layout()
plt.show()

# 2. Reconstruction Error per Feature (Bar Plot)
feature_errors_mean = np.mean(np.square(X_scaled_df - reconstructions), axis=0) # Calculate mean squared error per feature
plt.figure(figsize=(12, 6))
sns.barplot(x=X_scaled_df.columns, y=feature_errors_mean)
plt.xticks(rotation=90)
plt.title("Mean Squared Reconstruction Error per Feature")
plt.ylabel("Mean Squared Error")
plt.tight_layout()
plt.show()

# 3. Latent Space Visualization (using PCA for dimensionality reduction)
# Create a separate model for the encoder part
encoder_model = Model(inputs=autoencoder.input, outputs=autoencoder.get_layer('encoder_output').output)
latent_representation = encoder_model.predict(X_scaled_df, verbose=0)

# Apply PCA to reduce latent space to 2 components for visualization
pca_latent = PCA(n_components=2)
latent_pca = pca_latent.fit_transform(latent_representation)

plt.figure(figsize=(10, 8))
# Plot normal points in grey/blue, anomalies in red
plt.scatter(latent_pca[:, 0][anomalies_ae == 0], latent_pca[:, 1][anomalies_ae == 0],
            c='skyblue', label='Normal (Predicted)', alpha=0.5, s=10)
plt.scatter(latent_pca[:, 0][anomalies_ae == 1], latent_pca[:, 1][anomalies_ae == 1],
            c='red', label='Anomaly (Predicted)', alpha=0.7, s=20)
plt.title("Autoencoder Latent Space Visualization (PCA)")
plt.xlabel("PCA Component 1")
plt.ylabel("PCA Component 2")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

# Optional: Add t-SNE visualization if PCA doesn't show clear separation
# t-SNE is more computationally expensive and sensitive to parameters
# try:
#     tsne_latent = TSNE(n_components=2, random_state=42, perplexity=30, n_iter=300)
#     latent_tsne = tsne_latent.fit_transform(latent_representation)
#
#     plt.figure(figsize=(10, 8))
#     plt.scatter(latent_tsne[:, 0][anomalies_ae == 0], latent_tsne[:, 1][anomalies_ae == 0],
#                 c='skyblue', label='Normal (Predicted)', alpha=0.5, s=10)
#     plt.scatter(latent_tsne[:, 0][anomalies_ae == 1], latent_tsne[:, 1][anomalies_ae == 1],
#                 c='red', label='Anomaly (Predicted)', alpha=0.7, s=20)
#     plt.title("Autoencoder Latent Space Visualization (t-SNE)")
#     plt.xlabel("t-SNE Component 1")
#     plt.ylabel("t-SNE Component 2")
#     plt.legend()
#     plt.grid(True)
#     plt.tight_layout()
#     plt.show()
# except Exception as e:
#     print(f"Could not generate t-SNE plot: {e}")


# 4. Sample Reconstruction Comparison (Example for a few features)
# Select indices of a few normal samples and a few anomalous samples
normal_indices = np.where(anomalies_ae == 0)[0]
anomaly_indices = np.where(anomalies_ae == 1)[0]

# Choose a few samples (e.g., first 5 normal, first 5 anomalous)
n_display = 5
if len(normal_indices) > n_display and len(anomaly_indices) > n_display:
    selected_normal_idx = normal_indices[:n_display]
    selected_anomaly_idx = anomaly_indices[:n_display]

    # Select a few representative features to plot
    # Choose features with higher reconstruction errors or domain importance
    feature_indices_to_plot = [X_scaled_df.columns.get_loc(col) for col in ['Gage', 'Warp62', 'Crosslevel']] # Example features

    print(f"\nVisualizing reconstructions for {n_display} normal and {n_display} anomalous samples:")

    for i in range(n_display):
        fig, axes = plt.subplots(1, len(feature_indices_to_plot), figsize=(15, 4), sharey=True)

        # Normal sample
        normal_original = X_scaled_df.iloc[selected_normal_idx[i], feature_indices_to_plot].values
        normal_reconstructed = reconstructions[selected_normal_idx[i], feature_indices_to_plot]

        # Anomalous sample
        anomaly_original = X_scaled_df.iloc[selected_anomaly_idx[i], feature_indices_to_plot].values
        anomaly_reconstructed = reconstructions[selected_anomaly_idx[i], feature_indices_to_plot]

        for j, feature_idx in enumerate(feature_indices_to_plot):
            feature_name = X_scaled_df.columns[feature_idx]

            axes[j].plot([0, 1], [normal_original[j], normal_reconstructed[j]], 'bo-', label='Normal')
            axes[j].plot([0, 1], [anomaly_original[j], anomaly_reconstructed[j]], 'ro-', label='Anomaly')
            axes[j].set_xticks([0, 1])
            axes[j].set_xticklabels(['Original', 'Reconstructed'])
            axes[j].set_title(f'{feature_name}')
            axes[j].grid(True)
            if j == 0:
                 axes[j].set_ylabel('Scaled Feature Value')

        handles, labels = axes[0].get_legend_handles_labels()
        fig.suptitle(f"Sample Reconstructions - Sample {i+1}", y=1.02)
        fig.legend(handles, labels, loc='upper right')
        plt.tight_layout()
        plt.show()
else:
    print("\nNot enough normal or anomalous samples to visualize reconstructions.")
    print(f"Normal samples found: {len(normal_indices)}, Anomalous samples found: {len(anomaly_indices)}")
# ====================
# ANOMALY SCORING
# ====================
reconstructions = autoencoder.predict(X_scaled_df, verbose=0)
errors = np.square(X_scaled_df - reconstructions) * feature_weights
mse_weighted = np.mean(errors, axis=1)

# Use fixed quantile threshold (e.g., 10% anomalies)
# Note: This threshold is based on the assumption that 9.95% of the pre-injection data are anomalies.
# This is a common unsupervised approach but doesn't validate against true labels.
threshold = np.percentile(mse_weighted, 90)
# You can keep y_pred if you want to count detected anomalies based on this threshold
y_pred = (mse_weighted > threshold).astype(int)
detected_anomalies_count = np.sum(y_pred)
detected_anomalies_percentage = (detected_anomalies_count / len(X_scaled_df)) * 100

print("\n=== Autoencoder Anomaly Detection (Pre-Injection) ===")
print(f"Threshold (based on top 10% error): {threshold:.4f}")
print(f"Detected anomalies (based on threshold): {detected_anomalies_count} out of {len(X_scaled_df)} samples")
print(f"Percentage of detected anomalies: {detected_anomalies_percentage:.2f}%")



# PLOT ANOMALY SCORE DISTRIBUTION (using the calculated threshold)
plt.figure(figsize=(10, 6))
# You don't have y_true here, so plot all scores and highlight the threshold
sns.histplot(mse_weighted, bins=50, kde=True, color='blue', alpha=0.6)
plt.axvline(threshold, color='red', linestyle='--', label=f'Threshold = {threshold:.4f}')
plt.title("Autoencoder Weighted MSE Distribution (Pre-Injection)")
plt.xlabel("Weighted Reconstruction Error")
plt.ylabel("Frequency")
plt.legend()
plt.tight_layout()
plt.show()

# You can also visualize the samples above the threshold if desired
# For example, you could plot a scatter plot of some features highlighting these points
# This is optional and depends on your analysis needs.


In [ ]:
# ====================
# ANOMALY SCORING WITH MAD
# ====================
reconstructions = autoencoder.predict(X_scaled_df, verbose=0)
errors = np.square(X_scaled_df - reconstructions) * feature_weights
mse_weighted = np.mean(errors, axis=1)

# Median Absolute Deviation (MAD) Threshold
median = np.median(mse_weighted)
mad = 1.4826 * np.median(np.abs(mse_weighted - median))  # Robust STD estimate
mad_threshold = median + (3 * mad)  # 3 MADs = ~99.7% in normal distribution

# Identify anomalies
anomalies_mad = (mse_weighted > mad_threshold).astype(int)
detected_mad = np.sum(anomalies_mad)
percentage_mad = (detected_mad / len(X_scaled_df)) * 100

print("\n=== MAD-Based Anomaly Detection ===")
print(f"Median Error: {median:.4f}, MAD: {mad:.4f}")
print(f"MAD Threshold (median + 3*MAD): {mad_threshold:.4f}")
print(f"Detected anomalies: {detected_mad} out of {len(X_scaled_df)} samples")
print(f"Percentage: {percentage_mad:.2f}%")

In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(mse_weighted, bins=50, kde=True, color='blue', alpha=0.6)
plt.axvline(mad_threshold, color='red', linestyle='--',
            label=f'MAD Threshold = {mad_threshold:.4f}')
plt.title("Weighted MSE Distribution with MAD Threshold")
plt.xlabel("Reconstruction Error")
plt.ylabel("Frequency")
plt.legend()
plt.show()

In [ ]:
# ====================
# CLUSTERING-BASED FILTERING
# ====================
from sklearn.cluster import DBSCAN

# Get MAD-detected anomalies' feature values
anomalous_samples = X_scaled_df[anomalies_mad == 1]

# Cluster anomalies to separate true attacks from noise
dbscan = DBSCAN(eps=0.5, min_samples=5).fit(anomalous_samples)
cluster_labels = dbscan.labels_

# Only keep points in large clusters (likely true anomalies)
min_cluster_size = 10  # Adjust based on your data
cluster_counts = np.bincount(cluster_labels[cluster_labels >= 0])  # Ignore noise (-1)
valid_clusters = np.where(cluster_counts >= min_cluster_size)[0]
final_anomalies = np.isin(cluster_labels, valid_clusters)

# Map back to original indices
refined_anomalies = np.zeros(len(X_scaled_df), dtype=int)
anomaly_indices = np.where(anomalies_mad == 1)[0]
refined_anomalies[anomaly_indices[final_anomalies]] = 1

print("\n=== After Clustering Filtering ===")
print(f"Original MAD anomalies: {detected_mad}")
print(f"After clustering: {np.sum(refined_anomalies)}")
print(f"Percentage: {(np.sum(refined_anomalies)/len(X_scaled_df))*100:.2f}%")

In [ ]:
# Reduce anomalous samples to 2D for visualization
pca = PCA(n_components=2)
anomalies_pca = pca.fit_transform(anomalous_samples)

plt.figure(figsize=(10, 6))
scatter = plt.scatter(
    anomalies_pca[:, 0], anomalies_pca[:, 1],
    c=cluster_labels, cmap='viridis', alpha=0.6
)
plt.colorbar(scatter, label='Cluster ID')
plt.title("DBSCAN Clustering of MAD-Detected Anomalies")
plt.xlabel("PCA Component 1")
plt.ylabel("PCA Component 2")
plt.show()

VAE implementation

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.ensemble import IsolationForest

# Assume X_scaled_df, vae, vae_encoder_model, vae_decoder_model, and feature_weights are available as pre-trained

# Dictionary to store predictions and scores for each model
results = {}

# --- Isolation Forest ---
print("\n--- Isolation Forest ---")
iso_forest = IsolationForest(contamination=0.05, random_state=42)
iso_forest.fit(X_scaled_df)
y_pred_iso = iso_forest.predict(X_scaled_df)
scores_iso = -iso_forest.score_samples(X_scaled_df)  # Lower is more anomalous
n_anomalies_iso = np.sum(y_pred_iso == -1)
print(f"Detected anomalies: {n_anomalies_iso} out of {len(X_scaled_df)} samples")
print(f"Percentage of anomalies: {n_anomalies_iso / len(X_scaled_df) * 100:.2f}%")

# Visualize Isolation Forest scores with blue for normal, red for anomalies
plt.figure(figsize=(8, 5))
sns.histplot(scores_iso, bins=50, kde=True, color='blue', label='All Scores')
plt.axvline(np.percentile(scores_iso, 95), color='red', linestyle='--', label='Anomaly Threshold')
plt.title("Isolation Forest Anomaly Score Distribution (Before Injection)", fontweight='bold', color='black')
plt.xlabel("Anomaly Score (Lower is more anomalous)", color='black')
plt.ylabel("Frequency", color='black')
plt.legend()
plt.tight_layout()
plt.savefig('iso_forest_baseline_scores.png')
plt.show()
plt.close()

# Store predictions and scores
X_scaled_df_temp = X_scaled_df.copy()
X_scaled_df_temp['iso_forest_prediction'] = y_pred_iso
X_scaled_df_temp['iso_forest_score'] = scores_iso
print("\nTop 10 most anomalous points according to Isolation Forest:")
print(X_scaled_df_temp.sort_values(by='iso_forest_score').head(10))
results['Isolation Forest'] = (y_pred_iso, scores_iso)

# --- VAE ---
print("\n--- VAE ---")
reconstructions_vae = vae.predict(X_scaled_df, verbose=0)
reconstruction_error_vae = np.mean((X_scaled_df - reconstructions_vae) ** 2, axis=1)
vae_threshold = np.percentile(reconstruction_error_vae, 95)
y_pred_vae = (reconstruction_error_vae > vae_threshold).astype(int)
n_anomalies_vae = np.sum(y_pred_vae == 1)
print(f"Detected anomalies: {n_anomalies_vae} out of {len(X_scaled_df)} samples")
print(f"Percentage of anomalies: {n_anomalies_vae / len(X_scaled_df) * 100:.2f}%")

# Visualize VAE reconstruction errors with blue for normal, red for anomalies
plt.figure(figsize=(8, 5))
sns.histplot(reconstruction_error_vae, bins=50, kde=True, color='blue', label='All Scores')
plt.axvline(vae_threshold, color='red', linestyle='--', label='Anomaly Threshold')
plt.title("VAE Reconstruction Error Distribution (Before Injection)", fontweight='bold', color='black')
plt.xlabel("Reconstruction Error (Higher is more anomalous)", color='black')
plt.ylabel("Frequency", color='black')
plt.legend()
plt.tight_layout()
plt.savefig('vae_baseline_scores.png')
plt.show()
plt.close()

# Store predictions and scores
X_scaled_df_temp['vae_prediction'] = y_pred_vae
X_scaled_df_temp['vae_score'] = reconstruction_error_vae
print("\nTop 10 most anomalous points according to VAE:")
print(X_scaled_df_temp.sort_values(by='vae_score', ascending=False).head(10))
results['VAE'] = (y_pred_vae, reconstruction_error_vae)

# --- VAE + MAD ---
print("\n--- VAE + MAD ---")
reconstruction_error_mad = np.mean(feature_weights * (X_scaled_df - reconstructions_vae) ** 2, axis=1)
def mad_score(data):
    median = np.median(data)
    mad = np.median(np.abs(data - median))
    if mad == 0: mad = 1e-10
    return np.abs(data - median) / mad
mad_scores = mad_score(reconstruction_error_mad)
mad_threshold = np.percentile(mad_scores, 95)
y_pred_mad = (mad_scores > mad_threshold).astype(int)
n_anomalies_mad = np.sum(y_pred_mad == 1)
print(f"Detected anomalies: {n_anomalies_mad} out of {len(X_scaled_df)} samples")
print(f"Percentage of anomalies: {n_anomalies_mad / len(X_scaled_df) * 100:.2f}%")

# Visualize VAE + MAD MAD scores with blue for normal, red for anomalies
plt.figure(figsize=(8, 5))
sns.histplot(mad_scores, bins=50, kde=True, color='blue', label='All Scores')
plt.axvline(mad_threshold, color='red', linestyle='--', label='Anomaly Threshold')
plt.title("VAE + MAD MAD Score Distribution (Before Injection)", fontweight='bold', color='black')
plt.xlabel("MAD Score (Higher is more anomalous)", color='black')
plt.ylabel("Frequency", color='black')
plt.legend()
plt.tight_layout()
plt.savefig('vae_mad_baseline_scores.png')
plt.show()
plt.close()

# Store predictions and scores
X_scaled_df_temp['vae_mad_prediction'] = y_pred_mad
X_scaled_df_temp['vae_mad_score'] = mad_scores
print("\nTop 10 most anomalous points according to VAE + MAD:")
print(X_scaled_df_temp.sort_values(by='vae_mad_score', ascending=False).head(10))
results['VAE + MAD'] = (y_pred_mad, mad_scores)

# --- Visualizations ---

# 1. Feature-wise Reconstruction Errors (Bar Plot for VAE and VAE + MAD)
feature_errors_vae = np.mean(np.square(X_scaled_df - reconstructions_vae), axis=0)
plt.figure(figsize=(12, 6))
sns.barplot(x=X_scaled_df.columns, y=feature_errors_vae, color='orange')
plt.xticks(rotation=90)
plt.title("VAE Mean Squared Reconstruction Error per Feature (Before Injection)", fontweight='bold', color='black')
plt.ylabel("Mean Squared Error", color='black')
plt.tight_layout()
plt.savefig('vae_feature_errors_baseline.png')
plt.show()
plt.close()

# 2. PCA for Isolation Forest Input Space
pca_iso = PCA(n_components=2)
input_pca_iso = pca_iso.fit_transform(X_scaled_df)
plt.figure(figsize=(10, 8))
plt.scatter(input_pca_iso[:, 0], input_pca_iso[:, 1], c='blue', label='Normal Points', alpha=0.5, s=10)
plt.scatter(input_pca_iso[y_pred_iso == -1, 0], input_pca_iso[y_pred_iso == -1, 1], c='red', label='Anomalies', alpha=0.7, s=20)
plt.title("Isolation Forest Input Space Visualization (PCA) - Before Injection", fontweight='bold', color='black')
plt.xlabel("PCA Component 1", color='black')
plt.ylabel("PCA Component 2", color='black')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig('iso_pca_baseline.png')
plt.show()
plt.close()

# 3. PCA for VAE Latent Space (VAE Anomalies)
latent_representation_vae = vae_encoder_model.predict(X_scaled_df, verbose=0)[0]  # Using z_mean
pca_latent_vae = PCA(n_components=2)
latent_pca_vae = pca_latent_vae.fit_transform(latent_representation_vae)
plt.figure(figsize=(10, 8))
plt.scatter(latent_pca_vae[:, 0], latent_pca_vae[:, 1], c='blue', label='Normal Points', alpha=0.5, s=10)
plt.scatter(latent_pca_vae[y_pred_vae == 1, 0], latent_pca_vae[y_pred_vae == 1, 1], c='red', label='Anomalies (VAE)', alpha=0.7, s=20)
plt.title("VAE Latent Space Visualization (PCA) - Before Injection", fontweight='bold', color='black')
plt.xlabel("PCA Component 1", color='black')
plt.ylabel("PCA Component 2", color='black')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig('vae_pca_baseline.png')
plt.show()
plt.close()

# 4. PCA for VAE + MAD Latent Space (VAE + MAD Anomalies)
plt.figure(figsize=(10, 8))
plt.scatter(latent_pca_vae[:, 0], latent_pca_vae[:, 1], c='blue', label='Normal Points', alpha=0.5, s=10)
plt.scatter(latent_pca_vae[y_pred_mad == 1, 0], latent_pca_vae[y_pred_mad == 1, 1], c='red', label='Anomalies (VAE + MAD)', alpha=0.7, s=20)
plt.title("VAE + MAD Latent Space Visualization (PCA) - Before Injection", fontweight='bold', color='black')
plt.xlabel("PCA Component 1", color='black')
plt.ylabel("PCA Component 2", color='black')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig('vae_mad_pca_baseline.png')
plt.show()
plt.close()

# 5. t-SNE for Isolation Forest Input Space
tsne_iso = TSNE(n_components=2, random_state=42)
input_tsne_iso = tsne_iso.fit_transform(X_scaled_df)
plt.figure(figsize=(10, 8))
plt.scatter(input_tsne_iso[:, 0], input_tsne_iso[:, 1], c='blue', label='Normal Points', alpha=0.5, s=10)
plt.scatter(input_tsne_iso[y_pred_iso == -1, 0], input_tsne_iso[y_pred_iso == -1, 1], c='red', label='Anomalies', alpha=0.7, s=20)
plt.title("Isolation Forest Input Space Visualization (t-SNE) - Before Injection", fontweight='bold', color='black')
plt.xlabel("t-SNE 1", color='black')
plt.ylabel("t-SNE 2", color='black')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig('iso_input_tsne_baseline.png')
plt.show()
plt.close()

# 6. t-SNE for VAE Latent Space (VAE Anomalies)
tsne_vae = TSNE(n_components=2, random_state=42)
latent_tsne_vae = tsne_vae.fit_transform(latent_representation_vae)
plt.figure(figsize=(10, 8))
plt.scatter(latent_tsne_vae[:, 0], latent_tsne_vae[:, 1], c='blue', label='Normal Points', alpha=0.5, s=10)
plt.scatter(latent_tsne_vae[y_pred_vae == 1, 0], latent_tsne_vae[y_pred_vae == 1, 1], c='red', label='Anomalies (VAE)', alpha=0.7, s=20)
plt.title("VAE Latent Space Visualization (t-SNE) - Before Injection", fontweight='bold', color='black')
plt.xlabel("t-SNE 1", color='black')
plt.ylabel("t-SNE 2", color='black')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig('vae_tsne_baseline.png')
plt.show()
plt.close()

# 7. t-SNE for VAE + MAD Latent Space (VAE + MAD Anomalies)
tsne_mad = TSNE(n_components=2, random_state=42)
latent_tsne_mad = tsne_mad.fit_transform(latent_representation_vae)
plt.figure(figsize=(10, 8))
plt.scatter(latent_tsne_mad[:, 0], latent_tsne_mad[:, 1], c='blue', label='Normal Points', alpha=0.5, s=10)
plt.scatter(latent_tsne_mad[y_pred_mad == 1, 0], latent_tsne_mad[y_pred_mad == 1, 1], c='red', label='Anomalies (VAE + MAD)', alpha=0.7, s=20)
plt.title("VAE + MAD Latent Space Visualization (t-SNE) - Before Injection", fontweight='bold', color='black')
plt.xlabel("t-SNE 1", color='black')
plt.ylabel("t-SNE 2", color='black')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig('vae_mad_tsne_baseline.png')
plt.show()
plt.close()

# 8. Sample Reconstruction Comparison (VAE and VAE + MAD)
normal_indices = np.where(y_pred_vae == 0)[0]
anomaly_indices = np.where(y_pred_vae == 1)[0]
n_display = 5
if len(normal_indices) >= n_display and len(anomaly_indices) >= n_display:
    selected_normal_idx = normal_indices[:n_display]
    selected_anomaly_idx = anomaly_indices[:n_display]
    feature_indices_to_plot = [X_scaled_df.columns.get_loc(col) for col in ['Gage', 'Warp62', 'Crosslevel']]  # Example features

    print(f"\nVisualizing VAE reconstructions for {n_display} normal and {n_display} anomalous samples:")

    for i in range(n_display):
        fig, axes = plt.subplots(1, len(feature_indices_to_plot), figsize=(15, 4), sharey=True)

        # Normal sample (VAE)
        normal_original = X_scaled_df.iloc[selected_normal_idx[i], feature_indices_to_plot].values
        normal_reconstructed = reconstructions_vae[selected_normal_idx[i], feature_indices_to_plot]

        # Anomalous sample (VAE)
        anomaly_original = X_scaled_df.iloc[selected_anomaly_idx[i], feature_indices_to_plot].values
        anomaly_reconstructed = reconstructions_vae[selected_anomaly_idx[i], feature_indices_to_plot]

        for j, feature_idx in enumerate(feature_indices_to_plot):
            feature_name = X_scaled_df.columns[feature_idx]
            axes[j].plot([0, 1], [normal_original[j], normal_reconstructed[j]], 'bo-', label='Normal', color='blue')
            axes[j].plot(1, normal_reconstructed[j], 'b^', markersize=8, color='blue')
            axes[j].plot([0, 1], [anomaly_original[j], anomaly_reconstructed[j]], 'ro-', label='Anomaly', color='red')
            axes[j].plot(1, anomaly_reconstructed[j], 'r^', markersize=8, color='red')
            axes[j].set_xticks([0, 1])
            axes[j].set_xticklabels(['Original', 'Reconstructed'])
            axes[j].set_title(f'{feature_name}', color='black')
            axes[j].grid(True, color='grey', linestyle='--', alpha=0.7)
            if j == 0:
                axes[j].set_ylabel('Scaled Feature Value', color='black')

        handles, labels = axes[0].get_legend_handles_labels()
        unique_labels = {l: h for h, l in zip(handles, labels) if l not in unique_labels}
        fig.suptitle(f"VAE Sample Reconstructions - Sample {i+1}", fontweight='bold', color='black', y=1.02)
        fig.legend(unique_labels.values(), unique_labels.keys(), loc='upper right')
        plt.tight_layout()
        plt.savefig(f'vae_reconstruction_sample_{i+1}_baseline.png')
        plt.show()
        plt.close()
else:
    print("\nNot enough normal or anomalous samples (based on VAE predictions) to visualize reconstructions.")
    print(f"Normal samples found: {len(normal_indices)}, Anomalous samples found: {len(anomaly_indices)}")

# 9. Anomaly Detection Summary Table
anomaly_data = {
    'Model': ['Isolation Forest', 'VAE', 'VAE + MAD'],
    'Detected Anomalies': [n_anomalies_iso, n_anomalies_vae, n_anomalies_mad],
    'Percentage of Anomalies (%)': [n_anomalies_iso / len(X_scaled_df) * 100, n_anomalies_vae / len(X_scaled_df) * 100, n_anomalies_mad / len(X_scaled_df) * 100]
}
anomaly_df = pd.DataFrame(anomaly_data)
print("\nAnomaly Detection Summary Table (Before Injection):")
print(anomaly_df.to_string(index=False))

# Optional: Save results for later comparison
results_df = pd.DataFrame({
    'iso_forest_prediction': y_pred_iso,
    'iso_forest_score': scores_iso,
    'vae_prediction': y_pred_vae,
    'vae_score': reconstruction_error_vae,
    'vae_mad_prediction': y_pred_mad,
    'vae_mad_score': mad_scores
})
results_df.to_csv('baseline_anomaly_results.csv', index=False)
print("\nBaseline results saved to 'baseline_anomaly_results.csv'")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

# Data from the top 10 most anomalous points (as provided)
data = {
    'Index': [6411, 6410, 6409, 6412, 9, 71, 8, 6408, 160, 161],
    'iso_forest_score': [0.627699, 0.630945, 0.620447, 0.626825, 0.627619, 0.710485, 0.632833, 0.614082, 0.685250, 0.690763],
    'vae_score': [8.009924, 7.613307, 3.851094, 3.701563, 1.745508, 1.694780, 1.505846, 1.413122, 1.606287, 1.577244],
    'vae_mad_score': [171.925677, 163.665679, 81.571471, 78.441894, 34.098573, 30.862360, 29.173488, 28.235221, 28.128370, 28.002261]
}

# Convert to DataFrame
df_scores = pd.DataFrame(data)

# Set up the bar plot
fig, ax = plt.subplots(figsize=(12, 6))

# Bar positions
bar_width = 0.25
index = np.arange(len(df_scores['Index']))

# Plot bars for each model
plt.bar(index - bar_width, df_scores['iso_forest_score'], bar_width, label='Isolation Forest', color='blue', alpha=0.7)
plt.bar(index, df_scores['vae_score'], bar_width, label='VAE', color='red', alpha=0.7)
plt.bar(index + bar_width, df_scores['vae_mad_score'], bar_width, label='VAE + MAD', color='green', alpha=0.7)

# Customize the plot
plt.title("Anomaly Scores for Top 10 Most Anomalous Points (Before Injection)", fontweight='bold', color='black')
plt.xlabel("Index of Anomalous Points", color='black')
plt.ylabel("Anomaly Score", color='black')
plt.xticks(index, df_scores['Index'], rotation=45)
plt.legend()
plt.grid(True, color='grey', linestyle='--', alpha=0.7)
plt.tight_layout()

# Save and show the plot
plt.savefig('baseline_anomaly_scores_comparison.png')
plt.show()
plt.close()

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.ensemble import IsolationForest

# Assume X_scaled_df, vae, vae_encoder_model, vae_decoder_model, and feature_weights are available as pre-trained

# Dictionary to store predictions and scores for each model
results = {}

# --- Isolation Forest ---
print("\n--- Isolation Forest ---")
iso_forest = IsolationForest(contamination=0.05, random_state=42)
iso_forest.fit(X_scaled_df)
y_pred_iso = iso_forest.predict(X_scaled_df)
scores_iso = -iso_forest.score_samples(X_scaled_df)  # Lower is more anomalous
n_anomalies_iso = np.sum(y_pred_iso == -1)
print(f"Detected anomalies: {n_anomalies_iso} out of {len(X_scaled_df)} samples")
print(f"Percentage of anomalies: {n_anomalies_iso / len(X_scaled_df) * 100:.2f}%")

# Visualize Isolation Forest scores
plt.figure(figsize=(8, 5))
sns.histplot(scores_iso, bins=50, kde=True, color='skyblue')
plt.title("Isolation Forest Anomaly Score Distribution (Before Injection)")
plt.xlabel("Anomaly Score (Lower is more anomalous)")
plt.ylabel("Frequency")
plt.tight_layout()
plt.savefig('iso_forest_baseline_scores.png')
plt.show()
plt.close()

# Store predictions and scores
X_scaled_df_temp = X_scaled_df.copy()
X_scaled_df_temp['iso_forest_prediction'] = y_pred_iso
X_scaled_df_temp['iso_forest_score'] = scores_iso
print("\nTop 10 most anomalous points according to Isolation Forest:")
print(X_scaled_df_temp.sort_values(by='iso_forest_score').head(10))
results['Isolation Forest'] = (y_pred_iso, scores_iso)

# --- VAE ---
print("\n--- VAE ---")
reconstructions_vae = vae.predict(X_scaled_df, verbose=0)
reconstruction_error_vae = np.mean((X_scaled_df - reconstructions_vae) ** 2, axis=1)
vae_threshold = np.percentile(reconstruction_error_vae, 95)
y_pred_vae = (reconstruction_error_vae > vae_threshold).astype(int)
n_anomalies_vae = np.sum(y_pred_vae == 1)
print(f"Detected anomalies: {n_anomalies_vae} out of {len(X_scaled_df)} samples")
print(f"Percentage of anomalies: {n_anomalies_vae / len(X_scaled_df) * 100:.2f}%")

# Visualize VAE reconstruction errors
plt.figure(figsize=(8, 5))
sns.histplot(reconstruction_error_vae, bins=50, kde=True, color='orange')
plt.title("VAE Reconstruction Error Distribution (Before Injection)")
plt.xlabel("Reconstruction Error (Higher is more anomalous)")
plt.ylabel("Frequency")
plt.tight_layout()
plt.savefig('vae_baseline_scores.png')
plt.show()
plt.close()

# Store predictions and scores
X_scaled_df_temp['vae_prediction'] = y_pred_vae
X_scaled_df_temp['vae_score'] = reconstruction_error_vae
print("\nTop 10 most anomalous points according to VAE:")
print(X_scaled_df_temp.sort_values(by='vae_score', ascending=False).head(10))
results['VAE'] = (y_pred_vae, reconstruction_error_vae)

# --- VAE + MAD ---
print("\n--- VAE + MAD ---")
reconstruction_error_mad = np.mean(feature_weights * (X_scaled_df - reconstructions_vae) ** 2, axis=1)
def mad_score(data):
    median = np.median(data)
    mad = np.median(np.abs(data - median))
    if mad == 0: mad = 1e-10
    return np.abs(data - median) / mad
mad_scores = mad_score(reconstruction_error_mad)
mad_threshold = np.percentile(mad_scores, 95)
y_pred_mad = (mad_scores > mad_threshold).astype(int)
n_anomalies_mad = np.sum(y_pred_mad == 1)
print(f"Detected anomalies: {n_anomalies_mad} out of {len(X_scaled_df)} samples")
print(f"Percentage of anomalies: {n_anomalies_mad / len(X_scaled_df) * 100:.2f}%")

# Visualize VAE + MAD MAD scores
plt.figure(figsize=(8, 5))
sns.histplot(mad_scores, bins=50, kde=True, color='green')
plt.title("VAE + MAD MAD Score Distribution (Before Injection)")
plt.xlabel("MAD Score (Higher is more anomalous)")
plt.ylabel("Frequency")
plt.tight_layout()
plt.savefig('vae_mad_baseline_scores.png')
plt.show()
plt.close()

# Store predictions and scores
X_scaled_df_temp['vae_mad_prediction'] = y_pred_mad
X_scaled_df_temp['vae_mad_score'] = mad_scores
print("\nTop 10 most anomalous points according to VAE + MAD:")
print(X_scaled_df_temp.sort_values(by='vae_mad_score', ascending=False).head(10))
results['VAE + MAD'] = (y_pred_mad, mad_scores)

# --- Visualizations ---

# 1. Feature-wise Reconstruction Errors (Bar Plot for VAE and VAE + MAD)
feature_errors_vae = np.mean(np.square(X_scaled_df - reconstructions_vae), axis=0)
plt.figure(figsize=(12, 6))
sns.barplot(x=X_scaled_df.columns, y=feature_errors_vae)
plt.xticks(rotation=90)
plt.title("VAE Mean Squared Reconstruction Error per Feature (Before Injection)")
plt.ylabel("Mean Squared Error")
plt.tight_layout()
plt.savefig('vae_feature_errors_baseline.png')
plt.show()
plt.close()

# 2. PCA for Isolation Forest Input Space
pca_iso = PCA(n_components=2)
input_pca_iso = pca_iso.fit_transform(X_scaled_df)
plt.figure(figsize=(10, 8))
plt.scatter(input_pca_iso[y_pred_iso == 1, 0], input_pca_iso[y_pred_iso == 1, 1], c='skyblue', label='Normal (Isolation Forest)', alpha=0.5, s=10)
plt.scatter(input_pca_iso[y_pred_iso == -1, 0], input_pca_iso[y_pred_iso == -1, 1], c='red', label='Anomaly (Isolation Forest)', alpha=0.7, s=20)
plt.title("Isolation Forest Input Space Visualization (PCA) - Before Injection")
plt.xlabel("PCA Component 1")
plt.ylabel("PCA Component 2")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig('iso_pca_baseline.png')
plt.show()
plt.close()

# 3. PCA for VAE Latent Space (for VAE and VAE + MAD)
latent_representation_vae = vae_encoder_model.predict(X_scaled_df, verbose=0)[0]  # Using z_mean
pca_latent = PCA(n_components=2)
latent_pca = pca_latent.fit_transform(latent_representation_vae)
plt.figure(figsize=(10, 8))
plt.scatter(latent_pca[y_pred_vae == 0, 0], latent_pca[y_pred_vae == 0, 1], c='skyblue', label='Normal (VAE)', alpha=0.5, s=10)
plt.scatter(latent_pca[y_pred_vae == 1, 0], latent_pca[y_pred_vae == 1, 1], c='red', label='Anomaly (VAE)', alpha=0.7, s=20)
plt.scatter(latent_pca[y_pred_mad == 1, 0], latent_pca[y_pred_mad == 1, 1], c='green', label='Anomaly (VAE + MAD)', alpha=0.7, s=20, marker='^')
plt.title("VAE Latent Space Visualization (PCA) - Before Injection")
plt.xlabel("PCA Component 1")
plt.ylabel("PCA Component 2")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig('vae_latent_pca_baseline.png')
plt.show()
plt.close()

# 4. t-SNE for Isolation Forest Input Space
tsne_iso = TSNE(n_components=2, random_state=42)
input_tsne_iso = tsne_iso.fit_transform(X_scaled_df)
plt.figure(figsize=(10, 8))
plt.scatter(input_tsne_iso[y_pred_iso == 1, 0], input_tsne_iso[y_pred_iso == 1, 1], c='skyblue', label='Normal (Isolation Forest)', alpha=0.5, s=10)
plt.scatter(input_tsne_iso[y_pred_iso == -1, 0], input_tsne_iso[y_pred_iso == -1, 1], c='red', label='Anomaly (Isolation Forest)', alpha=0.7, s=20)
plt.title("Isolation Forest Input Space Visualization (t-SNE) - Before Injection")
plt.xlabel("t-SNE 1")
plt.ylabel("t-SNE 2")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig('iso_input_tsne_baseline.png')
plt.show()
plt.close()

# 5. t-SNE for VAE Latent Space (for VAE and VAE + MAD)
tsne = TSNE(n_components=2, random_state=42)
latent_tsne = tsne.fit_transform(latent_representation_vae)
plt.figure(figsize=(10, 8))
plt.scatter(latent_tsne[y_pred_vae == 0, 0], latent_tsne[y_pred_vae == 0, 1], c='skyblue', label='Normal (VAE)', alpha=0.5, s=10)
plt.scatter(latent_tsne[y_pred_vae == 1, 0], latent_tsne[y_pred_vae == 1, 1], c='red', label='Anomaly (VAE)', alpha=0.7, s=20)
plt.scatter(latent_tsne[y_pred_mad == 1, 0], latent_tsne[y_pred_mad == 1, 1], c='green', label='Anomaly (VAE + MAD)', alpha=0.7, s=20, marker='^')
plt.title("VAE Latent Space Visualization (t-SNE) - Before Injection")
plt.xlabel("t-SNE 1")
plt.ylabel("t-SNE 2")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig('vae_latent_tsne_baseline.png')
plt.show()
plt.close()

# 6. Sample Reconstruction Comparison (VAE and VAE + MAD)
normal_indices = np.where(y_pred_vae == 0)[0]
anomaly_indices = np.where(y_pred_vae == 1)[0]
n_display = 5
if len(normal_indices) >= n_display and len(anomaly_indices) >= n_display:
    selected_normal_idx = normal_indices[:n_display]
    selected_anomaly_idx = anomaly_indices[:n_display]
    feature_indices_to_plot = [X_scaled_df.columns.get_loc(col) for col in ['Gage', 'Warp62', 'Crosslevel']]  # Example features

    print(f"\nVisualizing VAE reconstructions for {n_display} normal and {n_display} anomalous samples:")

    for i in range(n_display):
        fig, axes = plt.subplots(1, len(feature_indices_to_plot), figsize=(15, 4), sharey=True)

        # Normal sample (VAE)
        normal_original = X_scaled_df.iloc[selected_normal_idx[i], feature_indices_to_plot].values
        normal_reconstructed = reconstructions[selected_normal_idx[i], feature_indices_to_plot]

        # Anomalous sample (VAE)
        anomaly_original = X_scaled_df.iloc[selected_anomaly_idx[i], feature_indices_to_plot].values
        anomaly_reconstructed = reconstructions[selected_anomaly_idx[i], feature_indices_to_plot]

        for j, feature_idx in enumerate(feature_indices_to_plot):
            feature_name = X_scaled_df.columns[feature_idx]
            axes[j].plot([0, 1], [normal_original[j], normal_reconstructed[j]], 'bo-', label='Normal (VAE)')
            axes[j].plot(1, normal_reconstructed[j], 'b^', markersize=8)
            axes[j].plot([0, 1], [anomaly_original[j], anomaly_reconstructed[j]], 'ro-', label='Anomaly (VAE)')
            axes[j].plot(1, anomaly_reconstructed[j], 'r^', markersize=8)
            axes[j].set_xticks([0, 1])
            axes[j].set_xticklabels(['Original', 'Reconstructed'])
            axes[j].set_title(f'{feature_name}')
            axes[j].grid(True)
            if j == 0:
                axes[j].set_ylabel('Scaled Feature Value')

        handles, labels = axes[0].get_legend_handles_labels()
        unique_labels = {l: h for h, l in zip(handles, labels) if l not in unique_labels}
        fig.suptitle(f"VAE Sample Reconstructions - Sample {i+1}", y=1.02)
        fig.legend(unique_labels.values(), unique_labels.keys(), loc='upper right')
        plt.tight_layout()
        plt.savefig(f'vae_reconstruction_sample_{i+1}_baseline.png')
        plt.show()
        plt.close()
else:
    print("\nNot enough normal or anomalous samples (based on VAE predictions) to visualize reconstructions.")
    print(f"Normal samples found: {len(normal_indices)}, Anomalous samples found: {len(anomaly_indices)}")

# Optional: Save results for later comparison
results_df = pd.DataFrame({
    'iso_forest_prediction': y_pred_iso,
    'iso_forest_score': scores_iso,
    'vae_prediction': y_pred_vae,
    'vae_score': reconstruction_error_vae,
    'vae_mad_prediction': y_pred_mad,
    'vae_mad_score': mad_scores
})
results_df.to_csv('baseline_anomaly_results.csv', index=False)
print("\nBaseline results saved to 'baseline_anomaly_results.csv'")

In [ ]:
all plots before injection isolation,VAE,VAE+MAD

Hybrid Approach using isolation forest before the encoders

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.ensemble import IsolationForest

# Assume X_scaled_df, vae, vae_encoder_model, vae_decoder_model, and feature_weights are available as pre-trained

# Dictionary to store predictions and scores for each model
results = {}

# --- Isolation Forest ---
print("\n--- Isolation Forest ---")
iso_forest = IsolationForest(contamination=0.05, random_state=42)
iso_forest.fit(X_scaled_df)
y_pred_iso = iso_forest.predict(X_scaled_df)
scores_iso = -iso_forest.score_samples(X_scaled_df)  # Lower is more anomalous
n_anomalies_iso = np.sum(y_pred_iso == -1)
print(f"Detected anomalies: {n_anomalies_iso} out of {len(X_scaled_df)} samples")
print(f"Percentage of anomalies: {n_anomalies_iso / len(X_scaled_df) * 100:.2f}%")

# Visualize Isolation Forest scores
plt.figure(figsize=(8, 5))
sns.histplot(scores_iso, bins=50, kde=True, color='skyblue')
plt.title("Isolation Forest Anomaly Score Distribution (Before Injection)")
plt.xlabel("Anomaly Score (Lower is more anomalous)")
plt.ylabel("Frequency")
plt.tight_layout()
plt.savefig('iso_forest_baseline_scores.png')
plt.show()
plt.close()

# Store predictions and scores
X_scaled_df_temp = X_scaled_df.copy()
X_scaled_df_temp['iso_forest_prediction'] = y_pred_iso
X_scaled_df_temp['iso_forest_score'] = scores_iso
print("\nTop 10 most anomalous points according to Isolation Forest:")
print(X_scaled_df_temp.sort_values(by='iso_forest_score').head(10))
results['Isolation Forest'] = (y_pred_iso, scores_iso)

# --- VAE ---
print("\n--- VAE ---")
reconstructions_vae = vae.predict(X_scaled_df, verbose=0)
reconstruction_error_vae = np.mean((X_scaled_df - reconstructions_vae) ** 2, axis=1)
vae_threshold = np.percentile(reconstruction_error_vae, 95)
y_pred_vae = (reconstruction_error_vae > vae_threshold).astype(int)
n_anomalies_vae = np.sum(y_pred_vae == 1)
print(f"Detected anomalies: {n_anomalies_vae} out of {len(X_scaled_df)} samples")
print(f"Percentage of anomalies: {n_anomalies_vae / len(X_scaled_df) * 100:.2f}%")

# Visualize VAE reconstruction errors
plt.figure(figsize=(8, 5))
sns.histplot(reconstruction_error_vae, bins=50, kde=True, color='orange')
plt.title("VAE Reconstruction Error Distribution (Before Injection)")
plt.xlabel("Reconstruction Error (Higher is more anomalous)")
plt.ylabel("Frequency")
plt.tight_layout()
plt.savefig('vae_baseline_scores.png')
plt.show()
plt.close()

# Store predictions and scores
X_scaled_df_temp['vae_prediction'] = y_pred_vae
X_scaled_df_temp['vae_score'] = reconstruction_error_vae
print("\nTop 10 most anomalous points according to VAE:")
print(X_scaled_df_temp.sort_values(by='vae_score', ascending=False).head(10))
results['VAE'] = (y_pred_vae, reconstruction_error_vae)

# --- VAE + MAD ---
print("\n--- VAE + MAD ---")
reconstruction_error_mad = np.mean(feature_weights * (X_scaled_df - reconstructions_vae) ** 2, axis=1)  # Reuse reconstructions
def mad_score(data):
    median = np.median(data)
    mad = np.median(np.abs(data - median))
    if mad == 0: mad = 1e-10
    return np.abs(data - median) / mad
mad_scores = mad_score(reconstruction_error_mad)
mad_threshold = np.percentile(mad_scores, 95)
y_pred_mad = (mad_scores > mad_threshold).astype(int)
n_anomalies_mad = np.sum(y_pred_mad == 1)
print(f"Detected anomalies: {n_anomalies_mad} out of {len(X_scaled_df)} samples")
print(f"Percentage of anomalies: {n_anomalies_mad / len(X_scaled_df) * 100:.2f}%")

# Visualize VAE + MAD MAD scores
plt.figure(figsize=(8, 5))
sns.histplot(mad_scores, bins=50, kde=True, color='green')
plt.title("VAE + MAD MAD Score Distribution (Before Injection)")
plt.xlabel("MAD Score (Higher is more anomalous)")
plt.ylabel("Frequency")
plt.tight_layout()
plt.savefig('vae_mad_baseline_scores.png')
plt.show()
plt.close()

# Store predictions and scores
X_scaled_df_temp['vae_mad_prediction'] = y_pred_mad
X_scaled_df_temp['vae_mad_score'] = mad_scores
print("\nTop 10 most anomalous points according to VAE + MAD:")
print(X_scaled_df_temp.sort_values(by='vae_mad_score', ascending=False).head(10))
results['VAE + MAD'] = (y_pred_mad, mad_scores)

# --- Visualizations Inspired by VAE Code ---

# 1. Feature-wise Reconstruction Errors (Bar Plot for VAE and VAE + MAD)
feature_errors_vae = np.mean(np.square(X_scaled_df - reconstructions_vae), axis=0)
plt.figure(figsize=(12, 6))
sns.barplot(x=X_scaled_df.columns, y=feature_errors_vae)
plt.xticks(rotation=90)
plt.title("VAE Mean Squared Reconstruction Error per Feature (Before Injection)")
plt.ylabel("Mean Squared Error")
plt.tight_layout()
plt.savefig('vae_feature_errors_baseline.png')
plt.show()
plt.close()

# 2. Latent Space Visualization with PCA (for VAE and VAE + MAD)
latent_representation_vae = vae_encoder_model.predict(X_scaled_df, verbose=0)[0]  # Using z_mean
pca_latent = PCA(n_components=2)
latent_pca = pca_latent.fit_transform(latent_representation_vae)
plt.figure(figsize=(10, 8))
plt.scatter(latent_pca[y_pred_vae == 0, 0], latent_pca[y_pred_vae == 0, 1], c='skyblue', label='Normal (VAE)', alpha=0.5, s=10)
plt.scatter(latent_pca[y_pred_vae == 1, 0], latent_pca[y_pred_vae == 1, 1], c='red', label='Anomaly (VAE)', alpha=0.7, s=20)
plt.scatter(latent_pca[y_pred_mad == 1, 0], latent_pca[y_pred_mad == 1, 1], c='green', label='Anomaly (VAE + MAD)', alpha=0.7, s=20, marker='^')
plt.title("VAE Latent Space Visualization (PCA) - Before Injection")
plt.xlabel("PCA Component 1")
plt.ylabel("PCA Component 2")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig('vae_latent_pca_baseline.png')
plt.show()
plt.close()

# 3. t-SNE for Latent Space (for VAE and VAE + MAD)
tsne = TSNE(n_components=2, random_state=42)
latent_tsne = tsne.fit_transform(latent_representation_vae)
plt.figure(figsize=(10, 8))
plt.scatter(latent_tsne[y_pred_vae == 0, 0], latent_tsne[y_pred_vae == 0, 1], c='skyblue', label='Normal (VAE)', alpha=0.5, s=10)
plt.scatter(latent_tsne[y_pred_vae == 1, 0], latent_tsne[y_pred_vae == 1, 1], c='red', label='Anomaly (VAE)', alpha=0.7, s=20)
plt.scatter(latent_tsne[y_pred_mad == 1, 0], latent_tsne[y_pred_mad == 1, 1], c='green', label='Anomaly (VAE + MAD)', alpha=0.7, s=20, marker='^')
plt.title("VAE Latent Space Visualization (t-SNE) - Before Injection")
plt.xlabel("t-SNE 1")
plt.ylabel("t-SNE 2")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig('vae_latent_tsne_baseline.png')
plt.show()
plt.close()

# 4. t-SNE for Isolation Forest Input Space
tsne_iso = TSNE(n_components=2, random_state=42)
input_tsne_iso = tsne_iso.fit_transform(X_scaled_df)
plt.figure(figsize=(10, 8))
plt.scatter(input_tsne_iso[y_pred_iso == 1, 0], input_tsne_iso[y_pred_iso == 1, 1], c='skyblue', label='Normal (Isolation Forest)', alpha=0.5, s=10)
plt.scatter(input_tsne_iso[y_pred_iso == -1, 0], input_tsne_iso[y_pred_iso == -1, 1], c='red', label='Anomaly (Isolation Forest)', alpha=0.7, s=20)
plt.title("Isolation Forest Input Space Visualization (t-SNE) - Before Injection")
plt.xlabel("t-SNE 1")
plt.ylabel("t-SNE 2")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig('iso_input_tsne_baseline.png')
plt.show()
plt.close()

# 5. Sample Reconstruction Comparison (VAE and VAE + MAD)
normal_indices = np.where(y_pred_vae == 0)[0]
anomaly_indices = np.where(y_pred_vae == 1)[0]
n_display = 5
if len(normal_indices) >= n_display and len(anomaly_indices) >= n_display:
    selected_normal_idx = normal_indices[:n_display]
    selected_anomaly_idx = anomaly_indices[:n_display]
    feature_indices_to_plot = [X_scaled_df.columns.get_loc(col) for col in ['Gage', 'Warp62', 'Crosslevel']]  # Example features

    print(f"\nVisualizing VAE reconstructions for {n_display} normal and {n_display} anomalous samples:")

    for i in range(n_display):
        fig, axes = plt.subplots(1, len(feature_indices_to_plot), figsize=(15, 4), sharey=True)

        # Normal sample (VAE)
        normal_original = X_scaled_df.iloc[selected_normal_idx[i], feature_indices_to_plot].values
        normal_reconstructed = reconstructions[selected_normal_idx[i], feature_indices_to_plot]

        # Anomalous sample (VAE)
        anomaly_original = X_scaled_df.iloc[selected_anomaly_idx[i], feature_indices_to_plot].values
        anomaly_reconstructed = reconstructions[selected_anomaly_idx[i], feature_indices_to_plot]

        for j, feature_idx in enumerate(feature_indices_to_plot):
            feature_name = X_scaled_df.columns[feature_idx]
            axes[j].plot([0, 1], [normal_original[j], normal_reconstructed[j]], 'bo-', label='Normal (VAE)')
            axes[j].plot(1, normal_reconstructed[j], 'b^', markersize=8)
            axes[j].plot([0, 1], [anomaly_original[j], anomaly_reconstructed[j]], 'ro-', label='Anomaly (VAE)')
            axes[j].plot(1, anomaly_reconstructed[j], 'r^', markersize=8)
            axes[j].set_xticks([0, 1])
            axes[j].set_xticklabels(['Original', 'Reconstructed'])
            axes[j].set_title(f'{feature_name}')
            axes[j].grid(True)
            if j == 0:
                axes[j].set_ylabel('Scaled Feature Value')

        handles, labels = axes[0].get_legend_handles_labels()
        unique_labels = {l: h for h, l in zip(handles, labels) if l not in unique_labels}
        fig.suptitle(f"VAE Sample Reconstructions - Sample {i+1}", y=1.02)
        fig.legend(unique_labels.values(), unique_labels.keys(), loc='upper right')
        plt.tight_layout()
        plt.savefig(f'vae_reconstruction_sample_{i+1}_baseline.png')
        plt.show()
        plt.close()
else:
    print("\nNot enough normal or anomalous samples (based on VAE predictions) to visualize reconstructions.")
    print(f"Normal samples found: {len(normal_indices)}, Anomalous samples found: {len(anomaly_indices)}")

# Optional: Save results for later comparison
results_df = pd.DataFrame({
    'iso_forest_prediction': y_pred_iso,
    'iso_forest_score': scores_iso,
    'vae_prediction': y_pred_vae,
    'vae_score': reconstruction_error_vae,
    'vae_mad_prediction': y_pred_mad,
    'vae_mad_score': mad_scores
})
results_df.to_csv('baseline_anomaly_results.csv', index=False)
print("\nBaseline results saved to 'baseline_anomaly_results.csv'")

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import backend as K
from tensorflow.keras.layers import Input, Dense, Lambda, Layer, Dropout, BatchNormalization
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.ensemble import IsolationForest # Import IsolationForest
import pandas as pd
from sklearn.preprocessing import StandardScaler


# Assuming X_scaled_df is already defined from previous cells.
# If not, make sure to execute the data loading, preprocessing, and scaling steps first.
# Example (assuming 'df' is your processed DataFrame before scaling):
# scaler = StandardScaler()
# X_scaled = scaler.fit_transform(df)
# X_scaled_df = pd.DataFrame(X_scaled, columns=df.columns)


# Configuration
input_dim = X_scaled_df.shape[1]  # 14 features
encoding_dim = 7  # Latent dimension for both models
batch_size = 64
validation_split = 0.2
dropout_rate = 0.2

# Feature weights for safety-critical features
# Make sure feature_weights is defined if you want to use it for scoring
# This definition is outside the function and should be available
feature_weights = np.array([1.5 if 'Gage' in col or 'Align' in col else 1.0 for col in X_scaled_df.columns])

# ========================
# KL Divergence Layer
# ========================
class KLDivergenceLayer(Layer):
    def call(self, inputs):
        z_mean, z_log_var = inputs
        kl_loss = -0.5 * (1 + z_log_var - tf.square(z_mean) - tf.exp(z_log_var))
        kl_loss = tf.reduce_sum(kl_loss, axis=-1)
        self.add_loss(tf.reduce_mean(kl_loss))
        return z_mean, z_log_var

# ========================
# VAE Config
# ========================
# Use the dimensions from your data (X_scaled_df)
# Reusing input_dim, encoding_dim, batch_size, validation_split, dropout_rate

# ========================
# Encoder
# ========================
def build_vae_encoder(input_dim, latent_dim, intermediate_dim, dropout_rate):
    inputs = Input(shape=(input_dim,), name='encoder_input')
    h = Dense(64, activation='relu')(inputs)
    h = BatchNormalization()(h)
    h = Dropout(dropout_rate)(h)
    h = Dense(intermediate_dim, activation='relu')(h)
    h = Dense(32, activation='relu')(h)
    h = Dense(16, activation='relu')(h)

    z_mean = Dense(latent_dim, name='z_mean')(h)
    z_log_var = Dense(latent_dim, name='z_log_var')(h)

    # Reparameterization trick
    def sampling(args):
        z_mean, z_log_var = args
        batch = tf.shape(z_mean)[0]
        dim = tf.shape(z_mean)[1]
        epsilon = tf.keras.backend.random_normal(shape=(batch, dim))
        return z_mean + tf.exp(0.5 * z_log_var) * epsilon

    z = Lambda(sampling, output_shape=(latent_dim,), name='z_sampling')([z_mean, z_log_var])

    return Model(inputs, [z_mean, z_log_var, z], name='encoder')

# ========================
# Decoder
# ========================
def build_vae_decoder(latent_dim, input_dim, dropout_rate):
    decoder_input = Input(shape=(latent_dim,), name='decoder_input')
    decoder_h = Dense(16, activation='relu')(decoder_input)
    decoder_h = Dense(32, activation='relu')(decoder_h)
    decoder_h = Dropout(dropout_rate)(decoder_h)
    decoder_h = BatchNormalization()(decoder_h)
    x_decoded_mean = Dense(input_dim, activation='linear', name='decoder_output')(decoder_h)

    return Model(decoder_input, x_decoded_mean, name='decoder')

# ========================
# VAE Model (combined)
# ========================
class VAE(Model):
    def __init__(self, encoder, decoder, **kwargs):
        super().__init__(**kwargs)
        self.encoder = encoder
        self.decoder = decoder
        self.total_loss_tracker = tf.keras.metrics.Mean(name="total_loss")
        self.reconstruction_loss_tracker = tf.keras.metrics.Mean(name="reconstruction_loss")
        self.kl_loss_tracker = tf.keras.metrics.Mean(name="kl_loss")

    @property
    def metrics(self):
        return [
            self.total_loss_tracker,
            self.reconstruction_loss_tracker,
            self.kl_loss_tracker,
        ]

    def train_step(self, data):
        # Unpack data as (x, y), where y is the target (x in autoencoders/VAEs)
        x, y = data

        with tf.GradientTape() as tape:
            # Forward pass
            z_mean, z_log_var, z = self.encoder(x)
            reconstruction = self.decoder(z)

            # Calculate reconstruction loss (MSE)
            # Sum over all dimensions except the batch dimension (axis 1 for 2D data)
            reconstruction_loss = tf.reduce_mean(
                tf.reduce_sum(
                    tf.square(y - reconstruction), axis=1 # Use y for target
                )
            )

            # Calculate KL divergence loss
            # -0.5 * sum(1 + log(sigma^2) - mu^2 - sigma^2)
            kl_loss = -0.5 * tf.reduce_mean(
                tf.reduce_sum(
                    1 + z_log_var - tf.square(z_mean) - tf.exp(z_log_var), axis=1
                )
            )

            # Total VAE loss
            total_loss = reconstruction_loss + kl_loss

        # Apply gradients
        grads = tape.gradient(total_loss, self.trainable_variables)
        self.optimizer.apply_gradients(zip(grads, self.trainable_variables))

        # Update metrics
        self.total_loss_tracker.update_state(total_loss)
        self.reconstruction_loss_tracker.update_state(reconstruction_loss)
        self.kl_loss_tracker.update_state(kl_loss)

        # Return metrics as a dictionary
        return {
            "loss": self.total_loss_tracker.result(),
            "reconstruction_loss": self.reconstruction_loss_tracker.result(),
            "kl_loss": self.kl_loss_tracker.result(),
        }

    def call(self, inputs):
         # Call is typically used for inference, not training step
         # For inference, we usually just want the reconstruction
         z_mean, z_log_var, z = self.encoder(inputs)
         reconstruction = self.decoder(z)
         return reconstruction


# ========================
# Hybrid Approach
# ========================

# Pre-filter with Isolation Forest (assuming X_scaled_df is available)
iso_forest = IsolationForest(contamination=0.05, random_state=42)
iso_forest.fit(X_scaled_df)
y_pred_iso = iso_forest.predict(X_scaled_df)
normal_mask = (y_pred_iso == 1)  # Keep only normal samples
X_normal = X_scaled_df[normal_mask].values
print(f"Normal samples after pre-filtering with Isolation Forest: {X_normal.shape[0]}")

# --- Train Hybrid Autoencoder ---
# (Keep this section as it was, assuming build_autoencoder is defined)
def build_autoencoder():
    input_layer = Input(shape=(input_dim,))
    x = Dense(64, activation='relu')(input_layer)
    x = BatchNormalization()(x)
    x = Dropout(dropout_rate)(x)
    x = Dense(32, activation='relu')(x)
    x = Dense(16, activation='relu')(x)
    encoded = Dense(encoding_dim, activation='relu', name='encoder_output')(x)
    x = Dense(16, activation='relu')(encoded)
    x = Dense(32, activation='relu')(x)
    x = Dropout(dropout_rate)(x)
    x = BatchNormalization()(x)
    decoded = Dense(input_dim, activation='linear')(x)
    autoencoder = Model(inputs=input_layer, outputs=decoded)
    autoencoder.compile(optimizer='adam', loss='mse')
    return autoencoder

autoencoder_hybrid = build_autoencoder()
history_auto = autoencoder_hybrid.fit(
    X_normal, X_normal, # Train AE on normal data, input is also the target
    epochs=100,
    batch_size=batch_size,
    validation_split=validation_split,
    callbacks=[EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)],
    verbose=1
)


# --- Train Hybrid VAE ---
# Build encoder and decoder models
vae_encoder = build_vae_encoder(input_dim, encoding_dim, 32, dropout_rate) # Using encoding_dim for latent_dim
vae_decoder = build_vae_decoder(encoding_dim, input_dim, dropout_rate)

# Instantiate the VAE Model subclass
vae_hybrid = VAE(vae_encoder, vae_decoder)
# Compile the VAE subclass model. Add a loss function here.
# The train_step calculates the actual loss, but compile expects one.
vae_hybrid.compile(optimizer='adam', loss='mse') # Added loss='mse' here

# Train the VAE model on X_normal
history_vae = vae_hybrid.fit(
    X_normal,
    X_normal,  # Explicitly pass X_normal as the target for train_step (x, y) tuple
    epochs=100,
    batch_size=batch_size,
    validation_split=validation_split,
    callbacks=[EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)],
    verbose=1
)

# Anomaly Scoring for Autoencoder
auto_reconstructions = autoencoder_hybrid.predict(X_scaled_df, verbose=0)
# Using squared error for consistency unless you explicitly define weights for scoring
auto_errors = np.square(X_scaled_df - auto_reconstructions)
auto_mse = np.mean(auto_errors, axis=1)
# Use a fixed threshold like before, based on percentiles of *this* data (before splitting, after scaling)
auto_threshold = np.percentile(auto_mse, 95)  # Example: top 5% errors as anomalies
auto_anomalies = (auto_mse > auto_threshold).astype(int)
auto_detected = np.sum(auto_anomalies)
auto_percentage = (auto_detected / len(X_scaled_df)) * 100


print("\n=== Hybrid Autoencoder Anomaly Detection (Pre-Injection) ===")
print(f"Threshold: {auto_threshold:.4f}")
print(f"Detected anomalies: {auto_detected} out of {len(X_scaled_df)} samples")
print(f"Percentage of detected anomalies: {auto_percentage:.2f}%")

# Anomaly Scoring for VAE
# To calculate the anomaly score for the *entire* dataset (X_scaled_df),
# we need the latent space parameters and reconstructions for X_scaled_df.
z_mean_all, z_log_var_all, z_all = vae_hybrid.encoder.predict(X_scaled_df, verbose=0)
vae_reconstructions_all = vae_hybrid.decoder.predict(z_all, verbose=0)

# Calculate per-sample Reconstruction Error (using MSE per sample)
vae_reconstruction_errors_per_sample = np.mean(np.square(X_scaled_df - vae_reconstructions_all), axis=1)

# Calculate per-sample KL divergence: -0.5 * sum(1 + log(sigma^2) - mu^2 - sigma^2) over latent dimensions
vae_kl_errors_per_sample = -0.5 * np.sum(1 + z_log_var_all - np.square(z_mean_all) - np.exp(z_log_var_all), axis=1)

# Total VAE anomaly score is the sum of per-sample reconstruction error and per-sample KL divergence
# This represents the total loss for each sample in the dataset.
vae_anomaly_scores = vae_reconstruction_errors_per_sample + vae_kl_errors_per_sample

# Use a threshold based on the data distribution (e.g., percentile)
vae_threshold = np.percentile(vae_anomaly_scores, 95)  # Example: top 5% errors as anomalies
vae_anomalies = (vae_anomaly_scores > vae_threshold).astype(int)
vae_detected = np.sum(vae_anomalies)
vae_percentage = (vae_detected / len(X_scaled_df)) * 100

print("\n=== Hybrid VAE Anomaly Detection (Pre-Injection) ===")
print(f"Threshold: {vae_threshold:.4f}")
print(f"Detected anomalies: {vae_detected} out of {len(X_scaled_df)} samples")
print(f"Percentage of detected anomalies: {vae_percentage:.2f}%")


# Visualizations (Update visualizations to use the new VAE anomaly score)
# 1. Weighted MSE Distribution
# Autoencoder plot remains the same
plt.figure(figsize=(10, 6))
sns.histplot(auto_mse, bins=50, kde=True, color='blue', alpha=0.6, label='Hybrid Autoencoder MSE')
plt.axvline(auto_threshold, color='red', linestyle='--', label=f'Autoencoder Threshold = {auto_threshold:.4f}')
plt.title("Hybrid Autoencoder Weighted MSE Distribution (Pre-Injection)")
plt.xlabel("Weighted Reconstruction Error")
plt.ylabel("Frequency")
plt.legend()
plt.tight_layout()
plt.show()

# VAE Anomaly Score Distribution
plt.figure(figsize=(10, 6))
sns.histplot(vae_anomaly_scores, bins=50, kde=True, color='green', alpha=0.6, label='Hybrid VAE Anomaly Score')
plt.axvline(vae_threshold, color='orange', linestyle='--', label=f'VAE Threshold = {vae_threshold:.4f}')
plt.title("Hybrid VAE Anomaly Score Distribution (Pre-Injection)")
plt.xlabel("VAE Anomaly Score (Reconstruction Error + KL Divergence)")
plt.ylabel("Frequency")
plt.legend()
plt.tight_layout()
plt.show()


# 2. Reconstruction Error per Feature
auto_feature_errors = np.mean(np.square(X_scaled_df - auto_reconstructions), axis=0)
# VAE reconstruction error per feature
vae_feature_errors = np.mean(np.square(X_scaled_df - vae_reconstructions_all), axis=0)

plt.figure(figsize=(12, 6))
bar_width = 0.35
x_pos = np.arange(len(X_scaled_df.columns))
plt.bar(x_pos, auto_feature_errors, bar_width, color='blue', alpha=0.6, label='Hybrid Autoencoder')
plt.bar(x_pos + bar_width, vae_feature_errors, bar_width, color='green', alpha=0.6, label='Hybrid VAE')
plt.xticks(x_pos + bar_width / 2, X_scaled_df.columns, rotation=90)
plt.title("Mean Squared Reconstruction Error per Feature (Hybrid Models)")
plt.ylabel("Mean Squared Error")
plt.legend()
plt.tight_layout()
plt.show()


# 3. Latent Space Visualization
# Autoencoder Latent Space remains the same
auto_encoder_model = Model(inputs=autoencoder_hybrid.input, outputs=autoencoder_hybrid.get_layer('encoder_output').output)
auto_latent = auto_encoder_model.predict(X_scaled_df, verbose=0)
auto_pca = PCA(n_components=2).fit_transform(auto_latent)
plt.figure(figsize=(10, 8))
plt.scatter(auto_pca[:, 0][auto_anomalies == 0], auto_pca[:, 1][auto_anomalies == 0],
            c='skyblue', label='Normal (Predicted)', alpha=0.5, s=10)
plt.scatter(auto_pca[:, 0][auto_anomalies == 1], auto_pca[:, 1][auto_anomalies == 1],
            c='red', label='Anomaly (Predicted)', alpha=0.7, s=20)
plt.title("Hybrid Autoencoder Latent Space Visualization (PCA)")
plt.xlabel("PCA Component 1")
plt.ylabel("PCA Component 2")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

# VAE Latent Space (using z_mean)
vae_pca = PCA(n_components=2).fit_transform(z_mean_all) # Use z_mean_all
plt.figure(figsize=(10, 8))
plt.scatter(vae_pca[:, 0][vae_anomalies == 0], vae_pca[:, 1][vae_anomalies == 0],
            c='skyblue', label='Normal (Predicted)', alpha=0.5, s=10)
plt.scatter(vae_pca[:, 0][vae_anomalies == 1], vae_pca[:, 1][vae_anomalies == 1],
            c='red', label='Anomaly (Predicted)', alpha=0.7, s=20)
plt.title("Hybrid VAE Latent Space Visualization (PCA of z_mean)")
plt.xlabel("PCA Component 1")
plt.ylabel("PCA Component 2")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

# 4. Sample Reconstruction Comparison
# Select indices of a few normal samples and a few anomalous samples based on VAE's predictions
normal_indices_vae = np.where(vae_anomalies == 0)[0]
anomaly_indices_vae = np.where(vae_anomalies == 1)[0]

# Choose a few samples (e.g., first 5 normal, first 5 anomalous)
n_display = 5
# Assuming X_scaled_df columns are available
feature_indices_to_plot = [X_scaled_df.columns.get_loc(col) for col in ['Gage', 'Warp62', 'Crosslevel']] # Example features


if len(normal_indices_vae) > n_display and len(anomaly_indices_vae) > n_display:
    selected_normal_idx_vae = normal_indices_vae[:n_display]
    selected_anomaly_idx_vae = anomaly_indices_vae[:n_display]

    print(f"\nVisualizing VAE reconstructions for {n_display} normal and {n_display} anomalous samples:")

    for i in range(n_display):
        fig, axes = plt.subplots(1, len(feature_indices_to_plot), figsize=(15, 4), sharey=True)

        # Normal sample
        normal_original = X_scaled_df.iloc[selected_normal_idx_vae[i], feature_indices_to_plot].values
        normal_reconstructed = vae_reconstructions_all[selected_normal_idx_vae[i], feature_indices_to_plot]

        # Anomalous sample
        anomaly_original = X_scaled_df.iloc[selected_anomaly_idx_vae[i], feature_indices_to_plot].values
        anomaly_reconstructed = vae_reconstructions_all[selected_anomaly_idx_vae[i], feature_indices_to_plot]

        for j, feature_idx in enumerate(feature_indices_to_plot):
            feature_name = X_scaled_df.columns[feature_idx]

            axes[j].plot([0, 1], [normal_original[j], normal_reconstructed[j]], 'bo-', label='Original')
            axes[j].plot([0, 1], [normal_reconstructed[j], normal_reconstructed[j]], 'b^', markersize=8)
            axes[j].plot([0, 1], [anomaly_original[j], anomaly_reconstructed[j]], 'ro-', label='Anomaly')
            axes[j].plot([0, 1], [anomaly_reconstructed[j], anomaly_reconstructed[j]], 'r^', markersize=8)

            axes[j].set_xticks([0, 1])
            axes[j].set_xticklabels(['Original', 'Reconstructed'])
            axes[j].set_title(f'{feature_name}')
            axes[j].grid(True)
            if j == 0:
                 axes[j].set_ylabel('Scaled Feature Value')

        handles, labels = axes[0].get_legend_handles_labels()
        unique_labels = {}
        for h, l in zip(handles, labels):
            if l not in unique_labels:
                unique_labels[l] = h
        fig.suptitle(f"VAE Sample Reconstructions - Sample {i+1}", y=1.02)
        fig.legend(unique_labels.values(), unique_labels.keys(), loc='upper right')
        plt.tight_layout()
        plt.show()
else:
    print("\nNot enough normal or anomalous samples (based on VAE predictions) to visualize reconstructions.")
    print(f"Normal samples found: {len(normal_indices_vae)}, Anomalous samples found: {len(anomaly_indices_vae)}")

plot of feature reconstruction

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA # Ensure PCA is imported

# Assuming all models (autoencoder, vae, autoencoder_hybrid, vae_hybrid) are already trained
# and X_scaled_df is the full scaled dataset

# Assuming vae_encoder_model was created earlier as the encoder part of the 'vae' model
# If not, create it:
# vae_encoder_model = Model(inputs=vae.input, outputs=vae.get_layer('encoder_output').output)
# Make sure 'encoder_output' is the name of the Dense layer outputting z_mean in your VAE definition.

# 1. Standalone Autoencoder Reconstruction Error
auto_reconstructions_standalone = autoencoder.predict(X_scaled_df, verbose=0)
auto_errors_standalone = np.square(X_scaled_df - auto_reconstructions_standalone) # Remove feature_weights multiplication here for fair comparison with VAE MSE unless explicitly needed
auto_mse_standalone = np.mean(auto_errors_standalone, axis=0)

# 2. Standalone VAE Reconstruction Error
# Use the standalone VAE model for reconstruction prediction
vae_reconstructions_standalone = vae.predict(X_scaled_df, verbose=0)
vae_errors_standalone = np.square(X_scaled_df - vae_reconstructions_standalone) # Remove feature_weights multiplication here
vae_mse_standalone = np.mean(vae_errors_standalone, axis=0)


# 3. Hybrid Autoencoder Reconstruction Error
auto_reconstructions_hybrid = autoencoder_hybrid.predict(X_scaled_df, verbose=0)
auto_errors_hybrid = np.square(X_scaled_df - auto_reconstructions_hybrid) # Remove feature_weights multiplication here
auto_mse_hybrid = np.mean(auto_errors_hybrid, axis=0)

# 4. Hybrid VAE Reconstruction Error
# The vae_hybrid model (custom VAE class) has encoder and decoder attributes
# vae_hybrid.predict() directly gives the reconstruction, as defined in its call method
vae_reconstructions_hybrid = vae_hybrid.predict(X_scaled_df, verbose=0)
vae_errors_hybrid = np.square(X_scaled_df - vae_reconstructions_hybrid) # Remove feature_weights multiplication here
vae_mse_hybrid = np.mean(vae_errors_hybrid, axis=0)

# Plotting
plt.figure(figsize=(12, 6))
bar_width = 0.2
x = np.arange(len(X_scaled_df.columns))

plt.bar(x - 1.5 * bar_width, auto_mse_standalone, bar_width, label='Standalone Autoencoder', color='blue', alpha=0.6)
plt.bar(x - 0.5 * bar_width, vae_mse_standalone, bar_width, label='Standalone VAE', color='green', alpha=0.6)
plt.bar(x + 0.5 * bar_width, auto_mse_hybrid, bar_width, label='Hybrid Autoencoder', color='orange', alpha=0.6)
plt.bar(x + 1.5 * bar_width, vae_mse_hybrid, bar_width, label='Hybrid VAE', color='red', alpha=0.6)

plt.xlabel('Features')
plt.ylabel('Mean Squared Reconstruction Error')
plt.title('Mean Squared Reconstruction Error per Feature Across Models')
plt.xticks(x, X_scaled_df.columns, rotation=90)
plt.legend()
plt.tight_layout()
plt.show()

Brief Discussion
Hybrid Impact: The hybrid approach (pre-filtering with Isolation Forest) appears to reduce reconstruction errors for some features (e.g., Gage, Warp62) in the Autoencoder, supporting the hypothesis that training on a normal subset refines the model. However, the VAE’s higher errors suggest the KL divergence might counteract this benefit.
Model Differences: The standalone VAE’s higher errors align with its probabilistic regularization, while the Autoencoder’s consistency reflects its deterministic nature. The hybrid VAE’s elevated errors could indicate over-regularization on the normal subset.
Novelty: This plot visually supports your hybrid approach’s novelty by showing how pre-filtering alters reconstruction, offering a comparative baseline for post-injection analysis.

Injection Version

In [ ]:
# Parameters
intrusion_len = 46
attack_windows = 20
n_samples = len(df)
spread = (n_samples - attack_windows * intrusion_len) // attack_windows
attack_points = [i * (spread + intrusion_len) for i in range(attack_windows)]

# Make a copy and add a label column
df_injected = df.copy()
df_injected['label'] = 0  # 0 = normal, 1 = injected anomaly

# Confirm configuration
print(f"Total samples: {n_samples}")
print(f"Attack windows: {attack_windows}")
print(f"Intrusion length: {intrusion_len}")
print(f"First attack point: {attack_points[0]}")
print(f"Expected total anomalies: {attack_windows * intrusion_len}")

# Perform the injection
for i, start in enumerate(attack_points):
    end = min(start + intrusion_len - 1, n_samples - 1)  # Safe boundary
    print(f"Window {i+1}: Modifying rows {start} to {end}")

    # Inject anomalies by modifying features
    df_injected.loc[start:end, 'Gage'] += 2.0
    df_injected.loc[start:end, 'Warp62'] *= 2.5
    df_injected.loc[start:end, 'Crosslevel'] -= 1.5
    df_injected.loc[start:end, 'label'] = 1

# Verify injection
total_injected = df_injected['label'].sum()
print("\n✅ Injection Summary:")
print(f"Total anomalies injected: {total_injected}")
print(f"Gage max (in anomalies): {df_injected[df_injected['label'] == 1]['Gage'].max():.2f}")
print(f"Warp62 max (in anomalies): {df_injected[df_injected['label'] == 1]['Warp62'].max():.2f}")
print(f"Crosslevel min (in anomalies): {df_injected[df_injected['label'] == 1]['Crosslevel'].min():.2f}")
print(f"Injection coverage: {100 * total_injected / n_samples:.2f}% of dataset")


isolation forest intrusion detection after injection

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import IsolationForest
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.preprocessing import StandardScaler
import seaborn as sns

# Assume df_injected is already in memory and includes a 'label' column (0=normal, 1=intrusion)
# Extract features and labels
X = df_injected.drop(columns=['label'])
y_true = df_injected['label'].values

# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Apply Isolation Forest
iso = IsolationForest(contamination=0.05, random_state=42)
y_pred = (iso.fit_predict(X_scaled) == -1).astype(int)  # Convert -1 to 1 (anomaly)

# Generate confusion matrix and classification report
cm = confusion_matrix(y_true, y_pred)
report = classification_report(y_true, y_pred, target_names=['Normal', 'Intrusion'], output_dict=True)

# Plot confusion matrix
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Reds', xticklabels=['Normal', 'Intrusion'], yticklabels=['Normal', 'Intrusion'])
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Isolation Forest Confusion Matrix (After Injection)")
plt.tight_layout()
plt.show()

report_df = pd.DataFrame(report).transpose()
print(report_df)


In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.metrics import confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA

# Assume df_injected, X_scaled_df, vae, vae_encoder_model, vae_decoder_model, and feature_weights are available

# Extract features and labels
X_injected = df_injected.drop(columns=['label'])
y_true = df_injected['label'].values

# Scale features
scaler = StandardScaler()
X_injected_scaled = scaler.fit_transform(X_injected)

# 1. Isolation Forest
iso = IsolationForest(contamination=0.05, random_state=42)
y_pred_iso = (iso.fit_predict(X_injected_scaled) == -1).astype(int)
cm_iso = confusion_matrix(y_true, y_pred_iso)
print("Isolation Forest Confusion Matrix:")
print(cm_iso)
print("\nIsolation Forest Classification Report:")
print(classification_report(y_true, y_pred_iso))

# 2. VAE
reconstructions_vae = vae.predict(X_injected_scaled)
reconstruction_error_vae = np.mean((X_injected_scaled - reconstructions_vae) ** 2, axis=1)
vae_threshold = np.percentile(reconstruction_error_vae, 95)
y_pred_vae = (reconstruction_error_vae > vae_threshold).astype(int)
cm_vae = confusion_matrix(y_true, y_pred_vae)
print("\nVAE Confusion Matrix:")
print(cm_vae)
print("\nVAE Classification Report:")
print(classification_report(y_true, y_pred_vae))

# 3. VAE + MAD
reconstructions_mad = vae.predict(X_injected_scaled)
reconstruction_error_mad = np.mean(feature_weights * (X_injected_scaled - reconstructions_mad) ** 2, axis=1)
def mad_score(data):
    median = np.median(data)
    mad = np.median(np.abs(data - median))
    if mad == 0: mad = 1e-10
    return np.abs(data - median) / mad
mad_scores = mad_score(reconstruction_error_mad)
mad_threshold = np.percentile(mad_scores, 95)
y_pred_mad = (mad_scores > mad_threshold).astype(int)
cm_mad = confusion_matrix(y_true, y_pred_mad)
print("\nVAE + MAD Confusion Matrix:")
print(cm_mad)
print("\nVAE + MAD Classification Report:")
print(classification_report(y_true, y_pred_mad))

# Visualizations (for all models)
models = {'Isolation Forest': y_pred_iso, 'VAE': y_pred_vae, 'VAE + MAD': y_pred_mad}
cms = {'Isolation Forest': cm_iso, 'VAE': cm_vae, 'VAE + MAD': cm_mad}
scores = {'Isolation Forest': -iso.score_samples(X_injected_scaled), 'VAE': reconstruction_error_vae, 'VAE + MAD': mad_scores}

for name, y_pred in models.items():
    # Confusion Matrix
    plt.figure(figsize=(6, 4))
    sns.heatmap(cms[name], annot=True, fmt='d', cmap='Blues', cbar=False)
    plt.title(f'Confusion Matrix ({name})')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.tight_layout()
    plt.savefig(f'{name.lower().replace(" ", "_")}_confusion_matrix.png')
    plt.show()
    plt.close()

    # Score Distribution
    plt.figure(figsize=(8, 5))
    sns.histplot(scores[name][y_true == 0], label='Normal', color='blue', kde=True)
    sns.histplot(scores[name][y_true == 1], label='Intrusion', color='red', kde=True)
    plt.title(f'{name} Score Distribution')
    plt.xlabel('Score' if name == 'Isolation Forest' else 'Reconstruction Error' if name == 'VAE' else 'MAD Score')
    plt.axvline(np.percentile(scores[name], 95), color='black', linestyle='--', label='Threshold')
    plt.legend()
    plt.tight_layout()
    plt.savefig(f'{name.lower().replace(" ", "_")}_scores.png')
    plt.show()
    plt.close()

# PCA for VAE latent space (shared across VAE and VAE + MAD)
latent_z = vae_encoder_model.predict(X_injected_scaled)[2]
pca = PCA(n_components=2)
latent_pca = pca.fit_transform(latent_z)
plt.figure(figsize=(8, 6))
plt.scatter(latent_pca[y_true == 0, 0], latent_pca[y_true == 0, 1], c='blue', label='Normal', alpha=0.5)
plt.scatter(latent_pca[y_true == 1, 0], latent_pca[y_true == 1, 1], c='red', label='Intrusion', alpha=0.5)
plt.title('PCA of Latent Space (VAE)')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.legend()
plt.tight_layout()
plt.savefig('vae_pca.png')
plt.show()
plt.close()

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score, roc_curve
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA

# Assume df_injected, X_scaled_df, vae, vae_encoder_model, vae_decoder_model, and feature_weights are available

# Extract features and labels
X_injected = df_injected.drop(columns=['label'])
y_true = df_injected['label'].values

# Scale features
scaler = StandardScaler()
X_injected_scaled = scaler.fit_transform(X_injected)

# 1. Isolation Forest
iso = IsolationForest(contamination=0.05, random_state=42)
y_pred_iso = (iso.fit_predict(X_injected_scaled) == -1).astype(int)
iso_scores = -iso.score_samples(X_injected_scaled)
cm_iso = confusion_matrix(y_true, y_pred_iso)
print("Isolation Forest Confusion Matrix:")
print(cm_iso)
print("\nIsolation Forest Classification Report:")
print(classification_report(y_true, y_pred_iso))
fpr_iso, tpr_iso, _ = roc_curve(y_true, iso_scores)
auc_iso = roc_auc_score(y_true, iso_scores)
print(f"ROC-AUC: {auc_iso:.3f}")

# 2. VAE
reconstructions_vae = vae.predict(X_injected_scaled)
reconstruction_error_vae = np.mean((X_injected_scaled - reconstructions_vae) ** 2, axis=1)
vae_threshold = np.percentile(reconstruction_error_vae, 95)
y_pred_vae = (reconstruction_error_vae > vae_threshold).astype(int)
cm_vae = confusion_matrix(y_true, y_pred_vae)
print("\nVAE Confusion Matrix:")
print(cm_vae)
print("\nVAE Classification Report:")
print(classification_report(y_true, y_pred_vae))
fpr_vae, tpr_vae, _ = roc_curve(y_true, reconstruction_error_vae)
auc_vae = roc_auc_score(y_true, reconstruction_error_vae)
print(f"ROC-AUC: {auc_vae:.3f}")

# 3. VAE + MAD
reconstructions_mad = vae.predict(X_injected_scaled)
reconstruction_error_mad = np.mean(feature_weights * (X_injected_scaled - reconstructions_mad) ** 2, axis=1)
def mad_score(data):
    median = np.median(data)
    mad = np.median(np.abs(data - median))
    if mad == 0: mad = 1e-10
    return np.abs(data - median) / mad
mad_scores = mad_score(reconstruction_error_mad)
mad_threshold = np.percentile(mad_scores, 95)
y_pred_mad = (mad_scores > mad_threshold).astype(int)
cm_mad = confusion_matrix(y_true, y_pred_mad)
print("\nVAE + MAD Confusion Matrix:")
print(cm_mad)
print("\nVAE + MAD Classification Report:")
print(classification_report(y_true, y_pred_mad))
fpr_mad, tpr_mad, _ = roc_curve(y_true, mad_scores)
auc_mad = roc_auc_score(y_true, mad_scores)
print(f"ROC-AUC: {auc_mad:.3f}")

# Visualizations
models = {'Isolation Forest': y_pred_iso, 'VAE': y_pred_vae, 'VAE + MAD': y_pred_mad}
cms = {'Isolation Forest': cm_iso, 'VAE': cm_vae, 'VAE + MAD': cm_mad}
scores = {'Isolation Forest': iso_scores, 'VAE': reconstruction_error_vae, 'VAE + MAD': mad_scores}
fprs = {'Isolation Forest': fpr_iso, 'VAE': fpr_vae, 'VAE + MAD': fpr_mad}
tprs = {'Isolation Forest': tpr_iso, 'VAE': tpr_vae, 'VAE + MAD': tpr_mad}

for name, y_pred in models.items():
    # Confusion Matrix
    plt.figure(figsize=(6, 4))
    sns.heatmap(cms[name], annot=True, fmt='d', cmap='Blues', cbar=False)
    plt.title(f'Confusion Matrix ({name})')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.tight_layout()
    plt.savefig(f'{name.lower().replace(" ", "_")}_confusion_matrix.png')
    plt.show()
    plt.close()

    # Score Distribution
    plt.figure(figsize=(8, 5))
    sns.histplot(scores[name][y_true == 0], label='Normal', color='blue', kde=True)
    sns.histplot(scores[name][y_true == 1], label='Intrusion', color='red', kde=True)
    plt.title(f'{name} Score Distribution')
    plt.xlabel('Anomaly Score' if name == 'Isolation Forest' else 'Reconstruction Error' if name == 'VAE' else 'MAD Score')
    plt.axvline(np.percentile(scores[name], 95), color='black', linestyle='--', label='Threshold')
    plt.legend()
    plt.tight_layout()
    plt.savefig(f'{name.lower().replace(" ", "_")}_scores.png')
    plt.show()
    plt.close()

# ROC Curve
plt.figure(figsize=(8, 6))
plt.plot(fprs['Isolation Forest'], tprs['Isolation Forest'], label=f'Isolation Forest (AUC = {auc_iso:.3f})')
plt.plot(fprs['VAE'], tprs['VAE'], label=f'VAE (AUC = {auc_vae:.3f})')
plt.plot(fprs['VAE + MAD'], tprs['VAE + MAD'], label=f'VAE + MAD (AUC = {auc_mad:.3f})')
plt.plot([0, 1], [0, 1], 'k--')
plt.title('ROC Curve Comparison')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.legend()
plt.tight_layout()
plt.savefig('roc_curve_comparison.png')
plt.show()
plt.close()

# PCA for VAE latent space
latent_z = vae_encoder_model.predict(X_injected_scaled)[2]
pca = PCA(n_components=2)
latent_pca = pca.fit_transform(latent_z)
plt.figure(figsize=(8, 6))
plt.scatter(latent_pca[y_true == 0, 0], latent_pca[y_true == 0, 1], c='blue', label='Normal', alpha=0.5)
plt.scatter(latent_pca[y_true == 1, 0], latent_pca[y_true == 1, 1], c='red', label='Intrusion', alpha=0.5)
plt.title('PCA of Latent Space (VAE)')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.legend()
plt.tight_layout()
plt.savefig('vae_pca.png')
plt.close()

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score, roc_curve, f1_score, recall_score
from sklearn.model_selection import KFold
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA

# Assume df_injected, vae, vae_encoder_model, vae_decoder_model, and feature_weights are available
# Extract features and labels
X_injected = df_injected.drop(columns=['label'])
y_true = df_injected['label'].values

# Scale features
scaler = StandardScaler()
X_injected_scaled = scaler.fit_transform(X_injected)

print("Data preparation completed. Shape of X_injected_scaled:", X_injected_scaled.shape)

In [ ]:
# Define model scoring functions
def iso_score_func(X_train, X_val, iso_model=None):
    if iso_model is None:
        iso_model = IsolationForest(contamination=0.05, random_state=42)
        iso_model.fit(X_train)
    else:
        iso_model.fit(X_train)
    return -iso_model.score_samples(X_val)

def vae_score_func(X_train, X_val, vae_model=None, feature_weights=None):
    if vae_model is None:
        raise ValueError("vae_model must be provided")
    reconstructions = vae_model.predict(X_val)
    if feature_weights is not None:
        return np.mean(feature_weights * (X_val - reconstructions) ** 2, axis=1)
    return np.mean((X_val - reconstructions) ** 2, axis=1)

def mad_score_func(X_train, X_val, vae_model=None, feature_weights=None):
    if vae_model is None:
        raise ValueError("vae_model must be provided")
    reconstructions = vae_model.predict(X_val)
    error = np.mean(feature_weights * (X_val - reconstructions) ** 2, axis=1) if feature_weights is not None else np.mean((X_val - reconstructions) ** 2, axis=1)
    median = np.median(error)
    mad = np.median(np.abs(error - median))
    if mad == 0:
        mad = 1e-10
    return np.abs(error - median) / mad

# Function to find optimal threshold via cross-validation
def find_optimal_threshold(X, y, model_func, n_splits=5, metric='f1', **model_kwargs):
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    thresholds = np.linspace(0, 1, 100)  # Percentiles from 0 to 100
    best_thresholds = {}

    for train_idx, val_idx in kf.split(X):
        X_train, X_val = X[train_idx], X[val_idx]
        y_train, y_val = y[train_idx], y[val_idx]  # Define y_val here

        # Get scores for the model
        scores = model_func(X_train, X_val, **model_kwargs)
        for thresh in thresholds:
            y_pred = (scores > np.percentile(scores, thresh * 100)).astype(int)
            if len(y_val) > 0 and len(y_pred) == len(y_val):  # Safety check
                if metric == 'f1':
                    score = f1_score(y_val, y_pred)
                elif metric == 'recall':
                    score = recall_score(y_val, y_pred)
                if thresh not in best_thresholds or score > best_thresholds[thresh]:
                    best_thresholds[thresh] = score

    optimal_thresh = max(best_thresholds, key=best_thresholds.get) * 100 if best_thresholds else 95
    return optimal_thresh

print("Functions defined successfully.")

In [ ]:
# Initialize models
iso = IsolationForest(contamination=0.05, random_state=42)

# Find optimal thresholds
iso_thresh_percentile = find_optimal_threshold(X_injected_scaled, y_true, iso_score_func, metric='f1', iso_model=iso)
vae_thresh_percentile = find_optimal_threshold(X_injected_scaled, y_true, vae_score_func, metric='f1', vae_model=vae, feature_weights=feature_weights)
mad_thresh_percentile = find_optimal_threshold(X_injected_scaled, y_true, mad_score_func, metric='f1', vae_model=vae, feature_weights=feature_weights)

print(f"Optimal Thresholds - Isolation Forest: {iso_thresh_percentile:.1f}%, VAE: {vae_thresh_percentile:.1f}%, VAE + MAD: {mad_thresh_percentile:.1f}%")

In [ ]:
# Generate predictions with optimal thresholds
y_pred_iso = (iso_score_func(X_injected_scaled, X_injected_scaled, iso_model=iso) > np.percentile(-iso.score_samples(X_injected_scaled), iso_thresh_percentile)).astype(int)
y_pred_vae = (vae_score_func(X_injected_scaled, X_injected_scaled, vae_model=vae, feature_weights=feature_weights) > np.percentile(reconstruction_error_vae, vae_thresh_percentile)).astype(int)
mad_scores = mad_score_func(X_injected_scaled, X_injected_scaled, vae_model=vae, feature_weights=feature_weights)
y_pred_mad = (mad_scores > np.percentile(mad_scores, mad_thresh_percentile)).astype(int)

# Confusion Matrices and Reports
cm_iso = confusion_matrix(y_true, y_pred_iso)
cm_vae = confusion_matrix(y_true, y_pred_vae)
cm_mad = confusion_matrix(y_true, y_pred_mad)

print("Isolation Forest Confusion Matrix:")
print(cm_iso)
print("\nIsolation Forest Classification Report:")
print(classification_report(y_true, y_pred_iso))

print("\nVAE Confusion Matrix:")
print(cm_vae)
print("\nVAE Classification Report:")
print(classification_report(y_true, y_pred_vae))

print("\nVAE + MAD Confusion Matrix:")
print(cm_mad)
print("\nVAE + MAD Classification Report:")
print(classification_report(y_true, y_pred_mad))

In [ ]:
# Visualizations
models = {'Isolation Forest': y_pred_iso, 'VAE': y_pred_vae, 'VAE + MAD': y_pred_mad}
cms = {'Isolation Forest': cm_iso, 'VAE': cm_vae, 'VAE + MAD': cm_mad}
scores = {'Isolation Forest': -iso.score_samples(X_injected_scaled), 'VAE': vae_score_func(X_injected_scaled, X_injected_scaled, vae_model=vae, feature_weights=feature_weights), 'VAE + MAD': mad_scores}
fprs = {}
tprs = {}
aucs = {}

for name, y_pred in models.items():
    # Confusion Matrix
    plt.figure(figsize=(6, 4))
    sns.heatmap(cms[name], annot=True, fmt='d', cmap='Blues', cbar=False)
    plt.title(f'Confusion Matrix ({name})')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.tight_layout()
    plt.savefig(f'{name.lower().replace(" ", "_")}_confusion_matrix.png')
    plt.show()
    plt.close()

    # Score Distribution
    plt.figure(figsize=(8, 5))
    sns.histplot(scores[name][y_true == 0], label='Normal', color='blue', kde=True)
    sns.histplot(scores[name][y_true == 1], label='Intrusion', color='red', kde=True)
    plt.title(f'{name} Score Distribution')
    plt.xlabel('Anomaly Score' if name == 'Isolation Forest' else 'Reconstruction Error' if name == 'VAE' else 'MAD Score')
    plt.axvline(np.percentile(scores[name], {'Isolation Forest': iso_thresh_percentile, 'VAE': vae_thresh_percentile, 'VAE + MAD': mad_thresh_percentile}[name]), color='black', linestyle='--', label='Threshold')
    plt.legend()
    plt.tight_layout()
    plt.savefig(f'{name.lower().replace(" ", "_")}_scores.png')
    plt.show()
    plt.close()

    # ROC Curve
    fpr, tpr, _ = roc_curve(y_true, scores[name])
    auc = roc_auc_score(y_true, scores[name])
    fprs[name] = fpr
    tprs[name] = tpr
    aucs[name] = auc

# ROC Curve Comparison
plt.figure(figsize=(8, 6))
for name in models:
    plt.plot(fprs[name], tprs[name], label=f'{name} (AUC = {aucs[name]:.3f})')
plt.plot([0, 1], [0, 1], 'k--')
plt.title('ROC Curve Comparison')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.legend()
plt.tight_layout()
plt.savefig('roc_curve_comparison.png')
plt.show()
plt.close()

# PCA/Latent Space Analysis
latent_z = vae_encoder_model.predict(X_injected_scaled)[2]
pca_vae = PCA(n_components=2)
latent_pca_vae = pca_vae.fit_transform(latent_z)

plt.figure(figsize=(8, 6))
plt.scatter(latent_pca_vae[y_true == 0, 0], latent_pca_vae[y_true == 0, 1], c='blue', label='Normal', alpha=0.5)
plt.scatter(latent_pca_vae[y_true == 1, 0], latent_pca_vae[y_true == 1, 1], c='red', label='Intrusion', alpha=0.5)
plt.title('PCA of Latent Space (VAE/VAE + MAD)')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.legend()
plt.tight_layout()
plt.savefig('vae_latent_pca.png')
plt.show()
plt.close()

pca_iso = PCA(n_components=2)
input_pca_iso = pca_iso.fit_transform(X_injected_scaled)

plt.figure(figsize=(8, 6))
plt.scatter(input_pca_iso[y_true == 0, 0], input_pca_iso[y_true == 0, 1], c='blue', label='Normal', alpha=0.5)
plt.scatter(input_pca_iso[y_true == 1, 0], input_pca_iso[y_true == 1, 1], c='red', label='Intrusion', alpha=0.5)
plt.title('PCA of Input Space (Isolation Forest)')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.legend()
plt.tight_layout()
plt.savefig('iso_input_pca.png')
plt.show()
plt.close()

In [ ]:
import numpy as np
from scipy import stats
import statsmodels.stats.contingency_tables as smct

# Provided contingency table
table = np.array([[17021, 0], [160, 1109]])

print("\nMcNemar's Test Contingency Table (VAE vs. VAE + MAD):")
print(table)

# Try scipy.stats.mcnemar (if updated)
try:
    result = stats.mcnemar(table, correction=True)
    print(f"McNemar's Chi2 (scipy): {result.statistic:.3f}, p-value: {result.pvalue:.3f}")
    if result.pvalue < 0.05:
        print("Significant difference (p < 0.05) using scipy")
    else:
        print("No significant difference (p >= 0.05) using scipy")
except AttributeError:
    print("scipy.mcnemar not available, using statsmodels instead.")

    # Fall back to statsmodels
    result = smct.mcnemar(table, exact=False, correction=True)
    print(f"McNemar's Chi2 (statsmodels): {result.statistic:.3f}, p-value: {result.pvalue:.3f}")
    if result.pvalue < 0.05:
        print("Significant difference (p < 0.05) using statsmodels")
    else:
        print("No significant difference (p >= 0.05) using statsmodels")

only VAE after injection

MAD+VAE for injection detection

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA

# Assume df_injected, X_scaled_df, vae, vae_encoder_model, vae_decoder_model, and feature_weights are available
# X_scaled_df is the scaled input data used for training, and df_injected is the injected dataset

# Extract features and labels from injected data
X_injected = df_injected.drop(columns=['label'])
y_true = df_injected['label'].values

# Scale injected features
scaler = StandardScaler()
X_injected_scaled = scaler.fit_transform(X_injected)

# Get VAE reconstruction
reconstructions = vae.predict(X_injected_scaled)
reconstruction_error = np.mean((X_injected_scaled - reconstructions) ** 2, axis=1)

# MAD-based anomaly detection
def mad_score(data):
    median = np.median(data)
    mad = np.median(np.abs(data - median))
    if mad == 0:
        mad = 1e-10  # Prevent division by zero
    return np.abs(data - median) / mad

# Calculate MAD scores for reconstruction errors
mad_scores = mad_score(reconstruction_error)

# Threshold for anomaly detection (tune this based on validation)
mad_threshold = np.percentile(mad_scores, 95)  # Top 5% as anomalies
y_pred_mad = (mad_scores > mad_threshold).astype(int)

# Evaluate performance
cm = confusion_matrix(y_true, y_pred_mad)
print("Confusion Matrix (VAE + MAD):")
print(cm)
print("\nClassification Report:")
print(classification_report(y_true, y_pred_mad))

# Visualizations
# 1. Confusion Matrix
plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False)
plt.title('Confusion Matrix (VAE + MAD)')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.savefig('vae_mad_confusion_matrix.png')
plt.show()
plt.close()

# 2. Reconstruction Error Distribution
plt.figure(figsize=(8, 5))
sns.histplot(reconstruction_error[y_true == 0], label='Normal', color='blue', kde=True)
sns.histplot(reconstruction_error[y_true == 1], label='Intrusion', color='red', kde=True)
plt.title('Reconstruction Error Distribution')
plt.xlabel('Mean Squared Error')
plt.legend()
plt.tight_layout()
plt.savefig('vae_mad_reconstruction_error.png')
plt.show()
plt.close()

# 3. MAD Score Distribution
plt.figure(figsize=(8, 5))
sns.histplot(mad_scores[y_true == 0], label='Normal', color='blue', kde=True)
sns.histplot(mad_scores[y_true == 1], label='Intrusion', color='red', kde=True)
plt.title('MAD Score Distribution')
plt.xlabel('MAD Score')
plt.axvline(mad_threshold, color='black', linestyle='--', label='Threshold')
plt.legend()
plt.tight_layout()
plt.savefig('vae_mad_scores.png')
plt.show()
plt.close()

# 4. PCA Visualization
latent_z = vae_encoder_model.predict(X_injected_scaled)[2]  # Get latent space (z)
pca = PCA(n_components=2)
latent_pca = pca.fit_transform(latent_z)

plt.figure(figsize=(8, 6))
plt.scatter(latent_pca[y_true == 0, 0], latent_pca[y_true == 0, 1], c='blue', label='Normal', alpha=0.5)
plt.scatter(latent_pca[y_true == 1, 0], latent_pca[y_true == 1, 1], c='red', label='Intrusion', alpha=0.5)
plt.title('PCA of Latent Space (VAE)')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.legend()
plt.tight_layout()
plt.savefig('vae_mad_pca.png')
plt.show()
plt.close()

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA

# Assume df_injected, X_scaled_df, vae, vae_encoder_model, vae_decoder_model, and feature_weights are available
# X_scaled_df is the scaled input data used for training, and df_injected is the injected dataset

# Extract features and labels from injected data
X_injected = df_injected.drop(columns=['label'])
y_true = df_injected['label'].values

# Scale injected features
scaler = StandardScaler()
X_injected_scaled = scaler.fit_transform(X_injected)

# Get VAE reconstruction
reconstructions = vae.predict(X_injected_scaled)
reconstruction_error = np.mean(feature_weights * (X_injected_scaled - reconstructions) ** 2, axis=1)

# MAD-based anomaly detection
def mad_score(data):
    median = np.median(data)
    mad = np.median(np.abs(data - median))
    if mad == 0:
        mad = 1e-10  # Prevent division by zero
    return np.abs(data - median) / mad

# Calculate MAD scores for reconstruction errors
mad_scores = mad_score(reconstruction_error)

# Tune threshold (revert to 95th percentile for testing)
mad_threshold = np.percentile(mad_scores, 95)  # Adjustable threshold
y_pred_mad = (mad_scores > mad_threshold).astype(int)

# Evaluate performance
cm = confusion_matrix(y_true, y_pred_mad)
print("Confusion Matrix (VAE + MAD):")
print(cm)
print("\nClassification Report:")
print(classification_report(y_true, y_pred_mad))

# Visualizations
# 1. Confusion Matrix
plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False)
plt.title('Confusion Matrix (VAE + MAD)')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.savefig('vae_mad_confusion_matrix.png')
plt.show()
plt.close()

# 2. Reconstruction Error Distribution
plt.figure(figsize=(8, 5))
sns.histplot(reconstruction_error[y_true == 0], label='Normal', color='blue', kde=True)
sns.histplot(reconstruction_error[y_true == 1], label='Intrusion', color='red', kde=True)
plt.title('Reconstruction Error Distribution')
plt.xlabel('Weighted Mean Squared Error')
plt.axvline(np.percentile(reconstruction_error, 95), color='black', linestyle='--', label='Threshold (Reconstruction)')
plt.legend()
plt.tight_layout()
plt.savefig('vae_mad_reconstruction_error.png')
plt.show()
plt.close()

# 3. MAD Score Distribution
plt.figure(figsize=(8, 5))
sns.histplot(mad_scores[y_true == 0], label='Normal', color='blue', kde=True)
sns.histplot(mad_scores[y_true == 1], label='Intrusion', color='red', kde=True)
plt.title('MAD Score Distribution')
plt.xlabel('MAD Score')
plt.axvline(mad_threshold, color='black', linestyle='--', label='Threshold')
plt.legend()
plt.tight_layout()
plt.savefig('vae_mad_scores.png')
plt.show()
plt.close()

# 4. PCA Visualization
latent_z = vae_encoder_model.predict(X_injected_scaled)[2]  # Get latent space (z)
pca = PCA(n_components=2)
latent_pca = pca.fit_transform(latent_z)

plt.figure(figsize=(8, 6))
plt.scatter(latent_pca[y_true == 0, 0], latent_pca[y_true == 0, 1], c='blue', label='Normal', alpha=0.5)
plt.scatter(latent_pca[y_true == 1, 0], latent_pca[y_true == 1, 1], c='red', label='Intrusion', alpha=0.5)
plt.title('PCA of Latent Space (VAE)')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.legend()
plt.tight_layout()
plt.savefig('vae_mad_pca.png')
plt.close()

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.metrics import confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
import math

# Assume df_injected, X_scaled_df, vae, vae_encoder_model, vae_decoder_model, and feature_weights are available

# Extract features and labels
X_injected = df_injected.drop(columns=['label'])
y_true = df_injected['label'].values

# Scale features
scaler = StandardScaler()
X_injected_scaled = scaler.fit_transform(X_injected)

# 1. Isolation Forest
iso = IsolationForest(contamination=0.05, random_state=42)
y_pred_iso = (iso.fit_predict(X_injected_scaled) == -1).astype(int)
cm_iso = confusion_matrix(y_true, y_pred_iso)
print("Isolation Forest Confusion Matrix:")
print(cm_iso)
print("\nIsolation Forest Classification Report:")
print(classification_report(y_true, y_pred_iso))

# 2. VAE
reconstructions_vae = vae.predict(X_injected_scaled)
reconstruction_error_vae = np.mean((X_injected_scaled - reconstructions_vae) ** 2, axis=1)
vae_threshold = np.percentile(reconstruction_error_vae, 95)
y_pred_vae = (reconstruction_error_vae > vae_threshold).astype(int)
cm_vae = confusion_matrix(y_true, y_pred_vae)
print("\nVAE Confusion Matrix:")
print(cm_vae)
print("\nVAE Classification Report:")
print(classification_report(y_true, y_pred_vae))

# 3. VAE + MAD
reconstructions_mad = vae.predict(X_injected_scaled)
reconstruction_error_mad = np.mean(feature_weights * (X_injected_scaled - reconstructions_mad) ** 2, axis=1)
def mad_score(data):
    median = np.median(data)
    mad = np.median(np.abs(data - median))
    if mad == 0: mad = 1e-10
    return np.abs(data - median) / mad
mad_scores = mad_score(reconstruction_error_mad)
mad_threshold = np.percentile(mad_scores, 95)
y_pred_mad = (mad_scores > mad_threshold).astype(int)
cm_mad = confusion_matrix(y_true, y_pred_mad)
print("\nVAE + MAD Confusion Matrix:")
print(cm_mad)
print("\nVAE + MAD Classification Report:")
print(classification_report(y_true, y_pred_mad))

# Extract metrics for radar plot (using provided percentile results)
models = ['Isolation Forest', 'VAE', 'VAE + MAD']
precision = [0.19, 0.73, 0.84]
recall = [0.19, 0.72, 0.83]
f1 = [0.19, 0.73, 0.83]

# Radar Plot
categories = ['Precision', 'Recall', 'F1-score']
n = len(categories)
angles = [i / float(n) * 2 * math.pi for i in range(n)]
angles += angles[:1]  # Complete the loop

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
ax.set_theta_offset(math.pi / 2)
ax.set_theta_direction(-1)

# Draw one axe per variable and add labels
plt.xticks(angles[:-1], categories)
ax.set_rlabel_position(0)
plt.yticks([0.2, 0.4, 0.6, 0.8, 1.0], ["0.2", "0.4", "0.6", "0.8", "1.0"], color="grey", size=10)
plt.ylim(0, 1)

# Plot data for each model
colors = ['blue', 'orange', 'green']
for i, model in enumerate(models):
    values = [precision[i], recall[i], f1[i]]
    values += values[:1]  # Complete the loop
    ax.plot(angles, values, linewidth=2, linestyle='solid', label=model, color=colors[i])
    ax.fill(angles, values, alpha=0.25, color=colors[i])

plt.title('Post-Injection Performance (Percentile-Based)', size=20, color='black', y=1.08)
plt.legend(loc='upper right', bbox_to_anchor=(1.1, 1.1))
plt.tight_layout()
plt.savefig('percentile_radar_plot.png')
plt.show()
plt.close()

# Line Plot
plt.figure(figsize=(10, 6))
markers = ['o', 's', '^']  # Circle, square, triangle
colors = ['blue', 'orange', 'green']
for i, model in enumerate(models):
    plt.plot(['Precision', 'Recall', 'F1'], [precision[i], recall[i], f1[i]],
             label=model, marker=markers[i], color=colors[i], linestyle='--', linewidth=1, markersize=8)
plt.title('Performance Metrics Comparison (Percentile-Based)')
plt.xlabel('Metric')
plt.ylabel('Score')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.7)
plt.ylim(0, 1)
plt.tight_layout()
plt.savefig('percentile_line_plot.png')
plt.show()
plt.close()

# Bar Plot
plt.figure(figsize=(10, 6))
x = np.arange(len(models))
width = 0.25
plt.bar(x - width, precision, width, label='Precision', color='blue', alpha=0.7)
plt.bar(x, recall, width, label='Recall', color='orange', alpha=0.7)
plt.bar(x + width, f1, width, label='F1-score', color='green', alpha=0.7)
plt.xlabel('Model')
plt.ylabel('Score')
plt.title('Performance Metrics Comparison (Percentile-Based)')
plt.xticks(x, models)
plt.legend()
plt.ylim(0, 1)
plt.tight_layout()
plt.savefig('percentile_bar_plot.png')
plt.show()
plt.close()

# Visualizations (for all models)
models = {'Isolation Forest': y_pred_iso, 'VAE': y_pred_vae, 'VAE + MAD': y_pred_mad}
cms = {'Isolation Forest': cm_iso, 'VAE': cm_vae, 'VAE + MAD': cm_mad}
scores = {'Isolation Forest': -iso.score_samples(X_injected_scaled), 'VAE': reconstruction_error_vae, 'VAE + MAD': mad_scores}

for name, y_pred in models.items():
    # Confusion Matrix
    plt.figure(figsize=(6, 4))
    sns.heatmap(cms[name], annot=True, fmt='d', cmap='Blues', cbar=False)
    plt.title(f'Confusion Matrix ({name})')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.tight_layout()
    plt.savefig(f'{name.lower().replace(" ", "_")}_confusion_matrix.png')
    plt.show()
    plt.close()

    # Score Distribution
    plt.figure(figsize=(8, 5))
    sns.histplot(scores[name][y_true == 0], label='Normal', color='blue', kde=True)
    sns.histplot(scores[name][y_true == 1], label='Intrusion', color='red', kde=True)
    plt.title(f'{name} Score Distribution')
    plt.xlabel('Score' if name == 'Isolation Forest' else 'Reconstruction Error' if name == 'VAE' else 'MAD Score')
    plt.axvline(np.percentile(scores[name], 95), color='black', linestyle='--', label='Threshold')
    plt.legend()
    plt.tight_layout()
    plt.savefig(f'{name.lower().replace(" ", "_")}_scores.png')
    plt.show()
    plt.close()

# PCA for VAE latent space (shared across VAE and VAE + MAD)
latent_z = vae_encoder_model.predict(X_injected_scaled)[2]
pca = PCA(n_components=2)
latent_pca = pca.fit_transform(latent_z)
plt.figure(figsize=(8, 6))
plt.scatter(latent_pca[y_true == 0, 0], latent_pca[y_true == 0, 1], c='blue', label='Normal', alpha=0.5)
plt.scatter(latent_pca[y_true == 1, 0], latent_pca[y_true == 1, 1], c='red', label='Intrusion', alpha=0.5)
plt.title('PCA of Latent Space (VAE)')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.legend()
plt.tight_layout()
plt.savefig('vae_pca.png')
plt.close()

# ... (previous code up to visualizations) ...

# Radar Plot (already included)
# ... (radar plot code) ...

# Precision-Recall Curve
from sklearn.metrics import precision_recall_curve, average_precision_score
plt.figure(figsize=(8, 6))
for name, score in scores.items():
    precision, recall, _ = precision_recall_curve(y_true, score)
    avg_precision = average_precision_score(y_true, score)
    plt.plot(recall, precision, label=f'{name} (AP = {avg_precision:.3f})')
plt.title('Precision-Recall Curve (Percentile-Based)')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig('precision_recall_curve_percentile.png')
plt.show()
plt.close()

# Normalized Confusion Matrix (example for one model)
for name, cm in cms.items():
    plt.figure(figsize=(6, 4))
    cm_normalized = cm / cm.sum(axis=1, keepdims=True)
    sns.heatmap(cm_normalized, annot=True, fmt='.2f', cmap='Blues', cbar=False)
    plt.title(f'Normalized Confusion Matrix ({name})')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.tight_layout()
    plt.savefig(f'{name.lower().replace(" ", "_")}_normalized_confusion_matrix.png')
    plt.show()
    plt.close()

# ... (add t-SNE, box plot, threshold sensitivity as shown above) ...

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.metrics import confusion_matrix, classification_report, f1_score, recall_score
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.model_selection import KFold
import math

# Assume df_injected, vae, vae_encoder_model, vae_decoder_model, and feature_weights are available

# Extract features and labels
X_injected = df_injected.drop(columns=['label'])
y_true = df_injected['label'].values

# Scale features
scaler = StandardScaler()
X_injected_scaled = scaler.fit_transform(X_injected)

def iso_score_func(X_train, X_val, iso_model=None):
    if iso_model is None:
        iso_model = IsolationForest(contamination=0.05, random_state=42)
        iso_model.fit(X_train)
    else:
        iso_model.fit(X_train)
    return -iso_model.score_samples(X_val)

def vae_score_func(X_train, X_val, vae_model=None, feature_weights=None):
    if vae_model is None:
        raise ValueError("vae_model must be provided")
    reconstructions = vae_model.predict(X_val)
    if feature_weights is not None:
        return np.mean(feature_weights * (X_val - reconstructions) ** 2, axis=1)
    return np.mean((X_val - reconstructions) ** 2, axis=1)

def mad_score_func(X_train, X_val, vae_model=None, feature_weights=None):
    if vae_model is None:
        raise ValueError("vae_model must be provided")
    reconstructions = vae_model.predict(X_val)
    error = np.mean(feature_weights * (X_val - reconstructions) ** 2, axis=1) if feature_weights is not None else np.mean((X_val - reconstructions) ** 2, axis=1)
    median = np.median(error)
    mad = np.median(np.abs(error - median))
    if mad == 0:
        mad = 1e-10
    return np.abs(error - median) / mad

# Function to find optimal threshold via cross-validation
def find_optimal_threshold(X, y, model_func, n_splits=5, metric='f1', **model_kwargs):
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    thresholds = np.linspace(0, 1, 100)
    best_thresholds = {}

    for train_idx, val_idx in kf.split(X):
        X_train, X_val = X[train_idx], X[val_idx]
        y_train, y_val = y[train_idx], y[val_idx]
        scores = model_func(X_train, X_val, **model_kwargs)
        for thresh in thresholds:
            y_pred = (scores > np.percentile(scores, thresh * 100)).astype(int)
            if len(y_val) > 0 and len(y_pred) == len(y_val):
                if metric == 'f1':
                    score = f1_score(y_val, y_pred)
                elif metric == 'recall':
                    score = recall_score(y_val, y_pred)
                if thresh not in best_thresholds or score > best_thresholds[thresh]:
                    best_thresholds[thresh] = score

    optimal_thresh = max(best_thresholds, key=best_thresholds.get) * 100 if best_thresholds else 95
    return optimal_thresh

# Initialize models
iso = IsolationForest(contamination=0.05, random_state=42)

# Find optimal thresholds
iso_thresh_percentile = find_optimal_threshold(X_injected_scaled, y_true, iso_score_func, metric='f1', iso_model=iso)
vae_thresh_percentile = find_optimal_threshold(X_injected_scaled, y_true, vae_score_func, metric='f1', vae_model=vae, feature_weights=feature_weights)
mad_thresh_percentile = find_optimal_threshold(X_injected_scaled, y_true, mad_score_func, metric='f1', vae_model=vae, feature_weights=feature_weights)

print(f"Optimal Thresholds - Isolation Forest: {iso_thresh_percentile:.1f}%, VAE: {vae_thresh_percentile:.1f}%, VAE + MAD: {mad_thresh_percentile:.1f}%")

# Generate predictions with optimal thresholds
y_pred_iso = (iso_score_func(X_injected_scaled, X_injected_scaled, iso_model=iso) > np.percentile(-iso.score_samples(X_injected_scaled), iso_thresh_percentile)).astype(int)
y_pred_vae = (vae_score_func(X_injected_scaled, X_injected_scaled, vae_model=vae, feature_weights=feature_weights) > np.percentile(vae_score_func(X_injected_scaled, X_injected_scaled, vae_model=vae, feature_weights=feature_weights), vae_thresh_percentile)).astype(int)
mad_scores = mad_score_func(X_injected_scaled, X_injected_scaled, vae_model=vae, feature_weights=feature_weights)
y_pred_mad = (mad_scores > np.percentile(mad_scores, mad_thresh_percentile)).astype(int)

# Confusion Matrices and Reports
cm_iso = confusion_matrix(y_true, y_pred_iso)
cm_vae = confusion_matrix(y_true, y_pred_vae)
cm_mad = confusion_matrix(y_true, y_pred_mad)

print("Isolation Forest Confusion Matrix:")
print(cm_iso)
print("\nIsolation Forest Classification Report:")
print(classification_report(y_true, y_pred_iso))

print("\nVAE Confusion Matrix:")
print(cm_vae)
print("\nVAE Classification Report:")
print(classification_report(y_true, y_pred_vae))

print("\nVAE + MAD Confusion Matrix:")
print(cm_mad)
print("\nVAE + MAD Classification Report:")
print(classification_report(y_true, y_pred_mad))

# Extract metrics for radar plot (using provided cross-validation results)
models = ['Isolation Forest', 'VAE', 'VAE + MAD']
precision = [0.22, 0.69, 0.76]  # Intrusion precision from cross-validation results
recall = [0.66, 0.95, 0.91]     # Intrusion recall from cross-validation results
f1 = [0.33, 0.80, 0.83]         # Intrusion F1-score from cross-validation results

# Radar Plot
categories = ['Precision', 'Recall', 'F1-score']
n = len(categories)
angles = [i / float(n) * 2 * math.pi for i in range(n)]
angles += angles[:1]  # Complete the loop

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
ax.set_theta_offset(math.pi / 2)
ax.set_theta_direction(-1)

# Draw one axe per variable and add labels
plt.xticks(angles[:-1], categories)
ax.set_rlabel_position(0)
plt.yticks([0.2, 0.4, 0.6, 0.8, 1.0], ["0.2", "0.4", "0.6", "0.8", "1.0"], color="grey", size=10)
plt.ylim(0, 1)

# Plot data for each model
colors = ['blue', 'orange', 'green']
for i, model in enumerate(models):
    values = [precision[i], recall[i], f1[i]]
    values += values[:1]  # Complete the loop
    ax.plot(angles, values, linewidth=2, linestyle='solid', label=model, color=colors[i])
    ax.fill(angles, values, alpha=0.25, color=colors[i])

plt.title('Post-Injection Performance (Cross-Validation)', size=20, color='black', y=1.08)
plt.legend(loc='upper right', bbox_to_anchor=(1.1, 1.1))
plt.tight_layout()
plt.savefig('cross_validation_radar_plot.png')
plt.show()
plt.close()

# Line Plot
plt.figure(figsize=(10, 6))
markers = ['o', 's', '^']  # Circle, square, triangle
colors = ['blue', 'orange', 'green']
for i, model in enumerate(models):
    plt.plot(['Precision', 'Recall', 'F1'], [precision[i], recall[i], f1[i]],
             label=model, marker=markers[i], color=colors[i], linestyle='--', linewidth=1, markersize=8)
plt.title('Performance Metrics Comparison (Cross-Validation)')
plt.xlabel('Metric')
plt.ylabel('Score')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.7)
plt.ylim(0, 1)
plt.tight_layout()
plt.savefig('cross_validation_line_plot.png')
plt.show()
plt.close()

# Bar Plot
plt.figure(figsize=(10, 6))
x = np.arange(len(models))
width = 0.25
plt.bar(x - width, precision, width, label='Precision', color='blue', alpha=0.7)
plt.bar(x, recall, width, label='Recall', color='orange', alpha=0.7)
plt.bar(x + width, f1, width, label='F1-score', color='green', alpha=0.7)
plt.xlabel('Model')
plt.ylabel('Score')
plt.title('Performance Metrics Comparison (Cross-Validation)')
plt.xticks(x, models)
plt.legend()
plt.ylim(0, 1)
plt.tight_layout()
plt.savefig('cross_validation_bar_plot.png')
plt.show()
plt.close()

# Visualizations (for all models)
models = {'Isolation Forest': y_pred_iso, 'VAE': y_pred_vae, 'VAE + MAD': y_pred_mad}
cms = {'Isolation Forest': cm_iso, 'VAE': cm_vae, 'VAE + MAD': cm_mad}
scores = {'Isolation Forest': -iso.score_samples(X_injected_scaled), 'VAE': vae_score_func(X_injected_scaled, X_injected_scaled, vae_model=vae, feature_weights=feature_weights), 'VAE + MAD': mad_scores}

for name, y_pred in models.items():
    # Confusion Matrix
    plt.figure(figsize=(6, 4))
    sns.heatmap(cms[name], annot=True, fmt='d', cmap='Blues', cbar=False)
    plt.title(f'Confusion Matrix ({name})')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.tight_layout()
    plt.savefig(f'{name.lower().replace(" ", "_")}_confusion_matrix.png')
    plt.show()
    plt.close()

    # Score Distribution
    plt.figure(figsize=(8, 5))
    sns.histplot(scores[name][y_true == 0], label='Normal', color='blue', kde=True)
    sns.histplot(scores[name][y_true == 1], label='Intrusion', color='red', kde=True)
    plt.title(f'{name} Score Distribution')
    plt.xlabel('Anomaly Score' if name == 'Isolation Forest' else 'Reconstruction Error' if name == 'VAE' else 'MAD Score')
    plt.axvline(np.percentile(scores[name], {'Isolation Forest': iso_thresh_percentile, 'VAE': vae_thresh_percentile, 'VAE + MAD': mad_thresh_percentile}[name]), color='black', linestyle='--', label='Threshold')
    plt.legend()
    plt.tight_layout()
    plt.savefig(f'{name.lower().replace(" ", "_")}_scores.png')
    plt.show()
    plt.close()

# PCA for VAE latent space (shared across VAE and VAE + MAD)
latent_z = vae_encoder_model.predict(X_injected_scaled)[2]
pca = PCA(n_components=2)
latent_pca = pca.fit_transform(latent_z)
plt.figure(figsize=(8, 6))
plt.scatter(latent_pca[y_true == 0, 0], latent_pca[y_true == 0, 1], c='blue', label='Normal', alpha=0.5)
plt.scatter(latent_pca[y_true == 1, 0], latent_pca[y_true == 1, 1], c='red', label='Intrusion', alpha=0.5)
plt.title('PCA of Latent Space (VAE/VAE + MAD)')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.legend()
plt.tight_layout()
plt.savefig('vae_latent_pca.png')
plt.close()

# ... (previous code up to visualizations) ...

# Radar Plot (already included)
# ... (radar plot code) ...

# Precision-Recall Curve
from sklearn.metrics import precision_recall_curve, average_precision_score
plt.figure(figsize=(8, 6))
for name, score in scores.items():
    precision, recall, _ = precision_recall_curve(y_true, score)
    avg_precision = average_precision_score(y_true, score)
    plt.plot(recall, precision, label=f'{name} (AP = {avg_precision:.3f})')
plt.title('Precision-Recall Curve (Percentile-Based)')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig('precision_recall_curve_percentile.png')
plt.show()
plt.close()

# Normalized Confusion Matrix (example for one model)
for name, cm in cms.items():
    plt.figure(figsize=(6, 4))
    cm_normalized = cm / cm.sum(axis=1, keepdims=True)
    sns.heatmap(cm_normalized, annot=True, fmt='.2f', cmap='Blues', cbar=False)
    plt.title(f'Normalized Confusion Matrix ({name})')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.tight_layout()
    plt.savefig(f'{name.lower().replace(" ", "_")}_normalized_confusion_matrix.png')
    plt.show()
    plt.close()

# ... (add t-SNE, box plot, threshold sensitivity as shown above) ...

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.metrics import confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import math

# Assume df_injected, X_scaled_df, vae, vae_encoder_model, vae_decoder_model, and feature_weights are available

# Extract features and labels
X_injected = df_injected.drop(columns=['label'])
y_true = df_injected['label'].values

# Scale features
scaler = StandardScaler()
X_injected_scaled = scaler.fit_transform(X_injected)

# 1. Isolation Forest
iso = IsolationForest(contamination=0.05, random_state=42)
y_pred_iso = (iso.fit_predict(X_injected_scaled) == -1).astype(int)
cm_iso = confusion_matrix(y_true, y_pred_iso)
print("Isolation Forest Confusion Matrix:")
print(cm_iso)
print("\nIsolation Forest Classification Report:")
print(classification_report(y_true, y_pred_iso))

# 2. VAE
reconstructions_vae = vae.predict(X_injected_scaled)
reconstruction_error_vae = np.mean((X_injected_scaled - reconstructions_vae) ** 2, axis=1)
vae_threshold = np.percentile(reconstruction_error_vae, 95)
y_pred_vae = (reconstruction_error_vae > vae_threshold).astype(int)
cm_vae = confusion_matrix(y_true, y_pred_vae)
print("\nVAE Confusion Matrix:")
print(cm_vae)
print("\nVAE Classification Report:")
print(classification_report(y_true, y_pred_vae))

# 3. VAE + MAD
reconstructions_mad = vae.predict(X_injected_scaled)
reconstruction_error_mad = np.mean(feature_weights * (X_injected_scaled - reconstructions_mad) ** 2, axis=1)
def mad_score(data):
    median = np.median(data)
    mad = np.median(np.abs(data - median))
    if mad == 0: mad = 1e-10
    return np.abs(data - median) / mad
mad_scores = mad_score(reconstruction_error_mad)
mad_threshold = np.percentile(mad_scores, 95)
y_pred_mad = (mad_scores > mad_threshold).astype(int)
cm_mad = confusion_matrix(y_true, y_pred_mad)
print("\nVAE + MAD Confusion Matrix:")
print(cm_mad)
print("\nVAE + MAD Classification Report:")
print(classification_report(y_true, y_pred_mad))

# Extract metrics for radar plot (using provided percentile results)
models = ['Isolation Forest', 'VAE', 'VAE + MAD']
precision = [0.19, 0.73, 0.84]
recall = [0.19, 0.72, 0.83]
f1 = [0.19, 0.73, 0.83]

# Radar Plot
categories = ['Precision', 'Recall', 'F1-score']
n = len(categories)
angles = [i / float(n) * 2 * math.pi for i in range(n)]
angles += angles[:1]

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
ax.set_theta_offset(math.pi / 2)
ax.set_theta_direction(-1)
plt.xticks(angles[:-1], categories)
ax.set_rlabel_position(0)
plt.yticks([0.2, 0.4, 0.6, 0.8, 1.0], ["0.2", "0.4", "0.6", "0.8", "1.0"], color="grey", size=10)
plt.ylim(0, 1)

colors = ['blue', 'orange', 'green']
for i, model in enumerate(models):
    values = [precision[i], recall[i], f1[i]]
    values += values[:1]
    ax.plot(angles, values, linewidth=2, linestyle='solid', label=model, color=colors[i])
    ax.fill(angles, values, alpha=0.25, color=colors[i])

plt.title('Post-Injection Performance (Percentile-Based)', size=20, color='black', y=1.08)
plt.legend(loc='upper right', bbox_to_anchor=(1.1, 1.1))
plt.tight_layout()
plt.savefig('percentile_radar_plot.png')
plt.show()
plt.close()

# Line Plot
plt.figure(figsize=(10, 6))
markers = ['o', 's', '^']
colors = ['blue', 'orange', 'green']
for i, model in enumerate(models):
    plt.plot(['Precision', 'Recall', 'F1'], [precision[i], recall[i], f1[i]],
             label=model, marker=markers[i], color=colors[i], linestyle='--', linewidth=1, markersize=8)
plt.title('Performance Metrics Comparison (Percentile-Based)')
plt.xlabel('Metric')
plt.ylabel('Score')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.7)
plt.ylim(0, 1)
plt.tight_layout()
plt.savefig('percentile_line_plot.png')
plt.show()
plt.close()

# Bar Plot
plt.figure(figsize=(10, 6))
x = np.arange(len(models))
width = 0.25
plt.bar(x - width, precision, width, label='Precision', color='blue', alpha=0.7)
plt.bar(x, recall, width, label='Recall', color='orange', alpha=0.7)
plt.bar(x + width, f1, width, label='F1-score', color='green', alpha=0.7)
plt.xlabel('Model')
plt.ylabel('Score')
plt.title('Performance Metrics Comparison (Percentile-Based)')
plt.xticks(x, models)
plt.legend()
plt.ylim(0, 1)
plt.tight_layout()
plt.savefig('percentile_bar_plot.png')
plt.show()
plt.close()

# Visualizations (for all models)
models = {'Isolation Forest': y_pred_iso, 'VAE': y_pred_vae, 'VAE + MAD': y_pred_mad}
cms = {'Isolation Forest': cm_iso, 'VAE': cm_vae, 'VAE + MAD': cm_mad}
scores = {'Isolation Forest': -iso.score_samples(X_injected_scaled), 'VAE': reconstruction_error_vae, 'VAE + MAD': mad_scores}

for name, y_pred in models.items():
    # Confusion Matrix
    plt.figure(figsize=(6, 4))
    sns.heatmap(cms[name], annot=True, fmt='d', cmap='Blues', cbar=False)
    plt.title(f'Confusion Matrix ({name})')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.tight_layout()
    plt.savefig(f'{name.lower().replace(" ", "_")}_confusion_matrix.png')
    plt.show()
    plt.close()

    # Score Distribution
    plt.figure(figsize=(8, 5))
    sns.histplot(scores[name][y_true == 0], label='Normal', color='blue', kde=True)
    sns.histplot(scores[name][y_true == 1], label='Intrusion', color='red', kde=True)
    plt.title(f'{name} Score Distribution')
    plt.xlabel('Score' if name == 'Isolation Forest' else 'Reconstruction Error' if name == 'VAE' else 'MAD Score')
    plt.axvline(np.percentile(scores[name], 95), color='black', linestyle='--', label='Threshold')
    plt.legend()
    plt.tight_layout()
    plt.savefig(f'{name.lower().replace(" ", "_")}_scores.png')
    plt.show()
    plt.close()

# PCA for VAE latent space (shared across VAE and VAE + MAD)
latent_z = vae_encoder_model.predict(X_injected_scaled)[2]
pca = PCA(n_components=2)
latent_pca = pca.fit_transform(latent_z)
plt.figure(figsize=(8, 6))
plt.scatter(latent_pca[y_true == 0, 0], latent_pca[y_true == 0, 1], c='blue', label='Normal', alpha=0.5)
plt.scatter(latent_pca[y_true == 1, 0], latent_pca[y_true == 1, 1], c='red', label='Intrusion', alpha=0.5)
plt.title('PCA of Latent Space (VAE/VAE + MAD)')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.legend()
plt.tight_layout()
plt.savefig('vae_pca.png')
plt.show()
plt.close()

# t-SNE for VAE latent space (shared across VAE and VAE + MAD)
tsne = TSNE(n_components=2, random_state=42)
latent_tsne = tsne.fit_transform(latent_z)
plt.figure(figsize=(8, 6))
plt.scatter(latent_tsne[y_true == 0, 0], latent_tsne[y_true == 0, 1], c='blue', label='Normal', alpha=0.5)
plt.scatter(latent_tsne[y_true == 1, 0], latent_tsne[y_true == 1, 1], c='red', label='Intrusion', alpha=0.5)
plt.title('t-SNE of Latent Space (VAE/VAE + MAD)')
plt.xlabel('t-SNE 1')
plt.ylabel('t-SNE 2')
plt.legend()
plt.tight_layout()
plt.savefig('vae_tsne.png')
plt.show()
plt.close()

# t-SNE for Isolation Forest input space
tsne_iso = TSNE(n_components=2, random_state=42)
input_tsne_iso = tsne_iso.fit_transform(X_injected_scaled)
plt.figure(figsize=(8, 6))
plt.scatter(input_tsne_iso[y_true == 0, 0], input_tsne_iso[y_true == 0, 1], c='blue', label='Normal', alpha=0.5)
plt.scatter(input_tsne_iso[y_true == 1, 0], input_tsne_iso[y_true == 1, 1], c='red', label='Intrusion', alpha=0.5)
plt.title('t-SNE of Input Space (Isolation Forest)')
plt.xlabel('t-SNE 1')
plt.ylabel('t-SNE 2')
plt.legend()
plt.tight_layout()
plt.savefig('iso_tsne.png')
plt.show()
plt.close()

In [ ]:
import numpy as np
import pandas as pd
from tensorflow.keras.layers import Input, Dense, Lambda
from tensorflow.keras.models import Model
from tensorflow.keras.losses import mse
from tensorflow.keras import backend as K
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf # Ensure tensorflow is imported

# Assume df_injected is your DataFrame with 'label' column
X = df_injected.drop(columns=['label']).values
y_true = df_injected['label'].values

# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Build VAE
input_dim = X_scaled.shape[1]
latent_dim = 10  # Latent space dimension

# Encoder
inputs = Input(shape=(input_dim,), name='vae_input') # Give input layer a name
h = Dense(20, activation='relu')(inputs)
z_mean = Dense(latent_dim, name='z_mean')(h) # Name z_mean layer
z_log_var = Dense(latent_dim, name='z_log_var')(h) # Name z_log_var layer

# Reparameterization trick
# Keep this as a Lambda layer operating on the KerasTensors z_mean and z_log_var
def sampling(args):
    z_mean, z_log_var = args
    batch = K.shape(z_mean)[0]
    epsilon = K.random_normal(shape=(batch, latent_dim))
    return z_mean + K.exp(0.5 * z_log_var) * epsilon

z = Lambda(sampling, output_shape=(latent_dim,), name='z_sampling')([z_mean, z_log_var])

# Decoder
# Define decoder layers as separate layers, not a model subclass yet
decoder_h = Dense(20, activation='relu', name='decoder_intermediate')
decoder_mean = Dense(input_dim, activation='linear', name='decoder_output') # Use 'linear' for scaled data

# Apply decoder layers to the sampled latent variable z
h_decoded = decoder_h(z)
x_decoded_mean = decoder_mean(h_decoded)

# Create the full VAE model using the Functional API
vae = Model(inputs, x_decoded_mean, name='vae')

# Add the KL Divergence loss directly to the model
# This calculation uses the KerasTensors z_mean and z_log_var within the model definition
# Calculate KL divergence for each sample, then mean over the batch
kl_loss = -0.5 * K.sum(1 + z_log_var - K.square(z_mean) - K.exp(z_log_var), axis=-1)
vae.add_loss(K.mean(kl_loss)) # Add the mean KL loss over the batch to the model's losses

# Compile the VAE model with only the reconstruction loss specified in compile()
# The added KL loss will be automatically included
# Note: You can adjust the weight of the KL loss here if needed, or include it in the add_loss call.
# If adding it in add_loss, just use 'mse' as the compile loss.
# If you want to weight it in compile, you'd need a custom train_step or adjust the mse part.
# A simple approach is to add weighted KL loss:
# total_vae_loss = mse(inputs, x_decoded_mean) + 0.01 * K.mean(kl_loss)
# vae.compile(optimizer='adam', loss=total_vae_loss) # <-- Alternative if you want weighted loss in compile
# For now, let's keep MSE as the primary loss and add KL loss separately.
vae.compile(optimizer='adam', loss=mse)


# Train the VAE (on normal data)
X_train = X_scaled[y_true == 0]
print(f"Training VAE on {X_train.shape[0]} normal samples.")
# Use verbose=1 to see training progress
history = vae.fit(X_train, X_train, epochs=50, batch_size=32, shuffle=True, validation_split=0.2, verbose=1)


# Predict reconstructions for the entire dataset
reconstructions = vae.predict(X_scaled, verbose=0) # Add verbose=0 for cleaner output
errors = np.mean(np.square(X_scaled - reconstructions), axis=1)

# MAD Thresholding
median = np.median(errors)
mad = 1.4826 * np.median(np.abs(errors - median))
mad_threshold = median + 3 * mad  # 3 MADs threshold

# Predict anomalies
y_pred_mad = (errors > mad_threshold).astype(int)

# Evaluate
cm_mad = confusion_matrix(y_true, y_pred_mad)
# Add zero_division=0 to avoid warnings if no instances of a class are predicted
report_mad = classification_report(y_true, y_pred_mad, target_names=['Normal', 'Intrusion'], output_dict=True, zero_division=0)

# Plot confusion matrix
plt.figure(figsize=(6, 5))
sns.heatmap(cm_mad, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Normal', 'Intrusion'],
            yticklabels=['Normal', 'Intrusion'])
plt.title("VAE + MAD Anomaly Detection\n(Threshold = Median + 3*MAD)")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.show()

print(f"MAD Threshold: {mad_threshold:.4f}")
print(classification_report(y_true, y_pred_mad, target_names=['Normal', 'Intrusion'], zero_division=0))

In [ ]:
from sklearn.cluster import DBSCAN

# Step 1: Use MAD to get initial anomalies
anomaly_indices = np.where(y_pred_mad == 1)[0]
anomalous_samples = X_scaled[anomaly_indices]

# Step 2: Cluster anomalies
dbscan = DBSCAN(eps=0.5, min_samples=5).fit(anomalous_samples)
cluster_labels = dbscan.labels_

# Step 3: Filter noise (keep only points in large clusters)
min_cluster_size = 10  # Adjust based on your data
valid_clusters = [c for c in np.unique(cluster_labels)
                  if (c != -1) and (np.sum(cluster_labels == c) >= min_cluster_size)]
filtered_anomalies = np.isin(cluster_labels, valid_clusters)

# Step 4: Map back to original indices
y_pred_dbscan = np.zeros(len(X_scaled), dtype=int)
y_pred_dbscan[anomaly_indices[filtered_anomalies]] = 1  # 1 = Confirmed anomaly

# Evaluate
cm_dbscan = confusion_matrix(y_true, y_pred_dbscan)
report_dbscan = classification_report(y_true, y_pred_dbscan, target_names=['Normal', 'Intrusion'], output_dict=True)

# Plot confusion matrix
plt.figure(figsize=(6, 5))
sns.heatmap(cm_dbscan, annot=True, fmt='d', cmap='Greens',
            xticklabels=['Normal', 'Intrusion'],
            yticklabels=['Normal', 'Intrusion'])
plt.title("DBSCAN-Filtered Anomalies\n(Min Cluster Size = 10)")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.show()

print(classification_report(y_true, y_pred_dbscan))

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import DBSCAN
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
from sklearn.preprocessing import StandardScaler # Ensure StandardScaler is imported

# Load data (with injected anomalies)
# Assuming df_injected is already loaded from previous cells
if 'df_injected' not in locals():
    print("Error: df_injected DataFrame not found. Please run previous cells to load and inject data.")


X = df_injected.drop(columns=['label'])
y_true = df_injected['label'].values

# Scale features (if your autoencoder expects scaled data)
# Assuming you are using the same scaler as before or need to fit a new one
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)


# 1. Compute reconstruction errors (from your autoencoder)
# Ensure 'autoencoder' is trained and available from previous cells
if 'autoencoder' not in locals():
    print("Error: autoencoder model not found. Please train the autoencoder in a previous cell.")
    # You might need to add the autoencoder definition and training code here
    # Example (basic structure, adapt to your model):
    # from tensorflow.keras.models import Model, load_model # Or import needed layers
    # try:
    #     autoencoder = load_model('path/to/your/trained_autoencoder_model') # Example loading
    # except Exception as e:
    #     print(f"Could not load autoencoder: {e}. Build and train a new one.")
    #     # Add autoencoder building and training code here...
    #     # autoencoder = build_autoencoder(...)
    #     # history = autoencoder.fit(...)

# Use X_scaled for prediction if the autoencoder was trained on scaled data
reconstructions = autoencoder.predict(X_scaled)
errors = np.mean(np.square(X_scaled - reconstructions), axis=1)


# 2. Tuned MAD Threshold (stricter: 4*MAD)
median = np.median(errors)
mad = 1.4826 * np.median(np.abs(errors - median))
mad_threshold = median + 4 * mad  # Increased from 3*MAD to reduce FPs

# Initial MAD predictions
y_pred_mad = (errors > mad_threshold).astype(int)
anomaly_indices = np.where(y_pred_mad == 1)[0]
# Use X_scaled for anomalous_samples if DBSCAN should operate on scaled data
anomalous_samples = X_scaled[anomaly_indices]


# 3. Optimized DBSCAN Parameters (automated tuning)
def optimize_dbscan(data, k=5):
    """Automatically find optimal eps using k-NN distance."""
    n_samples = data.shape[0]
    if n_samples < 2:
        print("Warning: Not enough samples to compute k-NN distances. Returning default eps=0.5.")
        return 0.5 # Or some other reasonable default
    # Adjust k dynamically
    adjusted_k = min(k, n_samples - 1)
    if adjusted_k < 1:
         print(f"Warning: adjusted_k is less than 1 ({adjusted_k}). Cannot compute NearestNeighbors. Returning default eps=0.5.")
         return 0.5

    neighbors = NearestNeighbors(n_neighbors=adjusted_k).fit(data)
    distances, _ = neighbors.kneighbors(data)
    # For k=1, distances[:, -1] is just the distance to the single nearest neighbor
    # For k>1, it's the distance to the k-th nearest neighbor
    distances = np.sort(distances[:, -1], axis=0)

    # Find knee point (optimal eps)
    # Need at least 3 points to compute second derivative
    if len(distances) < 3:
        print(f"Warning: Not enough points ({len(distances)}) to find knee point. Returning distance to the furthest point as eps.")
        return distances[-1] # Fallback to using max distance

    diffs = np.diff(distances, 2)
    if len(diffs) == 0:
         print("Warning: Cannot compute second derivative for knee point. Returning max distance.")
         return distances[-1]

    knee = diffs.argmax() + 1
    eps = distances[knee]

    plt.figure(figsize=(8, 4))
    plt.plot(distances)
    plt.axvline(knee, color='r', linestyle='--', label=f'Optimal eps: {eps:.2f}')
    plt.xlabel(f"Points sorted by {adjusted_k}-NN distance")
    plt.ylabel(f"{adjusted_k}-NN Distance")
    plt.title(f"DBSCAN Epsilon Selection ({adjusted_k}-NN Knee Point)")
    plt.legend()
    plt.show()

    return eps

# Check if there are any anomalous samples before attempting DBSCAN
if len(anomalous_samples) > 0:
    # Find optimal eps (plot shown)
    optimal_eps = optimize_dbscan(anomalous_samples)

    # Tuned DBSCAN (less strict min_samples)
    # Ensure eps is not zero or too small if anomalous_samples is trivial
    if optimal_eps <= 0:
        print(f"Warning: Optimal eps calculated as {optimal_eps}. Setting to a small positive value (1e-3).")
        optimal_eps = 1e-3 # Avoid zero or negative eps


    dbscan = DBSCAN(eps=optimal_eps, min_samples=3).fit(anomalous_samples)
    cluster_labels = dbscan.labels_

    # 4. Cluster Filtering (keep only medium/large clusters)
    min_cluster_size = 5  # Reduced from 10 to preserve recall
    valid_clusters = [c for c in np.unique(cluster_labels)
                     if (c != -1) and (np.sum(cluster_labels == c) >= min_cluster_size)]

    # Final predictions
    y_pred_final = np.zeros(len(X_scaled), dtype=int) # Use length of scaled data for prediction array
    # Only map back if there are any valid clusters found
    if len(valid_clusters) > 0:
         # Filter anomaly_indices based on filtered_anomalies boolean mask
         filtered_anomalies_mask = np.isin(cluster_labels, valid_clusters)
         final_anomaly_original_indices = anomaly_indices[filtered_anomalies_mask]
         y_pred_final[final_anomaly_original_indices] = 1

else:
    # If no anomalies were detected by MAD, no points will be predicted by DBSCAN
    print("No anomalies detected by the initial MAD threshold. DBSCAN filtering skipped.")
    y_pred_final = np.zeros(len(X_scaled), dtype=int) # Predict all as normal


# 5. Evaluation
print("=== Tuned MAD + DBSCAN Results ===")
print(f"MAD Threshold: {mad_threshold:.4f} (Median + 4*MAD)")
print(f"DBSCAN Parameters: eps={optimal_eps:.2f}, min_samples=3")
print(f"Clusters Kept: {len(valid_clusters) if 'valid_clusters' in locals() else 0} (Min size={min_cluster_size})") # Handle case where no valid clusters found

print("\nClassification Report:")
# Ensure y_true and y_pred_final have the same length (which they should if using len(X_scaled))
print(classification_report(y_true, y_pred_final, target_names=['Normal', 'Intrusion']))

# Confusion Matrix
cm = confusion_matrix(y_true, y_pred_final)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Normal', 'Intrusion'],
            yticklabels=['Normal', 'Intrusion'])
plt.title("Tuned MAD + DBSCAN Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.show()

# Error Distribution Plot
plt.figure(figsize=(10, 5))
plt.hist(errors, bins=50, color='skyblue', alpha=0.7, label='All Samples')
# Ensure y_true has the same length as errors and y_pred_final
plt.hist(errors[y_true == 1], bins=50, color='red', alpha=0.5, label='True Intrusions')
plt.axvline(mad_threshold, color='black', linestyle='--',
            label=f'MAD Threshold: {mad_threshold:.4f}')
plt.xlabel("Reconstruction Error")
plt.ylabel("Frequency")
plt.title("Error Distribution with Tuned MAD Threshold")
plt.legend()
plt.show()

In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Sequential, save_model
from tensorflow.keras.layers import InputLayer, Dense



# Step 1: Isolation Forest for initial cleaning
iso = IsolationForest(contamination=0.01, random_state=42)
iso.fit(X_scaled)
if_scores = -iso.decision_function(X_scaled)  # Higher = more anomalous

# Filter out top 1% anomalies
X_filtered = X_scaled[if_scores < np.percentile(if_scores, 99)]

# Step 2: Autoencoder training (fully corrected)
autoencoder = Sequential([
    InputLayer(shape=(X_scaled.shape[1],)),  # Corrected argument name
    Dense(64, activation='relu'),
    Dense(32, activation='relu'),
    Dense(16, activation='relu'),  # Bottleneck
    Dense(32, activation='relu'),
    Dense(64, activation='relu'),
    Dense(X_scaled.shape[1], activation='linear')
])
autoencoder.compile(optimizer='adam', loss='mse')
history = autoencoder.fit(X_filtered, X_filtered, epochs=50, batch_size=64, verbose=0)

# Step 3: Save baseline thresholds
reconstructions = autoencoder.predict(X_filtered, verbose=0)
errors = np.mean(np.square(X_filtered - reconstructions), axis=1)
mad_threshold = np.median(errors) + 3 * 1.4826 * np.median(np.abs(errors - np.median(errors)))

# Save models (modern format)
save_model(autoencoder, 'autoencoder.keras')  # Preferred .keras format
np.savez('pre_injection.npz',
         scaler_mean=scaler.mean_,
         scaler_scale=scaler.scale_,
         iso=iso,
         mad_threshold=mad_threshold)

In [ ]:
# file ipython-input-42-5b0a07a4a310
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score


# VAE: Compute anomaly scores (post-injection)
# Use the correctly scaled injected data X_scaled
vae_reconstructions = vae.predict(X_scaled, verbose=0)
vae_feature_errors = np.abs(X_scaled - vae_reconstructions)
# The features variable should be the columns of the scaled data used for training the VAE/AE
# Use X.columns as it corresponds to df_injected.drop(columns=['label']).columns
weights = np.array([1.5 if 'Gage' in col or 'Align' in col else 1.0 for col in X.columns])
vae_anomaly_scores = np.mean(vae_feature_errors * weights, axis=1)

# VAE: Set threshold
# Assuming vae_pre_threshold was calculated and stored during the pre-injection VAE analysis
# Use the pre-calculated threshold from the pre-injection data
if 'vae_pre_threshold' in locals():
    vae_threshold = vae_pre_threshold
    print(f"Using pre-calculated VAE threshold: {vae_threshold:.4f}")
else:
     # Fallback: Recalculate threshold based on post-injection data (less ideal for comparison)
     vae_threshold = np.percentile(vae_anomaly_scores, 90.05)
     print(f"vae_pre_threshold not found. Calculating VAE threshold on post-injection data: {vae_threshold:.4f}")


vae_predictions = (vae_anomaly_scores > vae_threshold).astype(int)

# Autoencoder: Compute anomaly scores (post-injection)
# Use the correctly scaled injected data X_scaled
ae_reconstructions = autoencoder.predict(X_scaled, verbose=0)
# Assuming feature_weights is defined globally or before this cell
# If AE also used weights, apply them here for consistency
ae_mse = np.mean(np.square(X_scaled - ae_reconstructions) * weights, axis=1) # Apply weights for AE too if used previously

# Autoencoder: Set threshold
# Assuming threshold (from the pre-injection AE analysis) was calculated and stored
# Based on globals, 'threshold' holds the AE pre-injection threshold
if 'threshold' in locals():
    ae_threshold = threshold
    print(f"Using pre-calculated AE threshold: {ae_threshold:.4f}")
else:
    # Fallback: Recalculate threshold based on post-injection data
    ae_threshold = np.percentile(ae_mse, 90.05)
    print(f"AE threshold not found. Calculating AE threshold on post-injection data: {ae_threshold:.4f}")


ae_predictions = (ae_mse > ae_threshold).astype(int)

# Pre-Injection Results (from your earlier code)
# Make sure 'anomalies_vae' and 'anomalies_ae' are the boolean arrays from pre-injection steps
# Check if they exist, otherwise calculate based on pre-calculated scores and thresholds
if 'anomalies_vae' not in locals() or 'anomalies_ae' not in locals():
     print("Warning: Pre-calculated anomaly flags (anomalies_vae, anomalies_ae) not found. Recalculating.")
     if 'vae_pre_injection_scores' in locals() and 'vae_pre_threshold' in locals():
          anomalies_vae = (vae_pre_injection_scores > vae_pre_threshold).astype(int)
     else:
          anomalies_vae = np.zeros(len(X_scaled_df), dtype=int) # Default to no anomalies if scores/threshold missing

     if 'mse_weighted' in locals() and 'threshold' in locals(): # 'threshold' holds AE pre-injection threshold
          anomalies_ae = (mse_weighted > threshold).astype(int)
     else:
          anomalies_ae = np.zeros(len(X_scaled_df), dtype=int) # Default to no anomalies

# Correct the variable name from 'anomalies' to 'anomalies_vae' (this seems correct based on globals)
vae_pre_anomalies = sum(anomalies_vae)  # VAE anomalies before injection
ae_pre_anomalies = sum(anomalies_ae)  # Autoencoder anomalies before injection
# Use len(X_scaled_df) as it represents the pre-injection data length
vae_pre_percentage = (vae_pre_anomalies / len(X_scaled_df)) * 100
ae_pre_percentage = (ae_pre_anomalies / len(X_scaled_df)) * 100

# Post-Injection Results
vae_post_anomalies = sum(vae_predictions)
ae_post_anomalies = sum(ae_predictions)
# Use the length of the post-injection data (which is the same as X_scaled)
vae_post_percentage = (vae_post_anomalies / len(X_scaled)) * 100
ae_post_percentage = (ae_post_anomalies / len(X_scaled)) * 100

# Print Anomaly Rate Comparison
print("=== Anomaly Rate Comparison 1 ===")
print(f"VAE Pre-Injection 1: {vae_pre_anomalies}/{len(X_scaled_df)} ({vae_pre_percentage:.2f}%)")
# Use the length of the post-injection data (which is the same as X_scaled)
print(f"VAE Post-Injection 1: {vae_post_anomalies}/{len(X_scaled)} ({vae_post_percentage:.2f}%)")
print(f"Autoencoder Pre-Injection 1: {ae_pre_anomalies}/{len(X_scaled_df)} ({ae_pre_percentage:.2f}%)")
# Use the length of the post-injection data (which is the same as X_scaled)
print(f"Autoencoder Post-Injection 1: {ae_post_anomalies}/{len(X_scaled)} ({ae_post_percentage:.2f}%)")

# Post-Injection Performance Metrics
# y_true is already defined from df_injected['label']
vae_precision, vae_recall, vae_f1 = precision_score(y_true, vae_predictions), recall_score(y_true, vae_predictions), f1_score(y_true, vae_predictions)
ae_precision, ae_recall, ae_f1 = precision_score(y_true, ae_predictions), recall_score(y_true, ae_predictions), f1_score(y_true, ae_predictions)

# False Positives Before Injection (all anomalies are false positives since no intrusions)
# These are the counts of anomalies detected in the original data
vae_pre_fp = vae_pre_anomalies
ae_pre_fp = ae_pre_anomalies

# False Positives After Injection
vae_cm = confusion_matrix(y_true, vae_predictions)
ae_cm = confusion_matrix(y_true, ae_predictions)
vae_post_fp = vae_cm[0, 1]  # False Positives: normal predicted as intrusion
ae_post_fp = ae_cm[0, 1]

# Print False Positive Comparison
print("\n=== False Positives Comparison ===")
print(f"VAE Pre-Injection False Positives 1: {vae_pre_fp}/{len(X_scaled_df)} ({(vae_pre_fp/len(X_scaled_df))*100:.2f}%)")
# Use the length of the post-injection data (which is the same as X_scaled)
print(f"VAE Post-Injection False Positives 1: {vae_post_fp}/{len(X_scaled)} ({(vae_post_fp/len(X_scaled))*100:.2f}%)")
print(f"Autoencoder Pre-Injection False Positives 1: {ae_pre_fp}/{len(X_scaled_df)} ({(ae_pre_fp/len(X_scaled_df))*100:.2f}%)")
# Use the length of the post-injection data (which is the same as X_scaled)
print(f"Autoencoder Post-Injection False Positives 1: {ae_post_fp}/{len(X_scaled)} ({(ae_post_fp/len(X_scaled))*100:.2f}%)")

# Visualization
plt.figure(figsize=(16, 12))

# Anomaly Rate Comparison (Bar Plot)
plt.subplot(2, 2, 1)
models = ['VAE', 'Autoencoder']
pre_rates = [vae_pre_percentage, ae_pre_percentage]
post_rates = [vae_post_percentage, ae_post_percentage]
x = np.arange(len(models))
width = 0.35
plt.bar(x - width/2, pre_rates, width, label='Pre-Injection', color='teal')
plt.bar(x + width/2, post_rates, width, label='Post-Injection', color='blue')
plt.xticks(x, models)
plt.ylabel('Anomaly Percentage (%)')
plt.title('Anomaly Detection Rate Before vs After Injection 1')
plt.legend()

# Performance Metrics (Post-Injection) - Radar Chart
plt.subplot(2, 2, 2)
metrics = ['Precision', 'Recall', 'F1-Score']
vae_metrics = [vae_precision, vae_recall, vae_f1]
ae_metrics = [ae_precision, ae_recall, ae_f1]
angles = np.linspace(0, 2 * np.pi, len(metrics), endpoint=False).tolist()
angles += angles[:1]
vae_metrics += vae_metrics[:1]
ae_metrics += ae_metrics[:1]
ax = plt.subplot(2, 2, 2, polar=True)
ax.fill(angles, vae_metrics, 'teal', alpha=0.5, label='VAE')
ax.fill(angles, ae_metrics, 'purple', alpha=0.5, label='Autoencoder')
ax.set_xticks(angles[:-1])
ax.set_xticklabels(metrics)
plt.title('Post-Injection Performance Metrics 1')
plt.legend(loc='upper right', bbox_to_anchor=(1.1, 1.1))

# Anomaly Score Distribution Comparison (VAE)
plt.subplot(2, 2, 3)
# FIX: Use the variable where the pre-injection VAE scores were actually stored
if 'vae_pre_injection_scores' in locals():
    sns.histplot(vae_pre_injection_scores, bins=50, kde=True, color='teal', alpha=0.5, label='Pre-Injection')
else:
    print("Warning: vae_pre_injection_scores not found for plotting.")

sns.histplot(vae_anomaly_scores, bins=50, kde=True, color='blue', alpha=0.3, label='Post-Injection')
# Use the threshold that was actually applied for VAE post-injection
plt.axvline(vae_threshold, color='r', linestyle='--', label=f'Threshold: {vae_threshold:.4f}')
plt.title('VAE Anomaly Score Distribution Before vs After 1')
plt.xlabel('Weighted Reconstruction Error')
plt.legend()

# Anomaly Score Distribution Comparison (Autoencoder)
plt.subplot(2, 2, 4)
# Use the pre-injection mse and the post-injection ae_mse
# Assuming 'mse_weighted' was the variable name for pre-injection Autoencoder MSE
if 'mse_weighted' in locals():
     sns.histplot(mse_weighted, bins=50, kde=True, color='purple', alpha=0.5, label='Pre-Injection')
else:
    print("Warning: mse_weighted (AE pre-injection scores) not found for plotting.")

sns.histplot(ae_mse, bins=50, kde=True, color='magenta', alpha=0.3, label='Post-Injection')
# Use the threshold that was actually applied for AE post-injection
plt.axvline(ae_threshold, color='r', linestyle='--', label=f'Threshold: {ae_threshold:.4f}')
plt.title('Autoencoder MSE Distribution Before vs After 1')
plt.xlabel('Reconstruction Error (MSE)')
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score

# Prepare the injected data for testing
# Assuming df_injected is already loaded and X_scaled_injected is created correctly from it
features_post_selection = X.columns.tolist()  # Get column names from the X DataFrame
# Note: If df_injected might have different features or order than X, regenerate X_scaled_injected
# inside this cell for safety, ensuring you fit/transform the scaler on the relevant data.
# For now, assuming X_scaled_injected is correctly generated from df_injected elsewhere.
# scaler = StandardScaler() # Make sure scaler is the one used for the pre-injection data, or refit if needed.
# X_scaled_injected = scaler.fit_transform(df_injected[features_post_selection])

y_true = df_injected['label'].values  # True labels: 0 (normal), 1 (intrusion)

# Pre-Injection Ground Truth (all normal, label=0)
# Assuming X_scaled_df represents the pre-injection data
y_true_pre = np.zeros(len(X_scaled_df))  # All samples are normal before injection

# Pre-Injection Predictions (already computed)
# Use the correct variable name 'anomalies_vae' which holds VAE pre-injection predictions
if 'anomalies_vae' in locals():
    vae_pre_predictions = anomalies_vae.astype(int)  # From VAE pre-injection
else:
    print("Warning: 'anomalies_vae' not found. Cannot evaluate VAE pre-injection performance.")
    # Provide a fallback or raise an error if pre-injection evaluation is critical

if 'anomalies_ae' in locals():
    ae_pre_predictions = anomalies_ae.astype(int)  # From Autoencoder pre-injection
else:
    print("Warning: 'anomalies_ae' not found. Cannot evaluate Autoencoder pre-injection performance.")
    # Provide a fallback or raise an error if pre-injection evaluation is critical


# Only proceed with pre-injection confusion matrices if pre-injection predictions were found
if 'vae_pre_predictions' in locals():
    vae_cm_pre = confusion_matrix(y_true_pre, vae_pre_predictions)
else:
    vae_cm_pre = np.zeros((2,2), dtype=int) # Placeholder if data is missing

if 'ae_pre_predictions' in locals():
    ae_cm_pre = confusion_matrix(y_true_pre, ae_pre_predictions)
else:
    ae_cm_pre = np.zeros((2,2), dtype=int) # Placeholder if data is missing


# VAE: Compute anomaly scores (post-injection)
# Ensure the vae model is available
if 'vae' not in locals():
     print("Error: 'vae' model not found. Cannot compute VAE post-injection scores.")
     vae_reconstructions = None # Set to None or handle error appropriately
else:
    vae_reconstructions = vae.predict(X_scaled_injected, verbose=0)

# Only proceed if reconstructions were successful
if vae_reconstructions is not None:
    vae_feature_errors = np.abs(X_scaled_injected - vae_reconstructions)
    # Ensure weights are defined (they should be if the previous cells ran)
    if 'weights' not in locals():
         weights = np.array([1.5 if 'Gage' in col or 'Align' in col else 1.0 for col in features_post_selection])

    vae_anomaly_scores = np.mean(vae_feature_errors * weights, axis=1)

    # VAE: Set threshold
    # Using a threshold based on the distribution of post-injection scores is common
    # You could also try to reuse a threshold from the pre-injection phase if trained on normal data only
    vae_threshold = np.percentile(vae_anomaly_scores, 90.05)  # Example: Match roughly 9.95% anomaly rate
    vae_predictions = (vae_anomaly_scores > vae_threshold).astype(int)
else:
    vae_predictions = np.zeros(len(X_scaled_injected), dtype=int) # Default predictions if VAE failed
    vae_anomaly_scores = np.zeros(len(X_scaled_injected))
    vae_threshold = 0.0 # Default threshold


# Autoencoder: Compute anomaly scores (post-injection)
# Ensure the autoencoder model is available
if 'autoencoder' not in locals():
    print("Error: 'autoencoder' model not found. Cannot compute AE post-injection scores.")
    ae_reconstructions = None # Set to None or handle error appropriately
else:
    ae_reconstructions = autoencoder.predict(X_scaled_injected, verbose=0)

# Only proceed if reconstructions were successful
if ae_reconstructions is not None:
    # Assuming feature_weights is defined globally or before this cell
    # If AE also used weights, apply them here for consistency
    if 'weights' not in locals():
         weights = np.array([1.5 if 'Gage' in col or 'Align' in col else 1.0 for col in features_post_selection])

    ae_mse = np.mean(np.square(X_scaled_injected - ae_reconstructions) * weights, axis=1) # Apply weights for AE too if used previously

    # Autoencoder: Set threshold
    # Using a threshold based on the distribution of post-injection scores
    ae_threshold = np.percentile(ae_mse, 90.05)
    ae_predictions = (ae_mse > ae_threshold).astype(int)
else:
    ae_predictions = np.zeros(len(X_scaled_injected), dtype=int) # Default predictions if AE failed
    ae_mse = np.zeros(len(X_scaled_injected))
    ae_threshold = 0.0 # Default threshold


# Post-Injection Confusion Matrices
vae_cm_post = confusion_matrix(y_true, vae_predictions)
ae_cm_post = confusion_matrix(y_true, ae_predictions)

# Pre-Injection Results (Use calculated counts if predictions were successful)
vae_pre_anomalies = sum(vae_pre_predictions) if 'vae_pre_predictions' in locals() else 0
ae_pre_anomalies = sum(ae_pre_predictions) if 'ae_pre_predictions' in locals() else 0
vae_pre_percentage = (vae_pre_anomalies / len(X_scaled_df)) * 100 if len(X_scaled_df) > 0 else 0
ae_pre_percentage = (ae_pre_anomalies / len(X_scaled_df)) * 100 if len(X_scaled_df) > 0 else 0

# Post-Injection Results
vae_post_anomalies = sum(vae_predictions)
ae_post_anomalies = sum(ae_predictions)
vae_post_percentage = (vae_post_anomalies / len(X_scaled_injected)) * 100 if len(X_scaled_injected) > 0 else 0
ae_post_percentage = (ae_post_anomalies / len(X_scaled_injected)) * 100 if len(X_scaled_injected) > 0 else 0


# Print Anomaly Rate Comparison
print("=== Anomaly Rate Comparison ===")
print(f"VAE Pre-Injection: {vae_pre_anomalies}/{len(X_scaled_df)} ({vae_pre_percentage:.2f}%)")
print(f"VAE Post-Injection: {vae_post_anomalies}/{len(X_scaled_injected)} ({vae_post_percentage:.2f}%)")
print(f"Autoencoder Pre-Injection: {ae_pre_anomalies}/{len(X_scaled_df)} ({ae_pre_percentage:.2f}%)")
print(f"Autoencoder Post-Injection: {ae_post_anomalies}/{len(X_scaled_injected)} ({ae_post_percentage:.2f}%)")

# Post-Injection Performance Metrics
# Calculate only if predictions are available and data length matches true labels
if len(y_true) == len(vae_predictions):
    vae_precision, vae_recall, vae_f1 = precision_score(y_true, vae_predictions), recall_score(y_true, vae_predictions), f1_score(y_true, vae_predictions)
else:
    vae_precision, vae_recall, vae_f1 = 0.0, 0.0, 0.0 # Default if mismatch

if len(y_true) == len(ae_predictions):
    ae_precision, ae_recall, ae_f1 = precision_score(y_true, ae_predictions), recall_score(y_true, ae_predictions), f1_score(y_true, ae_predictions)
else:
    ae_precision, ae_recall, ae_f1 = 0.0, 0.0, 0.0 # Default if mismatch


# False Positives Before Injection (all anomalies are false positives since no intrusions)
vae_pre_fp = vae_pre_anomalies
ae_pre_fp = ae_pre_anomalies

# False Positives After Injection
vae_post_fp = vae_cm_post[0, 1]  # False Positives: normal predicted as intrusion
ae_post_fp = ae_cm_post[0, 1]

# Print False Positive Comparison
print("\n=== False Positives Comparison ===")
print(f"VAE Pre-Injection False Positives 2: {vae_pre_fp}/{len(X_scaled_df)} ({(vae_pre_fp/len(X_scaled_df))*100:.2f}%)" if len(X_scaled_df) > 0 else "VAE Pre-Injection False Positives 2: N/A")
print(f"VAE Post-Injection False Positives 2: {vae_post_fp}/{len(X_scaled_injected)} ({(vae_post_fp/len(X_scaled_injected))*100:.2f}%)" if len(X_scaled_injected) > 0 else "VAE Post-Injection False Positives 2: N/A")
print(f"Autoencoder Pre-Injection False Positives 2: {ae_pre_fp}/{len(X_scaled_df)} ({(ae_pre_fp/len(X_scaled_df))*100:.2f}%)" if len(X_scaled_df) > 0 else "Autoencoder Pre-Injection False Positives 2: N/A")
print(f"Autoencoder Post-Injection False Positives 2: {ae_post_fp}/{len(X_scaled_injected)} ({(ae_post_fp/len(X_scaled_injected))*100:.2f}%)" if len(X_scaled_injected) > 0 else "Autoencoder Post-Injection False Positives 2: N/A")


# Visualization
plt.figure(figsize=(14, 20))  # Increased height to fit all plots

# Anomaly Rate Comparison (Bar Plot)
plt.subplot(4, 2, 1)
models = ['VAE', 'Autoencoder']
pre_rates = [vae_pre_percentage, ae_pre_percentage]
post_rates = [vae_post_percentage, ae_post_percentage]
x = np.arange(len(models))
width = 0.35
plt.bar(x - width/2, pre_rates, width, label='Pre-Injection', color='teal')
plt.bar(x + width/2, post_rates, width, label='Post-Injection', color='blue')
plt.xticks(x, models, fontsize=10)
plt.ylabel('Anomaly Percentage (%)', fontsize=10)
plt.title('Anomaly Detection Rate Before vs After Injection 2', fontsize=12)
plt.legend(fontsize=9)

# Performance Metrics (Post-Injection) - Radar Chart
plt.subplot(4, 2, 2)
metrics = ['Precision', 'Recall', 'F1-Score']
vae_metrics = [vae_precision, vae_recall, vae_f1]
ae_metrics = [ae_precision, ae_recall, ae_f1]
angles = np.linspace(0, 2 * np.pi, len(metrics), endpoint=False).tolist()
angles += angles[:1]
vae_metrics += vae_metrics[:1]
ae_metrics += ae_metrics[:1]
ax = plt.subplot(4, 2, 2, polar=True)
ax.fill(angles, vae_metrics, 'teal', alpha=0.5, label='VAE')
ax.fill(angles, ae_metrics, 'purple', alpha=0.5, label='Autoencoder')
ax.set_xticks(angles[:-1])
ax.set_xticklabels(metrics, fontsize=10)
plt.title('Post-Injection Performance Metrics 2', fontsize=12)
plt.legend(loc='upper right', bbox_to_anchor=(1.1, 1.1), fontsize=9)

# VAE Confusion Matrix - Pre-Injection
plt.subplot(4, 2, 3)
sns.heatmap(vae_cm_pre, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=['Normal', 'Anomaly'], yticklabels=['Normal', 'Anomaly'],
            annot_kws={'size': 10})
plt.title("VAE Confusion Matrix (Pre-Injection) 2", fontsize=12)
plt.xlabel("Predicted", fontsize=10)
plt.ylabel("True", fontsize=10)

# VAE Confusion Matrix - Post-Injection
plt.subplot(4, 2, 4)
sns.heatmap(vae_cm_post, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=['Normal', 'Intrusion'], yticklabels=['Normal', 'Intrusion'],
            annot_kws={'size': 10})
plt.title("VAE Confusion Matrix (Post-Injection) 2", fontsize=12)
plt.xlabel("Predicted", fontsize=10)
plt.ylabel("True", fontsize=10)

# Autoencoder Confusion Matrix - Pre-Injection
plt.subplot(4, 2, 5)
sns.heatmap(ae_cm_pre, annot=True, fmt='d', cmap='Purples', cbar=False,
            xticklabels=['Normal', 'Anomaly'], yticklabels=['Normal', 'Anomaly'],
            annot_kws={'size': 10})
plt.title("Autoencoder Confusion Matrix (Pre-Injection) 2", fontsize=12)
plt.xlabel("Predicted", fontsize=10)
plt.ylabel("True", fontsize=10)

# Autoencoder Confusion Matrix - Post-Injection
plt.subplot(4, 2, 6)
sns.heatmap(ae_cm_post, annot=True, fmt='d', cmap='Purples', cbar=False,
            xticklabels=['Normal', 'Intrusion'], yticklabels=['Normal', 'Intrusion'],
            annot_kws={'size': 10})
plt.title("Autoencoder Confusion Matrix (Post-Injection) 2", fontsize=12)
plt.xlabel("Predicted", fontsize=10)
plt.ylabel("True", fontsize=10)

# Anomaly Score Distribution Comparison (VAE)
plt.subplot(4, 2, 7)
# Ensure the correct variables for pre-injection scores are used
# Based on globals and previous cells, it seems 'vae_anomaly_scores' is used for POST-injection
# The VAE pre-injection scores were likely named 'vae_pre_injection_scores' or similar
# If vae_pre_injection_scores exists, use it
if 'vae_pre_injection_scores' in locals():
     sns.histplot(vae_pre_injection_scores, bins=50, kde=True, color='teal', alpha=0.5, label='Pre-Injection')
else:
     print("Warning: 'vae_pre_injection_scores' not found for VAE score distribution plot.")
     # Optionally plot post-injection scores only or show a blank plot
     # sns.histplot(vae_anomaly_scores, bins=50, kde=True, color='blue', alpha=0.3, label='Post-Injection')


sns.histplot(vae_anomaly_scores, bins=50, kde=True, color='blue', alpha=0.3, label='Post-Injection')
plt.axvline(vae_threshold, color='r', linestyle='--', label=f'Threshold: {vae_threshold:.2f}')
plt.title('VAE Anomaly Score Distribution Before vs After 2', fontsize=12)
plt.xlabel('Weighted Reconstruction Error', fontsize=10)
plt.ylabel('Frequency', fontsize=10)
plt.legend(fontsize=9)

# Anomaly Score Distribution Comparison (Autoencoder)
plt.subplot(4, 2, 8)
# Ensure the correct variables for pre-injection scores are used
# Based on globals and previous cells, 'mse_weighted' seems to be the AE pre-injection scores
if 'mse_weighted' in locals():
    sns.histplot(mse_weighted, bins=50, kde=True, color='purple', alpha=0.5, label='Pre-Injection')
else:
    print("Warning: 'mse_weighted' (AE pre-injection scores) not found for AE score distribution plot.")
     # Optionally plot post-injection scores only or show a blank plot
     # sns.histplot(ae_mse, bins=50, kde=True, color='magenta', alpha=0.3, label='Post-Injection')

sns.histplot(ae_mse, bins=50, kde=True, color='magenta', alpha=0.3, label='Post-Injection')
plt.axvline(ae_threshold, color='r', linestyle='--', label=f'Threshold: {ae_threshold:.3f}')
plt.title('Autoencoder MSE Distribution Before vs After 2', fontsize=12)
plt.xlabel('Reconstruction Error (MSE)', fontsize=10)
plt.ylabel('Frequency', fontsize=10)
plt.legend(fontsize=9)

plt.tight_layout(pad=2.0)  # Adjust spacing between subplots
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score

# Prepare the injected data for testing
features_post_selection = X.columns.tolist()
# Assuming scaler was fitted on the training data (X_scaled_df)
# If df_injected contains the exact same features and order as df,
# then using the scaler fitted on the original data is correct for consistency.
# If not, you might need to refit the scaler or ensure df_injected aligns with the features used for training.
# For now, assuming df_injected is aligned and using the pre-fitted scaler is intended.
# Access the pre-fitted scaler from globals if it's not in local scope
if 'scaler' not in locals():
    print("Warning: 'scaler' not found in local scope. Ensure it was fitted on training data.")
    # Fallback: Refit scaler on the injected data (less ideal if training data was different)
    # scaler = StandardScaler()
    # X_scaled_injected = scaler.fit_transform(df_injected[features_post_selection])
    # Note: This fallback is commented out as using the pre-fitted scaler is generally preferred.
    # You might need to ensure 'scaler' is available from previous cells.

# Assume X_scaled_injected is already correctly created from df_injected elsewhere,
# using the scaler fitted on the pre-injection data (X_scaled_df).
# If X_scaled_injected isn't in globals, you'll need to add:
# scaler = StandardScaler() # Make sure this is the *same* scaler used for X_scaled_df
# X_scaled_injected = scaler.fit_transform(df_injected[features_post_selection])
# For now, using the global variable X_scaled_injected.

y_true = df_injected['label'].values  # True labels: 0 (normal), 1 (intrusion)

# Pre-Injection Ground Truth (all normal, label=0)
# Assuming X_scaled_df represents the pre-injection data
y_true_pre = np.zeros(len(X_scaled_df))

# Pre-Injection Predictions
# FIX: Use the correct variable name 'anomalies_vae' for VAE pre-injection predictions
# Check if 'anomalies_vae' exists in globals
if 'anomalies_vae' in globals():
    vae_pre_predictions = globals()['anomalies_vae'].astype(int)
elif 'anomalies_vae' in locals(): # Check local scope as well
     vae_pre_predictions = anomalies_vae.astype(int)
else:
    print("Warning: 'anomalies_vae' not found. Cannot evaluate VAE pre-injection performance.")
    # Provide a fallback or handle the error
    vae_pre_predictions = np.zeros(len(X_scaled_df), dtype=int)


# 'anomalies_ae' was correctly referenced for AE pre-injection predictions
if 'anomalies_ae' in globals():
    ae_pre_predictions = globals()['anomalies_ae'].astype(int)
elif 'anomalies_ae' in locals(): # Check local scope as well
    ae_pre_predictions = anomalies_ae.astype(int)
else:
    print("Warning: 'anomalies_ae' not found. Cannot evaluate Autoencoder pre-injection performance.")
    # Provide a fallback or handle the error
    ae_pre_predictions = np.zeros(len(X_scaled_df), dtype=int)


# Pre-Injection Confusion Matrices
# Only compute if prediction length matches true labels
if len(y_true_pre) == len(vae_pre_predictions) and len(y_true_pre) > 0:
    vae_cm_pre = confusion_matrix(y_true_pre, vae_pre_predictions)
else:
    vae_cm_pre = np.zeros((2,2), dtype=int) # Placeholder if data is missing or lengths mismatch

if len(y_true_pre) == len(ae_pre_predictions) and len(y_true_pre) > 0:
    ae_cm_pre = confusion_matrix(y_true_pre, ae_pre_predictions)
else:
    ae_cm_pre = np.zeros((2,2), dtype=int) # Placeholder if data is missing or lengths mismatch


# Set Thresholds Using Pre-Injection Data
# FIX: Use the correct variable names for pre-injection scores
# Based on globals and previous cells, 'vae_pre_injection_scores' holds VAE scores
if 'vae_pre_injection_scores' in globals():
     vae_threshold = np.percentile(globals()['vae_pre_injection_scores'], 90.05)
     print(f"Using pre-calculated VAE threshold: {vae_threshold:.4f}")
elif 'vae_pre_injection_scores' in locals(): # Check local scope
     vae_threshold = np.percentile(vae_pre_injection_scores, 90.05)
     print(f"Using pre-calculated VAE threshold: {vae_threshold:.4f}")
else:
     print("Warning: 'vae_pre_injection_scores' not found. VAE threshold not set from pre-injection data.")
     # Provide a fallback - calculating on post-injection data is less ideal
     vae_threshold = np.percentile(vae_anomaly_scores_post, 90.05) if 'vae_anomaly_scores_post' in locals() and len(vae_anomaly_scores_post) > 0 else 0.0
     print(f"Calculating VAE threshold on post-injection data as fallback: {vae_threshold:.4f}")


# Based on globals and previous cells, 'mse_weighted' holds AE scores
if 'mse_weighted' in globals():
     ae_threshold = np.percentile(globals()['mse_weighted'], 90.05)
     print(f"Using pre-calculated AE threshold: {ae_threshold:.4f}")
elif 'mse_weighted' in locals(): # Check local scope
     ae_threshold = np.percentile(mse_weighted, 90.05)
     print(f"Using pre-calculated AE threshold: {ae_threshold:.4f}")
else:
    print("Warning: 'mse_weighted' (AE pre-injection scores) not found. AE threshold not set from pre-injection data.")
    # Provide a fallback - calculating on post-injection data is less ideal
    ae_threshold = np.percentile(ae_mse_post, 90.05) if 'ae_mse_post' in locals() and len(ae_mse_post) > 0 else 0.0
    print(f"Calculating AE threshold on post-injection data as fallback: {ae_threshold:.4f}")


# VAE: Compute anomaly scores (post-injection)
# Ensure the vae model is available
if 'vae' not in locals() and 'vae' not in globals():
     print("Error: 'vae' model not found. Cannot compute VAE post-injection scores.")
     vae_reconstructions = None # Set to None or handle error appropriately
else:
    # Access the model from globals if not local
    vae_model = vae if 'vae' in locals() else globals()['vae']
    # Predict on the injected data
    vae_reconstructions = vae_model.predict(X_scaled_injected, verbose=0)

# Only proceed if reconstructions were successful
if vae_reconstructions is not None and len(vae_reconstructions) == len(X_scaled_injected):
    vae_feature_errors = np.abs(X_scaled_injected - vae_reconstructions)
    # Ensure weights are defined (they should be if the previous cells ran)
    # You might adjust weights here specifically for post-injection analysis if desired,
    # or reuse the weights defined earlier based on feature importance.
    if 'weights' not in locals():
         weights = np.array([1.5 if 'Gage' in col or 'Align' in col else 1.0 for col in features_post_selection])

    vae_anomaly_scores_post = np.mean(vae_feature_errors * weights, axis=1) # Renamed for clarity

    # VAE: Apply pre-injection threshold
    vae_predictions = (vae_anomaly_scores_post > vae_threshold).astype(int)
else:
    print("Warning: VAE reconstruction failed or length mismatch. Setting VAE predictions to 0.")
    vae_predictions = np.zeros(len(X_scaled_injected), dtype=int) # Default predictions if VAE failed
    vae_anomaly_scores_post = np.zeros(len(X_scaled_injected)) # Default scores


# Autoencoder: Compute anomaly scores (post-injection)
# Ensure the autoencoder model is available
if 'autoencoder' not in locals() and 'autoencoder' not in globals():
    print("Error: 'autoencoder' model not found. Cannot compute AE post-injection scores.")
    ae_reconstructions = None # Set to None or handle error appropriately
else:
    # Access the model from globals if not local
    ae_model = autoencoder if 'autoencoder' in locals() else globals()['autoencoder']
    # Predict on the injected data
    ae_reconstructions = ae_model.predict(X_scaled_injected, verbose=0)


# Only proceed if reconstructions were successful
if ae_reconstructions is not None and len(ae_reconstructions) == len(X_scaled_injected):
    # Assuming feature_weights is defined globally or before this cell
    # If AE also used weights, apply them here for consistency
    if 'weights' not in locals():
         weights = np.array([1.5 if 'Gage' in col or 'Align' in col else 1.0 for col in features_post_selection])

    ae_mse_post = np.mean(np.square(X_scaled_injected - ae_reconstructions) * weights, axis=1) # Renamed for clarity

    # Autoencoder: Apply pre-injection threshold
    ae_predictions = (ae_mse_post > ae_threshold).astype(int)
else:
    print("Warning: AE reconstruction failed or length mismatch. Setting AE predictions to 0.")
    ae_predictions = np.zeros(len(X_scaled_injected), dtype=int) # Default predictions if AE failed
    ae_mse_post = np.zeros(len(X_scaled_injected)) # Default scores


# Post-Injection Confusion Matrices (Only compute if prediction length matches true labels)
if len(y_true) == len(vae_predictions) and len(y_true) > 0:
    vae_cm_post = confusion_matrix(y_true, vae_predictions)
else:
    print("Warning: Post-injection VAE predictions length mismatch with y_true or y_true is empty. VAE post-injection CM not computed.")
    vae_cm_post = np.zeros((2,2), dtype=int) # Placeholder if data is missing or lengths mismatch

if len(y_true) == len(ae_predictions) and len(y_true) > 0:
    ae_cm_post = confusion_matrix(y_true, ae_predictions)
else:
     print("Warning: Post-injection AE predictions length mismatch with y_true or y_true is empty. AE post-injection CM not computed.")
     ae_cm_post = np.zeros((2,2), dtype=int) # Placeholder if data is missing or lengths mismatch

# Ensemble: Strict (AND) and Relaxed (OR)
# Only compute if both VAE and AE predictions were successfully generated
if len(vae_predictions) == len(ae_predictions) == len(y_true) and len(y_true) > 0:
    ensemble_strict_predictions = np.logical_and(vae_predictions, ae_predictions).astype(int)
    ensemble_relaxed_predictions = np.logical_or(vae_predictions, ae_predictions).astype(int)

    # Ensemble Post-Injection Confusion Matrices
    ensemble_strict_cm_post = confusion_matrix(y_true, ensemble_strict_predictions)
    ensemble_relaxed_cm_post = confusion_matrix(y_true, ensemble_relaxed_predictions)
else:
    print("Warning: Cannot compute ensemble predictions or confusion matrices due to prediction length mismatch or empty data.")
    ensemble_strict_predictions = np.zeros_like(y_true) if len(y_true) > 0 else np.array([])
    ensemble_relaxed_predictions = np.zeros_like(y_true) if len(y_true) > 0 else np.array([])
    ensemble_strict_cm_post = np.zeros((2,2), dtype=int)
    ensemble_relaxed_cm_post = np.zeros((2,2), dtype=int)


# Pre-Injection Results (Use calculated counts if predictions were successful)
vae_pre_anomalies = sum(vae_pre_predictions) if len(vae_pre_predictions) == len(X_scaled_df) else 0
ae_pre_anomalies = sum(ae_pre_predictions) if len(ae_pre_predictions) == len(X_scaled_df) else 0
vae_pre_percentage = (vae_pre_anomalies / len(X_scaled_df)) * 100 if len(X_scaled_df) > 0 else 0
ae_pre_percentage = (ae_pre_anomalies / len(X_scaled_df)) * 100 if len(X_scaled_df) > 0 else 0

# Post-Injection Results
vae_post_anomalies = sum(vae_predictions)
ae_post_anomalies = sum(ae_predictions)
ensemble_strict_post_anomalies = sum(ensemble_strict_predictions)
ensemble_relaxed_post_anomalies = sum(ensemble_relaxed_predictions)

vae_post_percentage = (vae_post_anomalies / len(X_scaled_injected)) * 100 if len(X_scaled_injected) > 0 else 0
ae_post_percentage = (ae_post_anomalies / len(X_scaled_injected)) * 100 if len(X_scaled_injected) > 0 else 0
ensemble_strict_percentage = (ensemble_strict_post_anomalies / len(X_scaled_injected)) * 100 if len(X_scaled_injected) > 0 else 0
ensemble_relaxed_percentage = (ensemble_relaxed_post_anomalies / len(X_scaled_injected)) * 100 if len(X_scaled_injected) > 0 else 0


# Print Anomaly Rate Comparison
print("=== Anomaly Rate Comparison ===")
print(f"VAE Pre-Injection 5: {vae_pre_anomalies}/{len(X_scaled_df)} ({vae_pre_percentage:.2f}%)" if len(X_scaled_df)>0 else "VAE Pre-Injection 5: N/A (X_scaled_df not found)")
print(f"VAE Post-Injection 5: {vae_post_anomalies}/{len(X_scaled_injected)} ({vae_post_percentage:.2f}%)" if len(X_scaled_injected)>0 else "VAE Post-Injection 5: N/A (X_scaled_injected not found)")
print(f"Autoencoder Pre-Injection 5: {ae_pre_anomalies}/{len(X_scaled_df)} ({(ae_pre_anomalies/len(X_scaled_df))*100:.2f}%)" if len(X_scaled_df)>0 else "Autoencoder Pre-Injection 5: N/A (X_scaled_df not found)")
print(f"Autoencoder Post-Injection 5: {ae_post_anomalies}/{len(X_scaled_injected)} ({(ae_post_anomalies/len(X_scaled_injected))*100:.2f}%)" if len(X_scaled_injected)>0 else "Autoencoder Post-Injection 5: N/A (X_scaled_injected not found)")
print(f"Ensemble Strict Post-Injection 5: {ensemble_strict_post_anomalies}/{len(X_scaled_injected)} ({ensemble_strict_percentage:.2f}%)" if len(X_scaled_injected)>0 else "Ensemble Strict Post-Injection 5: N/A (X_scaled_injected not found)")
print(f"Ensemble Relaxed Post-Injection 5: {ensemble_relaxed_post_anomalies}/{len(X_scaled_injected)} ({(ensemble_relaxed_post_anomalies/len(X_scaled_injected))*100:.2f}%)" if len(X_scaled_injected)>0 else "Ensemble Relaxed Post-Injection 5: N/A (X_scaled_injected not found)")


# Post-Injection Performance Metrics
# Calculate only if predictions are available and data length matches true labels
if len(y_true) == len(vae_predictions) and len(y_true) > 0:
    vae_precision, vae_recall, vae_f1 = precision_score(y_true, vae_predictions), recall_score(y_true, vae_predictions), f1_score(y_true, vae_predictions)
else:
    vae_precision, vae_recall, vae_f1 = 0.0, 0.0, 0.0 # Default if mismatch or empty data

if len(y_true) == len(ae_predictions) and len(y_true) > 0:
    ae_precision, ae_recall, ae_f1 = precision_score(y_true, ae_predictions), recall_score(y_true, ae_predictions), f1_score(y_true, ae_predictions)
else:
    ae_precision, ae_recall, ae_f1 = 0.0, 0.0, 0.0 # Default if mismatch or empty data

if len(y_true) == len(ensemble_strict_predictions) and len(y_true) > 0:
    ensemble_strict_precision, ensemble_strict_recall, ensemble_strict_f1 = precision_score(y_true, ensemble_strict_predictions), recall_score(y_true, ensemble_strict_predictions), f1_score(y_true, ensemble_strict_predictions)
else:
    ensemble_strict_precision, ensemble_strict_recall, ensemble_strict_f1 = 0.0, 0.0, 0.0 # Default if mismatch or empty data

if len(y_true) == len(ensemble_relaxed_predictions) and len(y_true) > 0:
    ensemble_relaxed_precision, ensemble_relaxed_recall, ensemble_relaxed_f1 = precision_score(y_true, ensemble_relaxed_predictions), recall_score(y_true, ensemble_relaxed_predictions), f1_score(y_true, ensemble_relaxed_predictions)
else:
    ensemble_relaxed_precision, ensemble_relaxed_recall, ensemble_relaxed_f1 = 0.0, 0.0, 0.0 # Default if mismatch or empty data


# False Positives Before Injection
vae_pre_fp = vae_pre_anomalies
ae_pre_fp = ae_pre_anomalies

# False Positives After Injection
vae_post_fp = vae_cm_post[0, 1]
ae_post_fp = ae_cm_post[0, 1]
ensemble_strict_fp = ensemble_strict_cm_post[0, 1]
ensemble_relaxed_fp = ensemble_relaxed_cm_post[0, 1]

# Print False Positive Comparison
print("\n=== False Positives Comparison ===")
print(f"VAE Pre-Injection False Positives 5: {vae_pre_fp}/{len(X_scaled_df)} ({(vae_pre_fp/len(X_scaled_df))*100:.2f}%)" if len(X_scaled_df) > 0 else "VAE Pre-Injection False Positives 5: N/A")
print(f"VAE Post-Injection False Positives 5: {vae_post_fp}/{len(X_scaled_injected)} ({(vae_post_fp/len(X_scaled_injected))*100:.2f}%)" if len(X_scaled_injected) > 0 else "VAE Post-Injection False Positives 5: N/A")
print(f"Autoencoder Pre-Injection False Positives 5: {ae_pre_fp}/{len(X_scaled_df)} ({(ae_pre_fp/len(X_scaled_df))*100:.2f}%)" if len(X_scaled_df) > 0 else "Autoencoder Pre-Injection False Positives 5: N/A")
print(f"Autoencoder Post-Injection False Positives 5: {ae_post_fp}/{len(X_scaled_injected)} ({(ae_post_fp/len(X_scaled_injected))*100:.2f}%)" if len(X_scaled_injected) > 0 else "Autoencoder Post-Injection False Positives 5: N/A")
print(f"Ensemble Strict Post-Injection False Positives 5: {ensemble_strict_fp}/{len(X_scaled_injected)} ({(ensemble_strict_fp/len(X_scaled_injected))*100:.2f}%)" if len(X_scaled_injected) > 0 else "Ensemble Strict Post-Injection False Positives 5: N/A")
print(f"Ensemble Relaxed Post-Injection False Positives 5: {ensemble_relaxed_fp}/{len(X_scaled_injected)} ({(ensemble_relaxed_fp/len(X_scaled_injected))*100:.2f}%)" if len(X_scaled_injected) > 0 else "Ensemble Relaxed Post-Injection False Positives 5: N/A")


# Visualization
plt.figure(figsize=(14, 20))

# Anomaly Rate Comparison (Bar Plot)
plt.subplot(4, 2, 1)
models = ['VAE', 'Autoencoder', 'Ensemble Strict', 'Ensemble Relaxed']
pre_rates = [vae_pre_percentage, ae_pre_percentage, 0, 0]  # Ensemble not applicable pre-injection
post_rates = [vae_post_percentage, ae_post_percentage, ensemble_strict_percentage, ensemble_relaxed_percentage]
x = np.arange(len(models))
width = 0.35
plt.bar(x - width/2, pre_rates, width, label='Pre-Injection', color='teal')
plt.bar(x + width/2, post_rates, width, label='Post-Injection', color='blue')
plt.xticks(x, models, fontsize=10, rotation=45)
plt.ylabel('Anomaly Percentage (%)', fontsize=10)
plt.title('Anomaly Detection Rate Before vs After Injection 5', fontsize=12)
plt.legend(fontsize=9)

# Performance Metrics (Post-Injection) - Radar Chart
plt.subplot(4, 2, 2)
metrics = ['Precision', 'Recall', 'F1-Score']
vae_metrics = [vae_precision, vae_recall, vae_f1]
ae_metrics = [ae_precision, ae_recall, ae_f1]
ensemble_strict_metrics = [ensemble_strict_precision, ensemble_strict_recall, ensemble_strict_f1]
ensemble_relaxed_metrics = [ensemble_relaxed_precision, ensemble_relaxed_recall, ensemble_relaxed_f1]
angles = np.linspace(0, 2 * np.pi, len(metrics), endpoint=False).tolist()
# Ensure closing point is added only if metrics exist
if len(metrics) > 0:
    angles += angles[:1]
    vae_metrics += vae_metrics[:1]
    ae_metrics += ae_metrics[:1]
    ensemble_strict_metrics += ensemble_strict_metrics[:1]
    ensemble_relaxed_metrics += ensemble_relaxed_metrics[:1]

ax = plt.subplot(4, 2, 2, polar=True)
ax.fill(angles, vae_metrics, 'teal', alpha=0.5, label='VAE')
ax.fill(angles, ae_metrics, 'purple', alpha=0.5, label='Autoencoder')
ax.fill(angles, ensemble_strict_metrics, 'green', alpha=0.5, label='Ensemble Strict')
ax.fill(angles, ensemble_relaxed_metrics, 'orange', alpha=0.5, label='Ensemble Relaxed')
# Set xticks only if angles exist
if len(angles) > 1:
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(metrics, fontsize=10)
plt.title('Post-Injection Performance Metrics 5', fontsize=12)
plt.legend(loc='upper right', bbox_to_anchor=(1.1, 1.1), fontsize=9)

# VAE Confusion Matrix - Pre-Injection
plt.subplot(4, 2, 3)
sns.heatmap(vae_cm_pre, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=['Normal', 'Anomaly'], yticklabels=['Normal', 'Anomaly'],
            annot_kws={'size': 10})
plt.title("VAE Confusion Matrix (Pre-Injection) 5", fontsize=12)
plt.xlabel("Predicted", fontsize=10)
plt.ylabel("True", fontsize=10)

# VAE Confusion Matrix - Post-Injection
plt.subplot(4, 2, 4)
sns.heatmap(vae_cm_post, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=['Normal', 'Intrusion'], yticklabels=['Normal', 'Intrusion'],
            annot_kws={'size': 10})
plt.title("VAE Confusion Matrix (Post-Injection) 5", fontsize=12)
plt.xlabel("Predicted", fontsize=10)
plt.ylabel("True", fontsize=10)

# Autoencoder Confusion Matrix - Pre-Injection
plt.subplot(4, 2, 5)
sns.heatmap(ae_cm_pre, annot=True, fmt='d', cmap='Purples', cbar=False,
            xticklabels=['Normal', 'Anomaly'], yticklabels=['Normal', 'Anomaly'],
            annot_kws={'size': 10})
plt.title("Autoencoder Confusion Matrix (Pre-Injection) 5", fontsize=12)
plt.xlabel("Predicted", fontsize=10)
plt.ylabel("True", fontsize=10)

# Autoencoder Confusion Matrix - Post-Injection
plt.subplot(4, 2, 6)
sns.heatmap(ae_cm_post, annot=True, fmt='d', cmap='Purples', cbar=False,
            xticklabels=['Normal', 'Intrusion'], yticklabels=['Normal', 'Intrusion'],
            annot_kws={'size': 10})
plt.title("Autoencoder Confusion Matrix (Post-Injection) 5", fontsize=12)
plt.xlabel("Predicted", fontsize=10)
plt.ylabel("True", fontsize=10)

# Ensemble Strict Confusion Matrix - Post-Injection
plt.subplot(4, 2, 7)
sns.heatmap(ensemble_strict_cm_post, annot=True, fmt='d', cmap='Greens', cbar=False,
            xticklabels=['Normal', 'Intrusion'], yticklabels=['Normal', 'Intrusion'],
            annot_kws={'size': 10})
plt.title("Ensemble Strict Confusion Matrix (Post-Injection) 5", fontsize=12)
plt.xlabel("Predicted", fontsize=10)
plt.ylabel("True", fontsize=10)

# Ensemble Relaxed Confusion Matrix - Post-Injection
plt.subplot(4, 2, 8)
sns.heatmap(ensemble_relaxed_cm_post, annot=True, fmt='d', cmap='Oranges', cbar=False,
            xticklabels=['Normal', 'Intrusion'], yticklabels=['Normal', 'Intrusion'],
            annot_kws={'size': 10})
plt.title("Ensemble Relaxed Confusion Matrix (Post-Injection) 5", fontsize=12)
plt.xlabel("Predicted", fontsize=10)
plt.ylabel("True", fontsize=10)

plt.tight_layout(pad=2.0)
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score

# Prepare the injected data for testing
features_post_selection = X.columns.tolist()
# Assuming scaler was fitted on the training data (X_scaled_df)
# If df_injected contains the exact same features and order as df,
# then using the scaler fitted on the original data is correct for consistency.
# If not, you might need to refit the scaler or ensure df_injected aligns with the features used for training.
# For now, assuming df_injected is aligned and using the pre-fitted scaler is intended.
# Access the pre-fitted scaler from globals if it's not in local scope
if 'scaler' not in locals():
    print("Warning: 'scaler' not found in local scope. Ensure it was fitted on training data.")
    # Fallback: Refit scaler on the injected data (less ideal if training data was different)
    # scaler = StandardScaler()
    # X_scaled_injected = scaler.fit_transform(df_injected[features_post_selection])
    # Note: This fallback is commented out as using the pre-fitted scaler is generally preferred.
    # You might need to ensure 'scaler' is available from previous cells.

# Assume X_scaled_injected is already correctly created from df_injected elsewhere,
# using the scaler fitted on the pre-injection data (X_scaled_df).
# If X_scaled_injected isn't in globals, you'll need to add:
# scaler = StandardScaler() # Make sure this is the *same* scaler used for X_scaled_df
# X_scaled_injected = scaler.fit_transform(df_injected[features_post_selection])
# For now, using the global variable X_scaled_injected.

y_true = df_injected['label'].values  # True labels: 0 (normal), 1 (intrusion)

# Pre-Injection Ground Truth (all normal, label=0)
# Assuming X_scaled_df represents the pre-injection data
y_true_pre = np.zeros(len(X_scaled_df))

# Pre-Injection Predictions
# FIX: Use the correct variable name 'anomalies_vae' for VAE pre-injection predictions
# Check if 'anomalies_vae' exists in globals
if 'anomalies_vae' in globals():
    vae_pre_predictions = globals()['anomalies_vae'].astype(int)
elif 'anomalies_vae' in locals(): # Check local scope as well
     vae_pre_predictions = anomalies_vae.astype(int)
else:
    print("Warning: 'anomalies_vae' not found. Cannot evaluate VAE pre-injection performance.")
    # Provide a fallback or handle the error
    vae_pre_predictions = np.zeros(len(X_scaled_df), dtype=int)


# 'anomalies_ae' was correctly referenced for AE pre-injection predictions
if 'anomalies_ae' in globals():
    ae_pre_predictions = globals()['anomalies_ae'].astype(int)
elif 'anomalies_ae' in locals(): # Check local scope as well
    ae_pre_predictions = anomalies_ae.astype(int)
else:
    print("Warning: 'anomalies_ae' not found. Cannot evaluate Autoencoder pre-injection performance.")
    # Provide a fallback or handle the error
    ae_pre_predictions = np.zeros(len(X_scaled_df), dtype=int)


# Pre-Injection Confusion Matrices
# Only compute if prediction length matches true labels
if len(y_true_pre) == len(vae_pre_predictions) and len(y_true_pre) > 0:
    vae_cm_pre = confusion_matrix(y_true_pre, vae_pre_predictions)
else:
    vae_cm_pre = np.zeros((2,2), dtype=int) # Placeholder if data is missing or lengths mismatch

if len(y_true_pre) == len(ae_pre_predictions) and len(y_true_pre) > 0:
    ae_cm_pre = confusion_matrix(y_true_pre, ae_pre_predictions)
else:
    ae_cm_pre = np.zeros((2,2), dtype=int) # Placeholder if data is missing or lengths mismatch


# Set Thresholds Using Pre-Injection Data
# FIX: Use the correct variable names for pre-injection scores
# Based on globals and previous cells, 'vae_pre_injection_scores' holds VAE scores
if 'vae_pre_injection_scores' in globals():
     vae_threshold = np.percentile(globals()['vae_pre_injection_scores'], 90.05)
     print(f"Using pre-calculated VAE threshold: {vae_threshold:.4f}")
elif 'vae_pre_injection_scores' in locals(): # Check local scope
     vae_threshold = np.percentile(vae_pre_injection_scores, 90.05)
     print(f"Using pre-calculated VAE threshold: {vae_threshold:.4f}")
else:
     print("Warning: 'vae_pre_injection_scores' not found. VAE threshold not set from pre-injection data.")
     # Provide a fallback - calculating on post-injection data is less ideal
     vae_threshold = np.percentile(vae_anomaly_scores_post, 90.05) if 'vae_anomaly_scores_post' in locals() and len(vae_anomaly_scores_post) > 0 else 0.0
     print(f"Calculating VAE threshold on post-injection data as fallback: {vae_threshold:.4f}")


# Based on globals and previous cells, 'mse_weighted' holds AE scores
if 'mse_weighted' in globals():
     ae_threshold = np.percentile(globals()['mse_weighted'], 90.05)
     print(f"Using pre-calculated AE threshold: {ae_threshold:.4f}")
elif 'mse_weighted' in locals(): # Check local scope
     ae_threshold = np.percentile(mse_weighted, 90.05)
     print(f"Using pre-calculated AE threshold: {ae_threshold:.4f}")
else:
    print("Warning: 'mse_weighted' (AE pre-injection scores) not found. AE threshold not set from pre-injection data.")
    # Provide a fallback - calculating on post-injection data is less ideal
    ae_threshold = np.percentile(ae_mse_post, 90.05) if 'ae_mse_post' in locals() and len(ae_mse_post) > 0 else 0.0
    print(f"Calculating AE threshold on post-injection data as fallback: {ae_threshold:.4f}")


# VAE: Compute anomaly scores (post-injection)
# Ensure the vae model is available
if 'vae' not in locals() and 'vae' not in globals():
     print("Error: 'vae' model not found. Cannot compute VAE post-injection scores.")
     vae_reconstructions = None # Set to None or handle error appropriately
else:
    # Access the model from globals if not local
    vae_model = vae if 'vae' in locals() else globals()['vae']
    # Predict on the injected data
    vae_reconstructions = vae_model.predict(X_scaled_injected, verbose=0)

# Only proceed if reconstructions were successful
if vae_reconstructions is not None and len(vae_reconstructions) == len(X_scaled_injected):
    vae_feature_errors = np.abs(X_scaled_injected - vae_reconstructions)
    # Ensure weights are defined (they should be if the previous cells ran)
    # You might adjust weights here specifically for post-injection analysis if desired,
    # or reuse the weights defined earlier based on feature importance.
    if 'weights' not in locals():
         weights = np.array([1.5 if 'Gage' in col or 'Align' in col else 1.0 for col in features_post_selection])

    vae_anomaly_scores_post = np.mean(vae_feature_errors * weights, axis=1) # Renamed for clarity

    # VAE: Apply pre-injection threshold
    vae_predictions = (vae_anomaly_scores_post > vae_threshold).astype(int)
else:
    print("Warning: VAE reconstruction failed or length mismatch. Setting VAE predictions to 0.")
    vae_predictions = np.zeros(len(X_scaled_injected), dtype=int) # Default predictions if VAE failed
    vae_anomaly_scores_post = np.zeros(len(X_scaled_injected)) # Default scores


# Autoencoder: Compute anomaly scores (post-injection)
# Ensure the autoencoder model is available
if 'autoencoder' not in locals() and 'autoencoder' not in globals():
    print("Error: 'autoencoder' model not found. Cannot compute AE post-injection scores.")
    ae_reconstructions = None # Set to None or handle error appropriately
else:
    # Access the model from globals if not local
    ae_model = autoencoder if 'autoencoder' in locals() else globals()['autoencoder']
    # Predict on the injected data
    ae_reconstructions = ae_model.predict(X_scaled_injected, verbose=0)


# Only proceed if reconstructions were successful
if ae_reconstructions is not None and len(ae_reconstructions) == len(X_scaled_injected):
    # Assuming feature_weights is defined globally or before this cell
    # If AE also used weights, apply them here for consistency
    if 'weights' not in locals():
         weights = np.array([1.5 if 'Gage' in col or 'Align' in col else 1.0 for col in features_post_selection])

    ae_mse_post = np.mean(np.square(X_scaled_injected - ae_reconstructions) * weights, axis=1) # Renamed for clarity

    # Autoencoder: Apply pre-injection threshold
    ae_predictions = (ae_mse_post > ae_threshold).astype(int)
else:
    print("Warning: AE reconstruction failed or length mismatch. Setting AE predictions to 0.")
    ae_predictions = np.zeros(len(X_scaled_injected), dtype=int) # Default predictions if AE failed
    ae_mse_post = np.zeros(len(X_scaled_injected)) # Default scores


# Post-Injection Confusion Matrices (Only compute if prediction length matches true labels)
if len(y_true) == len(vae_predictions) and len(y_true) > 0:
    vae_cm_post = confusion_matrix(y_true, vae_predictions)
else:
    print("Warning: Post-injection VAE predictions length mismatch with y_true or y_true is empty. VAE post-injection CM not computed.")
    vae_cm_post = np.zeros((2,2), dtype=int) # Placeholder if data is missing or lengths mismatch

if len(y_true) == len(ae_predictions) and len(y_true) > 0:
    ae_cm_post = confusion_matrix(y_true, ae_predictions)
else:
     print("Warning: Post-injection AE predictions length mismatch with y_true or y_true is empty. AE post-injection CM not computed.")
     ae_cm_post = np.zeros((2,2), dtype=int) # Placeholder if data is missing or lengths mismatch

# Ensemble: Strict (AND) and Relaxed (OR)
# Only compute if both VAE and AE predictions were successfully generated
if len(vae_predictions) == len(ae_predictions) == len(y_true) and len(y_true) > 0:
    ensemble_strict_predictions = np.logical_and(vae_predictions, ae_predictions).astype(int)
    ensemble_relaxed_predictions = np.logical_or(vae_predictions, ae_predictions).astype(int)

    # Ensemble Post-Injection Confusion Matrices
    ensemble_strict_cm_post = confusion_matrix(y_true, ensemble_strict_predictions)
    ensemble_relaxed_cm_post = confusion_matrix(y_true, ensemble_relaxed_predictions)
else:
    print("Warning: Cannot compute ensemble predictions or confusion matrices due to prediction length mismatch or empty data.")
    ensemble_strict_predictions = np.zeros_like(y_true) if len(y_true) > 0 else np.array([])
    ensemble_relaxed_predictions = np.zeros_like(y_true) if len(y_true) > 0 else np.array([])
    ensemble_strict_cm_post = np.zeros((2,2), dtype=int)
    ensemble_relaxed_cm_post = np.zeros((2,2), dtype=int)


# Pre-Injection Results (Use calculated counts if predictions were successful)
vae_pre_anomalies = sum(vae_pre_predictions) if len(vae_pre_predictions) == len(X_scaled_df) else 0
ae_pre_anomalies = sum(ae_pre_predictions) if len(ae_pre_predictions) == len(X_scaled_df) else 0
vae_pre_percentage = (vae_pre_anomalies / len(X_scaled_df)) * 100 if len(X_scaled_df) > 0 else 0
ae_pre_percentage = (ae_pre_anomalies / len(X_scaled_df)) * 100 if len(X_scaled_df) > 0 else 0

# Post-Injection Results
vae_post_anomalies = sum(vae_predictions)
ae_post_anomalies = sum(ae_predictions)
ensemble_strict_post_anomalies = sum(ensemble_strict_predictions)
ensemble_relaxed_post_anomalies = sum(ensemble_relaxed_predictions)

vae_post_percentage = (vae_post_anomalies / len(X_scaled_injected)) * 100 if len(X_scaled_injected) > 0 else 0
ae_post_percentage = (ae_post_anomalies / len(X_scaled_injected)) * 100 if len(X_scaled_injected) > 0 else 0
ensemble_strict_percentage = (ensemble_strict_post_anomalies / len(X_scaled_injected)) * 100 if len(X_scaled_injected) > 0 else 0
ensemble_relaxed_percentage = (ensemble_relaxed_post_anomalies / len(X_scaled_injected)) * 100 if len(X_scaled_injected) > 0 else 0


# Print Anomaly Rate Comparison
print("=== Anomaly Rate Comparison ===")
print(f"VAE Pre-Injection 5: {vae_pre_anomalies}/{len(X_scaled_df)} ({vae_pre_percentage:.2f}%)" if len(X_scaled_df)>0 else "VAE Pre-Injection 5: N/A (X_scaled_df not found)")
print(f"VAE Post-Injection 5: {vae_post_anomalies}/{len(X_scaled_injected)} ({vae_post_percentage:.2f}%)" if len(X_scaled_injected)>0 else "VAE Post-Injection 5: N/A (X_scaled_injected not found)")
print(f"Autoencoder Pre-Injection 5: {ae_pre_anomalies}/{len(X_scaled_df)} ({(ae_pre_anomalies/len(X_scaled_df))*100:.2f}%)" if len(X_scaled_df)>0 else "Autoencoder Pre-Injection 5: N/A (X_scaled_df not found)")
print(f"Autoencoder Post-Injection 5: {ae_post_anomalies}/{len(X_scaled_injected)} ({(ae_post_anomalies/len(X_scaled_injected))*100:.2f}%)" if len(X_scaled_injected)>0 else "Autoencoder Post-Injection 5: N/A (X_scaled_injected not found)")
print(f"Ensemble Strict Post-Injection 5: {ensemble_strict_post_anomalies}/{len(X_scaled_injected)} ({ensemble_strict_percentage:.2f}%)" if len(X_scaled_injected)>0 else "Ensemble Strict Post-Injection 5: N/A (X_scaled_injected not found)")
print(f"Ensemble Relaxed Post-Injection 5: {ensemble_relaxed_post_anomalies}/{len(X_scaled_injected)} ({(ensemble_relaxed_post_anomalies/len(X_scaled_injected))*100:.2f}%)" if len(X_scaled_injected)>0 else "Ensemble Relaxed Post-Injection 5: N/A (X_scaled_injected not found)")


# Post-Injection Performance Metrics
# Calculate only if predictions are available and data length matches true labels
if len(y_true) == len(vae_predictions) and len(y_true) > 0:
    vae_precision, vae_recall, vae_f1 = precision_score(y_true, vae_predictions), recall_score(y_true, vae_predictions), f1_score(y_true, vae_predictions)
else:
    vae_precision, vae_recall, vae_f1 = 0.0, 0.0, 0.0 # Default if mismatch or empty data

if len(y_true) == len(ae_predictions) and len(y_true) > 0:
    ae_precision, ae_recall, ae_f1 = precision_score(y_true, ae_predictions), recall_score(y_true, ae_predictions), f1_score(y_true, ae_predictions)
else:
    ae_precision, ae_recall, ae_f1 = 0.0, 0.0, 0.0 # Default if mismatch or empty data

if len(y_true) == len(ensemble_strict_predictions) and len(y_true) > 0:
    ensemble_strict_precision, ensemble_strict_recall, ensemble_strict_f1 = precision_score(y_true, ensemble_strict_predictions), recall_score(y_true, ensemble_strict_predictions), f1_score(y_true, ensemble_strict_predictions)
else:
    ensemble_strict_precision, ensemble_strict_recall, ensemble_strict_f1 = 0.0, 0.0, 0.0 # Default if mismatch or empty data

if len(y_true) == len(ensemble_relaxed_predictions) and len(y_true) > 0:
    ensemble_relaxed_precision, ensemble_relaxed_recall, ensemble_relaxed_f1 = precision_score(y_true, ensemble_relaxed_predictions), recall_score(y_true, ensemble_relaxed_predictions), f1_score(y_true, ensemble_relaxed_predictions)
else:
    ensemble_relaxed_precision, ensemble_relaxed_recall, ensemble_relaxed_f1 = 0.0, 0.0, 0.0 # Default if mismatch or empty data


# False Positives Before Injection
vae_pre_fp = vae_pre_anomalies
ae_pre_fp = ae_pre_anomalies

# False Positives After Injection
vae_post_fp = vae_cm_post[0, 1]
ae_post_fp = ae_cm_post[0, 1]
ensemble_strict_fp = ensemble_strict_cm_post[0, 1]
ensemble_relaxed_fp = ensemble_relaxed_cm_post[0, 1]

# Print False Positive Comparison
print("\n=== False Positives Comparison ===")
print(f"VAE Pre-Injection False Positives 5: {vae_pre_fp}/{len(X_scaled_df)} ({(vae_pre_fp/len(X_scaled_df))*100:.2f}%)" if len(X_scaled_df) > 0 else "VAE Pre-Injection False Positives 5: N/A")
print(f"VAE Post-Injection False Positives 5: {vae_post_fp}/{len(X_scaled_injected)} ({(vae_post_fp/len(X_scaled_injected))*100:.2f}%)" if len(X_scaled_injected) > 0 else "VAE Post-Injection False Positives 5: N/A")
print(f"Autoencoder Pre-Injection False Positives 5: {ae_pre_fp}/{len(X_scaled_df)} ({(ae_pre_fp/len(X_scaled_df))*100:.2f}%)" if len(X_scaled_df) > 0 else "Autoencoder Pre-Injection False Positives 5: N/A")
print(f"Autoencoder Post-Injection False Positives 5: {ae_post_fp}/{len(X_scaled_injected)} ({(ae_post_fp/len(X_scaled_injected))*100:.2f}%)" if len(X_scaled_injected) > 0 else "Autoencoder Post-Injection False Positives 5: N/A")
print(f"Ensemble Strict Post-Injection False Positives 5: {ensemble_strict_fp}/{len(X_scaled_injected)} ({(ensemble_strict_fp/len(X_scaled_injected))*100:.2f}%)" if len(X_scaled_injected) > 0 else "Ensemble Strict Post-Injection False Positives 5: N/A")
print(f"Ensemble Relaxed Post-Injection False Positives 5: {ensemble_relaxed_fp}/{len(X_scaled_injected)} ({(ensemble_relaxed_fp/len(X_scaled_injected))*100:.2f}%)" if len(X_scaled_injected) > 0 else "Ensemble Relaxed Post-Injection False Positives 5: N/A")


# Visualization
plt.figure(figsize=(14, 20))

# Anomaly Rate Comparison (Bar Plot)
plt.subplot(4, 2, 1)
models = ['VAE', 'Autoencoder', 'Ensemble Strict', 'Ensemble Relaxed'] # Removed Ensemble Weighted from here
pre_rates = [vae_pre_percentage, ae_pre_percentage, 0, 0]  # Ensemble not applicable pre-injection
post_rates = [vae_post_percentage, ae_post_percentage, ensemble_strict_percentage, ensemble_relaxed_percentage]
x = np.arange(len(models))
width = 0.35
plt.bar(x - width/2, pre_rates, width, label='Pre-Injection', color='teal')
plt.bar(x + width/2, post_rates, width, label='Post-Injection', color='blue')
plt.xticks(x, models, fontsize=10, rotation=45)
plt.ylabel('Anomaly Percentage (%)', fontsize=10)
plt.title('Anomaly Detection Rate Before vs After Injection 5', fontsize=12)
plt.legend(fontsize=9)

# Performance Metrics (Post-Injection) - Radar Chart
plt.subplot(4, 2, 2)
metrics = ['Precision', 'Recall', 'F1-Score']
vae_metrics = [vae_precision, vae_recall, vae_f1]
ae_metrics = [ae_precision, ae_recall, ae_f1]
ensemble_strict_metrics = [ensemble_strict_precision, ensemble_strict_recall, ensemble_strict_f1]
ensemble_relaxed_metrics = [ensemble_relaxed_precision, ensemble_relaxed_recall, ensemble_relaxed_f1]
# ensemble_weighted_metrics = [ensemble_weighted_precision, ensemble_weighted_recall, ensemble_weighted_f1] # Removed Weighted from metrics list
angles = np.linspace(0, 2 * np.pi, len(metrics), endpoint=False).tolist()
# Ensure closing point is added only if metrics exist
if len(metrics) > 0:
    angles += angles[:1]
    vae_metrics += vae_metrics[:1]
    ae_metrics += ae_metrics[:1]
    ensemble_strict_metrics += ensemble_strict_metrics[:1]
    ensemble_relaxed_metrics += ensemble_relaxed_metrics[:1]
    # ensemble_weighted_metrics += ensemble_weighted_metrics[:1] # Removed Weighted from metrics list

ax = plt.subplot(4, 2, 2, polar=True)
ax.fill(angles, vae_metrics, 'teal', alpha=0.5, label='VAE')
ax.fill(angles, ae_metrics, 'purple', alpha=0.5, label='Autoencoder')
ax.fill(angles, ensemble_strict_metrics, 'green', alpha=0.5, label='Ensemble Strict')
ax.fill(angles, ensemble_relaxed_metrics, 'orange', alpha=0.5, label='Ensemble Relaxed')
# ax.fill(angles, ensemble_weighted_metrics, 'red', alpha=0.5, label='Ensemble Weighted') # Removed Weighted plot
# Set xticks only if angles exist
if len(angles) > 1:
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(metrics, fontsize=10)
plt.title('Post-Injection Performance Metrics 5', fontsize=12)
plt.legend(loc='upper right', bbox_to_anchor=(1.1, 1.1), fontsize=9)

# VAE Confusion Matrix - Pre-Injection
plt.subplot(4, 2, 3)
sns.heatmap(vae_cm_pre, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=['Normal', 'Anomaly'], yticklabels=['Normal', 'Anomaly'],
            annot_kws={'size': 10})
plt.title("VAE Confusion Matrix (Pre-Injection) 5", fontsize=12)
plt.xlabel("Predicted", fontsize=10)
plt.ylabel("True", fontsize=10)

# VAE Confusion Matrix - Post-Injection
plt.subplot(4, 2, 4)
sns.heatmap(vae_cm_post, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=['Normal', 'Intrusion'], yticklabels=['Normal', 'Intrusion'],
            annot_kws={'size': 10})
plt.title("VAE Confusion Matrix (Post-Injection) 5", fontsize=12)
plt.xlabel("Predicted", fontsize=10)
plt.ylabel("True", fontsize=10)

# Autoencoder Confusion Matrix - Pre-Injection
plt.subplot(4, 2, 5)
sns.heatmap(ae_cm_pre, annot=True, fmt='d', cmap='Purples', cbar=False,
            xticklabels=['Normal', 'Anomaly'], yticklabels=['Normal', 'Anomaly'],
            annot_kws={'size': 10})
plt.title("Autoencoder Confusion Matrix (Pre-Injection) 5", fontsize=12)
plt.xlabel("Predicted", fontsize=10)
plt.ylabel("True", fontsize=10)

# Autoencoder Confusion Matrix - Post-Injection
plt.subplot(4, 2, 6)
sns.heatmap(ae_cm_post, annot=True, fmt='d', cmap='Purples', cbar=False,
            xticklabels=['Normal', 'Intrusion'], yticklabels=['Normal', 'Intrusion'],
            annot_kws={'size': 10})
plt.title("Autoencoder Confusion Matrix (Post-Injection) 5", fontsize=12)
plt.xlabel("Predicted", fontsize=10)
plt.ylabel("True", fontsize=10)

# Ensemble Strict Confusion Matrix - Post-Injection
plt.subplot(4, 2, 7)
sns.heatmap(ensemble_strict_cm_post, annot=True, fmt='d', cmap='Greens', cbar=False,
            xticklabels=['Normal', 'Intrusion'], yticklabels=['Normal', 'Intrusion'],
            annot_kws={'size': 10})
plt.title("Ensemble Strict Confusion Matrix (Post-Injection) 5", fontsize=12)
plt.xlabel("Predicted", fontsize=10)
plt.ylabel("True", fontsize=10)

# Ensemble Relaxed Confusion Matrix - Post-Injection
plt.subplot(4, 2, 8)
sns.heatmap(ensemble_relaxed_cm_post, annot=True, fmt='d', cmap='Oranges', cbar=False,
            xticklabels=['Normal', 'Intrusion'], yticklabels=['Normal', 'Intrusion'],
            annot_kws={'size': 10})
plt.title("Ensemble Relaxed Confusion Matrix (Post-Injection) 5", fontsize=12)
plt.xlabel("Predicted", fontsize=10)
plt.ylabel("True", fontsize=10)

plt.tight_layout(pad=2.0)
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score

# Prepare the injected data for testing
features_post_selection = X.columns.tolist()
scaler = StandardScaler()
X_scaled_injected = scaler.fit_transform(df_injected[features_post_selection])
y_true = df_injected['label'].values

# Pre-Injection Ground Truth (all normal, label=0)
y_true_pre = np.zeros(len(X_scaled_df))

# Pre-Injection Predictions
# Corrected: Use anomalies_vae for VAE pre-injection predictions
if 'anomalies_vae' in globals():
     vae_pre_predictions = globals()['anomalies_vae'].astype(int)
elif 'anomalies_vae' in locals():
     vae_pre_predictions = anomalies_vae.astype(int)
else:
    print("Warning: 'anomalies_vae' not found. Cannot evaluate VAE pre-injection performance.")
    vae_pre_predictions = np.zeros(len(X_scaled_df), dtype=int) # Fallback

ae_pre_predictions = anomalies_ae.astype(int)


# Pre-Injection Confusion Matrices
vae_cm_pre = confusion_matrix(y_true_pre, vae_pre_predictions)
ae_cm_pre = confusion_matrix(y_true_pre, ae_pre_predictions)

# Set Thresholds Using Pre-Injection Data
# Use the correct variable names for pre-injection scores
if 'vae_pre_injection_scores' in globals():
     vae_threshold = np.percentile(globals()['vae_pre_injection_scores'], 90.05)
elif 'vae_pre_injection_scores' in locals():
     vae_threshold = np.percentile(vae_pre_injection_scores, 90.05)
else:
     print("Warning: 'vae_pre_injection_scores' not found. VAE threshold not set from pre-injection data.")
     vae_threshold = 0.0 # Default or implement a fallback calculation

if 'mse_weighted' in globals():
     ae_threshold = np.percentile(globals()['mse_weighted'], 90.05)
elif 'mse_weighted' in locals():
     ae_threshold = np.percentile(mse_weighted, 90.05)
else:
    print("Warning: 'mse_weighted' (AE pre-injection scores) not found. AE threshold not set from pre-injection data.")
    ae_threshold = 0.0 # Default or implement a fallback calculation


# VAE: Compute anomaly scores (post-injection)
# Ensure the vae model is available
if 'vae' not in locals() and 'vae' not in globals():
     print("Error: 'vae' model not found. Cannot compute VAE post-injection scores.")
     vae_reconstructions = None # Set to None or handle error appropriately
else:
    # Access the model from globals if not local
    vae_model = vae if 'vae' in locals() else globals()['vae']
    # Predict on the injected data
    vae_reconstructions = vae_model.predict(X_scaled_injected, verbose=0)

# Only proceed if reconstructions were successful
if vae_reconstructions is not None and len(vae_reconstructions) == len(X_scaled_injected):
    vae_feature_errors = np.abs(X_scaled_injected - vae_reconstructions)
    # Ensure weights are defined (they should be if the previous cells ran)
    # You might adjust weights here specifically for post-injection analysis if desired,
    # or reuse the weights defined earlier based on feature importance.
    if 'weights' not in locals():
         weights = np.array([1.5 if 'Gage' in col or 'Align' in col else 1.0 for col in features_post_selection])

    vae_anomaly_scores = np.mean(vae_feature_errors * weights, axis=1)
    vae_predictions = (vae_anomaly_scores > vae_threshold).astype(int)
else:
    print("Warning: VAE reconstruction failed or length mismatch. Setting VAE predictions to 0.")
    vae_predictions = np.zeros(len(X_scaled_injected), dtype=int) # Default predictions if VAE failed
    vae_anomaly_scores = np.zeros(len(X_scaled_injected)) # Default scores


# Autoencoder: Compute anomaly scores (post-injection)
# Ensure the autoencoder model is available
if 'autoencoder' not in locals() and 'autoencoder' not in globals():
    print("Error: 'autoencoder' model not found. Cannot compute AE post-injection scores.")
    ae_reconstructions = None # Set to None or handle error appropriately
else:
    # Access the model from globals if not local
    ae_model = autoencoder if 'autoencoder' in locals() else globals()['autoencoder']
    # Predict on the injected data
    ae_reconstructions = ae_model.predict(X_scaled_injected, verbose=0)


# Only proceed if reconstructions were successful
if ae_reconstructions is not None and len(ae_reconstructions) == len(X_scaled_injected):
    # Assuming feature_weights is defined globally or before this cell
    # If AE also used weights, apply them here for consistency
    if 'weights' not in locals():
         weights = np.array([1.5 if 'Gage' in col or 'Align' in col else 1.0 for col in features_post_selection])

    ae_mse = np.mean(np.square(X_scaled_injected - ae_reconstructions) * weights, axis=1)

    # Autoencoder: Apply pre-injection threshold
    ae_predictions = (ae_mse > ae_threshold).astype(int)
else:
    print("Warning: AE reconstruction failed or length mismatch. Setting AE predictions to 0.")
    ae_predictions = np.zeros(len(X_scaled_injected), dtype=int) # Default predictions if AE failed
    ae_mse = np.zeros(len(X_scaled_injected)) # Default scores


# Ensemble: Strict (AND), Relaxed (OR), and Weighted Voting
# Only compute if both VAE and AE predictions were successfully generated
if len(vae_predictions) == len(ae_predictions) == len(y_true) and len(y_true) > 0:
    ensemble_strict_predictions = np.logical_and(vae_predictions, ae_predictions).astype(int)
    ensemble_relaxed_predictions = np.logical_or(vae_predictions, ae_predictions).astype(int)

    # Weighted Voting: Normalize scores relative to thresholds
    # Avoid division by zero if scores are all the same or equal to threshold
    vae_score_range = np.max(vae_anomaly_scores) - vae_threshold
    ae_score_range = np.max(ae_mse) - ae_threshold

    vae_normalized_scores = np.zeros_like(vae_anomaly_scores)
    ae_normalized_scores = np.zeros_like(ae_mse)

    if vae_score_range > 0:
        vae_normalized_scores = np.maximum(vae_anomaly_scores - vae_threshold, 0) / vae_score_range

    if ae_score_range > 0:
         ae_normalized_scores = np.maximum(ae_mse - ae_threshold, 0) / ae_score_range


    # Equal weights for now
    combined_scores = (vae_normalized_scores + ae_normalized_scores) / 2
    # Adjusted threshold to 0.05 for better sensitivity
    weighted_threshold = 0.05
    ensemble_weighted_predictions = (combined_scores > weighted_threshold).astype(int)

    # Ensemble Post-Injection Confusion Matrices
    ensemble_strict_cm_post = confusion_matrix(y_true, ensemble_strict_predictions)
    ensemble_relaxed_cm_post = confusion_matrix(y_true, ensemble_relaxed_predictions)
    ensemble_weighted_cm_post = confusion_matrix(y_true, ensemble_weighted_predictions)

else:
    print("Warning: Cannot compute ensemble predictions or confusion matrices due to prediction length mismatch or empty data.")
    ensemble_strict_predictions = np.zeros_like(y_true) if len(y_true) > 0 else np.array([])
    ensemble_relaxed_predictions = np.zeros_like(y_true) if len(y_true) > 0 else np.array([])
    # Initialize ensemble_weighted_predictions and combined_scores as well
    ensemble_weighted_predictions = np.zeros_like(y_true) if len(y_true) > 0 else np.array([])
    combined_scores = np.zeros_like(y_true, dtype=float) if len(y_true) > 0 else np.array([])

    ensemble_strict_cm_post = np.zeros((2,2), dtype=int)
    ensemble_relaxed_cm_post = np.zeros((2,2), dtype=int)
    ensemble_weighted_cm_post = np.zeros((2,2), dtype=int)
    weighted_threshold = 0.0 # Default threshold if not computed


# Pre-Injection Results
vae_pre_anomalies = sum(vae_pre_predictions) if len(vae_pre_predictions) == len(X_scaled_df) else 0
ae_pre_anomalies = sum(ae_pre_predictions) if len(ae_pre_predictions) == len(X_scaled_df) else 0
vae_pre_percentage = (vae_pre_anomalies / len(X_scaled_df)) * 100 if len(X_scaled_df) > 0 else 0
ae_pre_percentage = (ae_pre_anomalies / len(X_scaled_df)) * 100 if len(X_scaled_df) > 0 else 0


# Post-Injection Results
vae_post_anomalies = sum(vae_predictions) if len(vae_predictions) > 0 else 0
ae_post_anomalies = sum(ae_predictions) if len(ae_predictions) > 0 else 0
ensemble_strict_post_anomalies = sum(ensemble_strict_predictions) if len(ensemble_strict_predictions) > 0 else 0
ensemble_relaxed_post_anomalies = sum(ensemble_relaxed_predictions) if len(ensemble_relaxed_predictions) > 0 else 0
ensemble_weighted_post_anomalies = sum(ensemble_weighted_predictions) if len(ensemble_weighted_predictions) > 0 else 0

vae_post_percentage = (vae_post_anomalies / len(X_scaled_injected)) * 100 if len(X_scaled_injected) > 0 else 0
ae_post_percentage = (ae_post_anomalies / len(X_scaled_injected)) * 100 if len(X_scaled_injected) > 0 else 0
ensemble_strict_percentage = (ensemble_strict_post_anomalies / len(X_scaled_injected)) * 100 if len(X_scaled_injected) > 0 else 0
ensemble_relaxed_percentage = (ensemble_relaxed_post_anomalies / len(X_scaled_injected)) * 100 if len(X_scaled_injected) > 0 else 0
ensemble_weighted_percentage = (ensemble_weighted_post_anomalies / len(X_scaled_injected)) * 100 if len(X_scaled_injected) > 0 else 0

# Print Anomaly Rate Comparison
print("=== Anomaly Rate Comparison ===")
print(f"VAE Pre-Injection 7: {vae_pre_anomalies}/{len(X_scaled_df)} ({vae_pre_percentage:.2f}%)")
print(f"VAE Post-Injection 7: {vae_post_anomalies}/{len(X_scaled_injected)} ({vae_post_percentage:.2f}%)")
print(f"Autoencoder Pre-Injection 7: {ae_pre_anomalies}/{len(X_scaled_injected)} ({ae_pre_percentage:.2f}%)")
print(f"Autoencoder Post-Injection 7: {ae_post_anomalies}/{len(X_scaled_injected)} ({ae_post_percentage:.2f}%)")
print(f"Ensemble Strict Post-Injection 7: {ensemble_strict_post_anomalies}/{len(X_scaled_injected)} ({ensemble_strict_percentage:.2f}%)")
print(f"Ensemble Relaxed Post-Injection 7: {ensemble_relaxed_post_anomalies}/{len(X_scaled_injected)} ({ensemble_relaxed_percentage:.2f}%)")
print(f"Ensemble Weighted Post-Injection 7: {ensemble_weighted_post_anomalies}/{len(X_scaled_injected)} ({ensemble_weighted_percentage:.2f}%)")

# Post-Injection Performance Metrics
vae_precision, vae_recall, vae_f1 = precision_score(y_true, vae_predictions), recall_score(y_true, vae_predictions), f1_score(y_true, vae_predictions)
ae_precision, ae_recall, ae_f1 = precision_score(y_true, ae_predictions), recall_score(y_true, ae_predictions), f1_score(y_true, ae_predictions)
ensemble_strict_precision, ensemble_strict_recall, ensemble_strict_f1 = precision_score(y_true, ensemble_strict_predictions), recall_score(y_true, ensemble_strict_predictions), f1_score(y_true, ensemble_strict_predictions)
ensemble_relaxed_precision, ensemble_relaxed_recall, ensemble_relaxed_f1 = precision_score(y_true, ensemble_relaxed_predictions), recall_score(y_true, ensemble_relaxed_predictions), f1_score(y_true, ensemble_relaxed_predictions)
ensemble_weighted_precision, ensemble_weighted_recall, ensemble_weighted_f1 = precision_score(y_true, ensemble_weighted_predictions), recall_score(y_true, ensemble_weighted_predictions), f1_score(y_true, ensemble_weighted_predictions)

# False Positives Before Injection
vae_pre_fp = vae_pre_anomalies
ae_pre_fp = ae_pre_anomalies

# False Positives After Injection
vae_post_fp = vae_cm_post[0, 1]
ae_post_fp = ae_cm_post[0, 1]
ensemble_strict_fp = ensemble_strict_cm_post[0, 1]
ensemble_relaxed_fp = ensemble_relaxed_cm_post[0, 1]
ensemble_weighted_fp = ensemble_weighted_cm_post[0, 1]

# Print False Positive Comparison
print("\n=== False Positives Comparison ===")
print(f"VAE Pre-Injection False Positives 7: {vae_pre_fp}/{len(X_scaled_df)} ({(vae_pre_fp/len(X_scaled_df))*100:.2f}%)")
print(f"VAE Post-Injection False Positives 7: {vae_post_fp}/{len(X_scaled_injected)} ({(vae_post_fp/len(X_scaled_injected))*100:.2f}%)")
print(f"Autoencoder Pre-Injection False Positives 7: {ae_pre_fp}/{len(X_scaled_df)} ({(ae_pre_fp/len(X_scaled_df))*100:.2f}%)")
print(f"Autoencoder Post-Injection False Positives 7: {ae_post_fp}/{len(X_scaled_injected)} ({(ae_post_fp/len(X_scaled_injected))*100:.2f}%)")
print(f"Ensemble Strict Post-Injection False Positives 7: {ensemble_strict_fp}/{len(X_scaled_injected)} ({(ensemble_strict_fp/len(X_scaled_injected))*100:.2f}%)")
print(f"Ensemble Relaxed Post-Injection False Positives 7: {ensemble_relaxed_fp}/{len(X_scaled_injected)} ({(ensemble_relaxed_fp/len(X_scaled_injected))*100:.2f}%)")
print(f"Ensemble Weighted Post-Injection False Positives 7: {ensemble_weighted_fp}/{len(X_scaled_injected)} ({(ensemble_weighted_fp/len(X_scaled_injected))*100:.2f}%)")

# Visualization
plt.figure(figsize=(14, 20))

# Anomaly Rate Comparison (Bar Plot)
plt.subplot(4, 2, 1)
models = ['VAE', 'Autoencoder', 'Ensemble Strict', 'Ensemble Relaxed', 'Ensemble Weighted']
pre_rates = [vae_pre_percentage, ae_pre_percentage, 0, 0, 0]
post_rates = [vae_post_percentage, ae_post_percentage, ensemble_strict_percentage, ensemble_relaxed_percentage, ensemble_weighted_percentage]
x = np.arange(len(models))
width = 0.35
plt.bar(x - width/2, pre_rates, width, label='Pre-Injection', color='teal')
plt.bar(x + width/2, post_rates, width, label='Post-Injection', color='blue')
plt.xticks(x, models, fontsize=10, rotation=45)
plt.ylabel('Anomaly Percentage (%)', fontsize=10)
plt.title('Anomaly Detection Rate Before vs After Injection 7', fontsize=12)
plt.legend(fontsize=9)

# Performance Metrics (Post-Injection) - Radar Chart
plt.subplot(4, 2, 2)
metrics = ['Precision', 'Recall', 'F1-Score']
vae_metrics = [vae_precision, vae_recall, vae_f1]
ae_metrics = [ae_precision, ae_recall, ae_f1]
ensemble_strict_metrics = [ensemble_strict_precision, ensemble_strict_recall, ensemble_strict_f1]
ensemble_relaxed_metrics = [ensemble_relaxed_precision, ensemble_relaxed_recall, ensemble_relaxed_f1]
ensemble_weighted_metrics = [ensemble_weighted_precision, ensemble_weighted_recall, ensemble_weighted_f1]
angles = np.linspace(0, 2 * np.pi, len(metrics), endpoint=False).tolist()
angles += angles[:1]
vae_metrics += vae_metrics[:1]
ae_metrics += ae_metrics[:1]
ensemble_strict_metrics += ensemble_strict_metrics[:1]
ensemble_relaxed_metrics += ensemble_relaxed_metrics[:1]
ensemble_weighted_metrics += ensemble_weighted_metrics[:1]
ax = plt.subplot(4, 2, 2, polar=True)
ax.fill(angles, vae_metrics, 'teal', alpha=0.5, label='VAE')
ax.fill(angles, ae_metrics, 'purple', alpha=0.5, label='Autoencoder')
ax.fill(angles, ensemble_strict_metrics, 'green', alpha=0.5, label='Ensemble Strict')
ax.fill(angles, ensemble_relaxed_metrics, 'orange', alpha=0.5, label='Ensemble Relaxed')
ax.fill(angles, ensemble_weighted_metrics, 'red', alpha=0.5, label='Ensemble Weighted')
ax.set_xticks(angles[:-1])
ax.set_xticklabels(metrics, fontsize=10)
plt.title('Post-Injection Performance Metrics 7', fontsize=12)
plt.legend(loc='upper right', bbox_to_anchor=(1.1, 1.1), fontsize=9)

# VAE Confusion Matrix - Pre-Injection
plt.subplot(4, 2, 3)
sns.heatmap(vae_cm_pre, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=['Normal', 'Anomaly'], yticklabels=['Normal', 'Anomaly'],
            annot_kws={'size': 10})
plt.title("VAE Confusion Matrix (Pre-Injection) 7", fontsize=12)
plt.xlabel("Predicted", fontsize=10)
plt.ylabel("True", fontsize=10)

# VAE Confusion Matrix - Post-Injection
plt.subplot(4, 2, 4)
sns.heatmap(vae_cm_post, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=['Normal', 'Intrusion'], yticklabels=['Normal', 'Intrusion'],
            annot_kws={'size': 10})
plt.title("VAE Confusion Matrix (Post-Injection) 7", fontsize=12)
plt.xlabel("Predicted", fontsize=10)
plt.ylabel("True", fontsize=10)

# Autoencoder Confusion Matrix - Pre-Injection
plt.subplot(4, 2, 5)
sns.heatmap(ae_cm_pre, annot=True, fmt='d', cmap='Purples', cbar=False,
            xticklabels=['Normal', 'Anomaly'], yticklabels=['Normal', 'Anomaly'],
            annot_kws={'size': 10})
plt.title("Autoencoder Confusion Matrix (Pre-Injection) 7", fontsize=12)
plt.xlabel("Predicted", fontsize=10)
plt.ylabel("True", fontsize=10)

# Autoencoder Confusion Matrix - Post-Injection
plt.subplot(4, 2, 6)
sns.heatmap(ae_cm_post, annot=True, fmt='d', cmap='Purples', cbar=False,
            xticklabels=['Normal', 'Intrusion'], yticklabels=['Normal', 'Intrusion'],
            annot_kws={'size': 10})
plt.title("Autoencoder Confusion Matrix (Post-Injection) 7", fontsize=12)
plt.xlabel("Predicted", fontsize=10)
plt.ylabel("True", fontsize=10)

# Ensemble Strict Confusion Matrix - Post-Injection
plt.subplot(4, 2, 7)
sns.heatmap(ensemble_strict_cm_post, annot=True, fmt='d', cmap='Greens', cbar=False,
            xticklabels=['Normal', 'Intrusion'], yticklabels=['Normal', 'Intrusion'],
            annot_kws={'size': 10})
plt.title("Ensemble Strict Confusion Matrix (Post-Injection) 7", fontsize=12)
plt.xlabel("Predicted", fontsize=10)
plt.ylabel("True", fontsize=10)

# Combined Scores Distribution with Weighted Threshold
plt.subplot(4, 2, 8)
sns.histplot(combined_scores, bins=50, kde=True, color='red', alpha=0.5, label='Combined Scores')
plt.axvline(weighted_threshold, color='black', linestyle='--', label=f'Threshold: {weighted_threshold:.2f}')
plt.title('Combined Scores Distribution (Weighted Ensemble) 7', fontsize=12)
plt.xlabel('Combined Normalized Score', fontsize=10)
plt.ylabel('Frequency', fontsize=10)
plt.legend(fontsize=9)

plt.tight_layout(pad=2.0)
plt.show()

# Post-Injection Performance Metrics
vae_precision, vae_recall, vae_f1 = precision_score(y_true, vae_predictions), recall_score(y_true, vae_predictions), f1_score(y_true, vae_predictions)
ae_precision, ae_recall, ae_f1 = precision_score(y_true, ae_predictions), recall_score(y_true, ae_predictions), f1_score(y_true, ae_predictions)
ensemble_strict_precision, ensemble_strict_recall, ensemble_strict_f1 = precision_score(y_true, ensemble_strict_predictions), recall_score(y_true, ensemble_strict_predictions), f1_score(y_true, ensemble_strict_predictions)
ensemble_relaxed_precision, ensemble_relaxed_recall, ensemble_relaxed_f1 = precision_score(y_true, ensemble_relaxed_predictions), recall_score(y_true, ensemble_relaxed_predictions), f1_score(y_true, ensemble_relaxed_predictions)
ensemble_weighted_precision, ensemble_weighted_recall, ensemble_weighted_f1 = precision_score(y_true, ensemble_weighted_predictions), recall_score(y_true, ensemble_weighted_predictions), f1_score(y_true, ensemble_weighted_predictions)

# False Positives Before Injection
vae_pre_fp = vae_pre_anomalies
ae_pre_fp = ae_pre_anomalies

# False Positives After Injection
vae_post_fp = vae_cm_post[0, 1]
ae_post_fp = ae_cm_post[0, 1]
ensemble_strict_fp = ensemble_strict_cm_post[0, 1]
ensemble_relaxed_fp = ensemble_relaxed_cm_post[0, 1]
ensemble_weighted_fp = ensemble_weighted_cm_post[0, 1]

# Print False Positive Comparison
print("\n=== False Positives Comparison ===")
print(f"VAE Pre-Injection False Positives 7: {vae_pre_fp}/{len(X_scaled_df)} ({(vae_pre_fp/len(X_scaled_df))*100:.2f}%)")
print(f"VAE Post-Injection False Positives 7: {vae_post_fp}/{len(X_scaled_injected)} ({(vae_post_fp/len(X_scaled_injected))*100:.2f}%)")
print(f"Autoencoder Pre-Injection False Positives 7: {ae_pre_fp}/{len(X_scaled_df)} ({(ae_pre_fp/len(X_scaled_df))*100:.2f}%)")
print(f"Autoencoder Post-Injection False Positives 7: {ae_post_fp}/{len(X_scaled_injected)} ({(ae_post_fp/len(X_scaled_injected))*100:.2f}%)")
print(f"Ensemble Strict Post-Injection False Positives 7: {ensemble_strict_fp}/{len(X_scaled_injected)} ({(ensemble_strict_fp/len(X_scaled_injected))*100:.2f}%)")
print(f"Ensemble Relaxed Post-Injection False Positives 7: {ensemble_relaxed_fp}/{len(X_scaled_injected)} ({(ensemble_relaxed_fp/len(X_scaled_injected))*100:.2f}%)")
print(f"Ensemble Weighted Post-Injection False Positives 7: {ensemble_weighted_fp}/{len(X_scaled_injected)} ({(ensemble_weighted_fp/len(X_scaled_injected))*100:.2f}%)")

# Visualization
plt.figure(figsize=(14, 20))

# Anomaly Rate Comparison (Bar Plot)
plt.subplot(4, 2, 1)
models = ['VAE', 'Autoencoder', 'Ensemble Strict', 'Ensemble Relaxed', 'Ensemble Weighted']
pre_rates = [vae_pre_percentage, ae_pre_percentage, 0, 0, 0]
post_rates = [vae_post_percentage, ae_post_percentage, ensemble_strict_percentage, ensemble_relaxed_percentage, ensemble_weighted_percentage]
x = np.arange(len(models))
width = 0.35
plt.bar(x - width/2, pre_rates, width, label='Pre-Injection', color='teal')
plt.bar(x + width/2, post_rates, width, label='Post-Injection', color='blue')
plt.xticks(x, models, fontsize=10, rotation=45)
plt.ylabel('Anomaly Percentage (%)', fontsize=10)
plt.title('Anomaly Detection Rate Before vs After Injection 7', fontsize=12)
plt.legend(fontsize=9)

# Performance Metrics (Post-Injection) - Radar Chart
plt.subplot(4, 2, 2)
metrics = ['Precision', 'Recall', 'F1-Score']
vae_metrics = [vae_precision, vae_recall, vae_f1]
ae_metrics = [ae_precision, ae_recall, ae_f1]
ensemble_strict_metrics = [ensemble_strict_precision, ensemble_strict_recall, ensemble_strict_f1]
ensemble_relaxed_metrics = [ensemble_relaxed_precision, ensemble_relaxed_recall, ensemble_relaxed_f1]
ensemble_weighted_metrics = [ensemble_weighted_precision, ensemble_weighted_recall, ensemble_weighted_f1]
angles = np.linspace(0, 2 * np.pi, len(metrics), endpoint=False).tolist()
angles += angles[:1]
vae_metrics += vae_metrics[:1]
ae_metrics += ae_metrics[:1]
ensemble_strict_metrics += ensemble_strict_metrics[:1]
ensemble_relaxed_metrics += ensemble_relaxed_metrics[:1]
ensemble_weighted_metrics += ensemble_weighted_metrics[:1]
ax = plt.subplot(4, 2, 2, polar=True)
ax.fill(angles, vae_metrics, 'teal', alpha=0.5, label='VAE')
ax.fill(angles, ae_metrics, 'purple', alpha=0.5, label='Autoencoder')
ax.fill(angles, ensemble_strict_metrics, 'green', alpha=0.5, label='Ensemble Strict')
ax.fill(angles, ensemble_relaxed_metrics, 'orange', alpha=0.5, label='Ensemble Relaxed')
ax.fill(angles, ensemble_weighted_metrics, 'red', alpha=0.5, label='Ensemble Weighted')
ax.set_xticks(angles[:-1])
ax.set_xticklabels(metrics, fontsize=10)
plt.title('Post-Injection Performance Metrics 7', fontsize=12)
plt.legend(loc='upper right', bbox_to_anchor=(1.1, 1.1), fontsize=9)

# VAE Confusion Matrix - Pre-Injection
plt.subplot(4, 2, 3)
sns.heatmap(vae_cm_pre, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=['Normal', 'Anomaly'], yticklabels=['Normal', 'Anomaly'],
            annot_kws={'size': 10})
plt.title("VAE Confusion Matrix (Pre-Injection) 7", fontsize=12)
plt.xlabel("Predicted", fontsize=10)
plt.ylabel("True", fontsize=10)

# VAE Confusion Matrix - Post-Injection
plt.subplot(4, 2, 4)
sns.heatmap(vae_cm_post, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=['Normal', 'Intrusion'], yticklabels=['Normal', 'Intrusion'],
            annot_kws={'size': 10})
plt.title("VAE Confusion Matrix (Post-Injection) 7", fontsize=12)
plt.xlabel("Predicted", fontsize=10)
plt.ylabel("True", fontsize=10)

# Autoencoder Confusion Matrix - Pre-Injection
plt.subplot(4, 2, 5)
sns.heatmap(ae_cm_pre, annot=True, fmt='d', cmap='Purples', cbar=False,
            xticklabels=['Normal', 'Anomaly'], yticklabels=['Normal', 'Anomaly'],
            annot_kws={'size': 10})
plt.title("Autoencoder Confusion Matrix (Pre-Injection ) 7", fontsize=12)
plt.xlabel("Predicted", fontsize=10)
plt.ylabel("True", fontsize=10)

# Autoencoder Confusion Matrix - Post-Injection
plt.subplot(4, 2, 6)
sns.heatmap(ae_cm_post, annot=True, fmt='d', cmap='Purples', cbar=False,
            xticklabels=['Normal', 'Intrusion'], yticklabels=['Normal', 'Intrusion'],
            annot_kws={'size': 10})
plt.title("Autoencoder Confusion Matrix (Post-Injection)", fontsize=12)
plt.xlabel("Predicted", fontsize=10)
plt.ylabel("True", fontsize=10)

# Ensemble Strict Confusion Matrix - Post-Injection
plt.subplot(4, 2, 7)
sns.heatmap(ensemble_strict_cm_post, annot=True, fmt='d', cmap='Greens', cbar=False,
            xticklabels=['Normal', 'Intrusion'], yticklabels=['Normal', 'Intrusion'],
            annot_kws={'size': 10})
plt.title("Ensemble Strict Confusion Matrix (Post-Injection) 7", fontsize=12)
plt.xlabel("Predicted", fontsize=10)
plt.ylabel("True", fontsize=10)

# Combined Scores Distribution with Weighted Threshold
plt.subplot(4, 2, 8)
sns.histplot(combined_scores, bins=50, kde=True, color='red', alpha=0.5, label='Combined Scores')
plt.axvline(weighted_threshold, color='black', linestyle='--', label=f'Threshold: {weighted_threshold:.2f}')
plt.title('Combined Scores Distribution (Weighted Ensemble) 7', fontsize=12)
plt.xlabel('Combined Normalized Score', fontsize=10)
plt.ylabel('Frequency', fontsize=10)
plt.legend(fontsize=9)

plt.tight_layout(pad=2.0)
plt.show()

| Version | Model               | Detection Rate (%) | TP / 920 |
| ------- | ------------------- | ------------------ | -------- |
| 1       | VAE                 | 9.95               | \~78     |
| 1       | Autoencoder         | 9.95               | \~78     |
| 2       | VAE                 | 9.95               | 519      |
| 2       | Autoencoder         | 9.95               | 914      |
| 3       | VAE                 | 16.74              | 884      |
| 3       | Autoencoder         | 17.94              | 920 ✅    |
| 4       | Ensemble (Strict)   | 9.79               | 890      |
| 5       | Ensemble (Relaxed)  | 24.98              | 920 ✅    |
| 6       | Ensemble (Weighted) | 0.72               | 69       |
| 7       | Ensemble (Strict)   | 9.78               | 887      |
| 7       | Ensemble (Relaxed)  | 24.95              | 920 ✅    |
| 7       | Ensemble (Weighted) | 9.67               | 772      |


| Version | Model               | FP Count | FP Rate (%)            |
| ------- | ------------------- | -------- | ---------------------- |
| 1       | VAE                 | 1741     | 9.52                   |
| 1       | Autoencoder         | 1739     | 9.51                   |
| 2       | VAE                 | 1301     | 7.11                   |
| 2       | Autoencoder         | 906      | 4.95 ✅                 |
| 3       | VAE                 | 2178     | 11.91                  |
| 3       | Autoencoder         | 2362     | 12.91                  |
| 4       | Ensemble (Strict)   | 901      | 4.93 ✅                 |
| 5       | Ensemble (Strict)   | 896      | 4.90 ✅                 |
| 5       | Ensemble (Relaxed)  | 3648     | 19.95                  |
| 6       | Ensemble (Weighted) | 63       | **0.34** ✅ (lowest FP) |
| 7       | Ensemble (Strict)   | 902      | 4.93                   |
| 7       | Ensemble (Relaxed)  | 3644     | 19.92                  |
| 7       | Ensemble (Weighted) | 997      | 5.45                   |
